## Initializations

In [0]:
from typing import Any

from delta import DeltaTable  # type: ignore
from pyspark.sql.functions import (
    coalesce,
    col,
    concat_ws,
    concat,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    substring,
    trim,
    when,
    months_between,
    dateadd,
    datediff,
    regexp_replace,
    regexp_extract,
    lower,
    min,
    try_divide
)

import os
import logging

import delta_table_creator, transform, user_defined_functions 

spark.conf.set("spark.databricks.io.cache.enabled", "false")

In [0]:
#Import the module
import sys
sys.path.append("../aws_utils")

import aws_utils
# Get Environment Prefix

env = aws_utils.get_env_prefix()

## Table Creation

In [0]:
schema = (
    "ExposureDerivativeID string,"
    "ExposureKey string,"
    "SourceSystem string,"
    "PortfolioKey string,"
    "PortfolioName string,"
    "IngestionTimestamp timestamp,"
    "BatchId string,"
    "ExposureKeyProfits string,"
    "ExposureSourceSystemKeyProfits string,"
    "ExposureKeyQCCepal string,"
    "ExposureSourceSystemKeyQCCepal string,"
    "ExposureKeyQCAlpha string,"
    "ExposureSourceSystemKeyQCAlpha string,"
    "ExposureKeyArotron string,"
    "ExposureSourceSystemKeyArotron string,"
    "ExposureDebtorKey string,"
    "CaseCode int,"
    "AgreementKey string,"
    "PartyKey string,"
    "PartyName string,"
    "CDI string,"
    "PartyTaxRegistrationNumber string,"
    "PartyRelationshipTypeCode string,"
    "PartyRelationshipType string,"
    "PrimaryPartyFlag boolean,"
    "AllBorrowersDeceasedFlag boolean,"
    "ProductKey string,"
    "ProductCode int,"
    "SubProductDescription string,"
    "ExposureTypeCode string,"
    "ExposureType string,"
    "ExposureStatusCode string,"
    "ExposureStatus string,"
    "ExposureStatusDate date,"
    "Opendate date,"
    "ActiveExpiryDate date,"
    "Maturitydate date,"
    "OriginBank string,"
    "OriginationAmount decimal(19,2),"
    "InitialAssignmentAmount decimal(19,2),"
    "Currency string,"
    "BankingUnitCode int,"
    "BankingUnit string,"
    "MechanismCode string,"
    "Mechanism string,"
    "BranchCode string,"
    "Branch string,"
    "PurposeReasonCode int,"
    "PurposeReason string,"
    "PurposeKTCode int,"
    "PurposeKT string,"
    "MaterialityStartDate timestamp,"
    "DaysPastDueWithMateriality int,"
    "DaysPastDue int,"
    "DaysPastDueinterest string,"
    "FirstArreardate date,"
    "Overduedate date,"
    "ClosedDate date,"
    "FirstInstallmentDate date,"
    "IsActiveFlag boolean,"
     "IsActiveFlagInt smallint,"
    "BucketCode string,"
    "Bucket string,"
    "WriteOffDate date,"
    "WriteoffAmount decimal(19,2),"
    "LPSCode string,"
    "LPS string,"
    "LPSEng string,"
    "LPSDate string,"
    "DenouncedDate date,"
    "DaysPastDuePrincipal int,"
    "ExpectedOverduedate date,"
    "OutstandingTotalBalance decimal (19,2),"
    "OffBalance decimal (19,2),"
    "OnBalance decimal(19,2),"
    "DueAmount decimal (26,4),"
    "BogBalance decimal(19,2),"
    "GBV double,"
    "GBVTriggerEventFlag boolean,"
    "GBVTriggerEventDate date,"
    "GBVTriggerEventType string,"
    "TailExposure double,"
    "GBVCappedAmount double,"
    "GBVCappedDate date,"
    "GBVFunctionResult double,"
    "EverDenouncedDate date,"
    "EverDenouncedFlag boolean,"
    "IsSecuredFlag boolean,"
    "InterestNextDueDate date,"
    "ClosedReasonCode string,"
    "ClosedReason string,"
    "PEERestructureTypeCode string,"
    "PEEAdjustmentCode int,"
    "Defaultdate date,"
    "LatestDefaultDate date,"
    "NPLDefaultAmount decimal(19,2),"
    "NonConditionalWriteOffDate date,"
    "NonConditionalWriteOffAmount decimal(19,2),"
    "ConditionalWriteOffDate date,"
    "ConditionalWriteOffAmount decimal(19,2),"
    "NextInstallmentDate date,"
    "NextInstallmentInterestDate date,"
    "NextInstallmentCapitalDate date,"
    "NextInstallmentMinimumDate date,"
    "NextInstallmentAmount decimal(19,2),"
    "NextInstallmentCapitalAmount decimal(19,2),"
    "ExposureTenor int,"
    "IsSoldFlag boolean,"
    "IsSubsidyFlag boolean,"
    "IsGuaranteeFlag boolean,"
    "MasterExposureKey string,"
    "SlaveFlag boolean,"
    "PaymentOrderFlag boolean,"
    "PaymentOrderDate date,"
    "NumberOfUnpaidBills string,"
    "SettlementBucketCode int,"
    "SettlementBucket int,"
    "DaysPastDueFinal int,"
    "HasActiveSettlementFlag boolean,"
    "UnderLawTypeCode string,"
    "UnderLawType string,"
    "LawStatusCode string,"
    "LawStatus string,"
    "LawApplicationStatusDate date,"
    "LawCourtDate date," 
    "LawProvisionalCourtDecisionDate date," 
    "InstalmentFrequency int,"
    "InstallmentFixedAmount decimal(19,2),"
    "NumberOfInstallments int,"
    "PreviousInstallmentdate date,"
    "LastBillingdate date,"
    "LastRestructuringTypeCode int,"
    "LastRestructuringType string,"
    "LastRestructuringDate date,"
    "RestructuringPEEDate date,"
    "RestructuringPEETypeCode int,"
    "RestructuringPEEType string,"
    "TreatmentBranchCode int,"
    "TreatmentBranch string,"
    "ObjectiveStatusCode string,"
    "ObjectiveStatus string,"
    "ObjectiveStatusdate date,"
    "GracePeriodEnddate date,"
    "GracePeriodTypeCode string,"
    "GracePeriodType string,"
    "GracePeriodinterestPayableFlag string,"
    "GracePeriodNextinterestdate timestamp,"
    "GracePeriodinterestFrequency decimal(3,0),"
    "Principal decimal(19,2),"
    "interest decimal(19,2),"
    "Unrealizedinterest decimal(19,2),"
    "Commission decimal(19,2),"
    "Expense decimal(19,2),"
    "Accrued decimal(19,2),"
    "DuePrincipal decimal(19,2),"
    "Dueinterest decimal(19,2),"
    "DueUnrealizedinterest decimal(19,2),"
    "DueCommission decimal(19,2),"
    "DueExpense decimal(19,2),"
    "DueAccrued decimal(19,2),"
    "Penaltyinterest decimal(19,2),"
    "PenaltyUnrealizedinterest decimal(19,2),"
    "InterestRate decimal(19,6),"
    "interestCode int,"
    "interestDescription string,"
    "BaseRate decimal(19,6),"
    "BaseCode int,"
    "BaseDescription string,"
    "N128Rate decimal(19,6),"
    "SpreadRate decimal(19,6),"
    "PenaltyRate decimal(19,6),"
    "InterestRateTypeCode string,"
    "InterestRateType string,"
    "SpreadCurrencyCode string,"
    "SpreadCurrency string,"
    "SpreadPeriod string,"
    "InterestDuration int,"
    "DayCounts int,"
    "InterestFrequency string,"
    "InterestFixedDate date,"
    "interestValiditydate date,"
    "InterestChangeStartdate date,"
    "InterestChangeEnddate date,"
    "InterestRateUpdatedate date,"
    "InterestCalcStopFlag string,"
    "AutoRenewalFlag string,"
    "AssetSegmentCode string,"
    "AssetSegment string,"
    "DiasNumber string,"
    "ProvisionFrozenInterestAmount string,"
    "AccountNticeBUKey string,"
    "SayFlag string,"
    "InsuranceCompany string,"
    "PhysicalNumber string,"
    "PreCourtDate string,"
    "CourtDate string,"
    "ContractSigningDate string,"
    "ExposureLimitAmount string,"
    "BaloonAmount decimal(15,2),"
    "AdministrationStatusCode int,"
    "AdministrationStatus int,"
    "Tenor string,"
    "SPVCode string,"
    "SPV string,"
    "TotalOverdueInsterest decimal(4,0),"
    "TotalOverdueInstallments decimal(4,0),"
    "OutstandingTotalBalanceMLT decimal(19,2),"
    "NextInterestChangeDate date,"
    "PrincipalNextDueDate date,"
    "FutureInstallmentIncreaseGroup string,"
    "FutureInitialInstallmentGroup string,"
    "FutureStepUpSegment string,"
    "FutureStepUpIndicator string,"
    "PastInstallmentIncreaseGroup string,"
    "PastInitialInstallmentGroup string,"
    "PastStepUpSegment string,"
    "PastStepUpIndicator string,"
    "TotalPrincipal decimal(20,2),"
    "TotalInterest decimal(24,2),"
    "TotalAccruals decimal(20,2),"
    "TotalExpense decimal(20,2),"
    "Coupon decimal(21,6),"
    "ActiveModificationProgramStatus string,"
    "DenouncementIndicator string,"
    "ActiveExposureRestructureStartDate date,"
    "MonthsOfCleanPayHistory bigint,"
    "TotalPaymentAgreementPaid decimal(29,2),"
    "TotalPaymentAgreementDue decimal(38,7),"
    "ExposureRestructureOrigination string,"
    "ExposureRestructureStatus string,"
    "ExposureRestructureType string,"
    "CodeofConductStage string,"
    "LetterStatusCode string,"
    "LetterStatus string,"
    "LetterStatusDate date,"
    "CoCStatusCode string,"
    "CoCStatus string,"
    "CoCAccountStatusDate date,"
    "FirstLetterDate date,"
    "FirstLetterReminderDate date,"
    "FirstLetterFlag boolean,"
    "FirstLetterReminderFlag boolean,"
    "SecondLetterDate date,"
    "SecondLetterFlag boolean,"
    "ThirdLetterDate date,"
    "ThirdLetterFlag boolean,"
    "ExclusionReasonCode string,"
    "ExclusionReason string,"
    "ExclusionDate date,"
    "L3869Flag int,"
    "L3869Status string,"
    "L3869SubmissionDate date,"
    "L3869ProvisionalCourtDecisionDate date,"
    "L3869CourtDate date,"
    "PrincipalAtDefault decimal(19,2),"
    "InterestAtDefault decimal(19,2),"
    "AmountDueAtDefault decimal(26,4),"
    "LastModificationActionType string,"
    "LastModificationDate date,"
    "HasBalloonFlag boolean,"
    "HasZeroInterestTodayFlag boolean,"
    "InterestRateTypeTaxonomy string,"
    "InterestRateResetInterval string,"
    "HasSplitBalanceFlag boolean,"
    "FixInterestEndDate date,"
    "RepaymentMethod string,"
    "RepaymentFrequency int,"
    "HasStepUpInstallmentFlag boolean,"
    "UnpaidInterestHandling string,"
    "HasFutureSpreadFlag boolean,"
    "IsBulletFlag boolean,"
    "HasFutureInterestChangesFlag boolean,"
    "OCWFlag boolean,"
    "OCWApplicationStatus string,"
    "L3869InterestFreePeriodEndDate timestamp,"
    "InArrearsFlag boolean,"
    "HasPaymentPlanFlag boolean,"
    "DateLastCurrent date,"
    "DateLastInArrears date,"
    "LastInstallmentDate date,"
    "SettlementDaysPastDue int,"
    "LegalActionPriority string,"
    "IsHighBalanceFlag boolean,"
    "ApplicationDecision string,"
    "ExtrajudicialLetterEverFlag boolean,"
    "ExtrajudicialLetterLastDate date,"
    "CourtRegion string,"
    "SubServicer string,"
    "IFRSGBV decimal(19,2),"
    "DueDebtAmount decimal(19,4),"
    "TotalUnrealizedInterest decimal(19,2),"
    "NetDueAmount decimal(19,2),"
    "TotalInterestAccruals decimal(19,2),"
    "TotalInterestL3869 decimal(19,2),"
    "JustificationCode int,"
    "JustificationComments string,"
    "FutureMargin decimal(9,4),"
    "FutureMarginApplyDate timestamp,"
    "FirstPeriodNumberOfInstallements string,"
    "InstalmentSNFrom decimal(4,0),"
    "InstalmentSNTo decimal(4,0),"
    "newInstalmentAmount decimal(15,2),"
    "L3869WaterfallMainBorrower string,"
    "ForcedOnboarding boolean,"
    "HolisticFlag boolean,"
    "ForcedOnboardReason string,"
    "BusinessDate date"
)


partition_columns = ["BusinessDate"]
run_date = dbutils.widgets.get("run_date")
unity = dbutils.widgets.get("unity")
table_name = "Dexposure"
derivative_table_path = f"`{env}-cdh`.derivativeactive.{table_name}"


# ================== Create Delta Table ==================

logging.info("Creating Delta Table...")
if spark.catalog.tableExists(derivative_table_path) is False:
    create_delta_table(
        spark,
        derivative_table_path,
        partition_columns,
        schema,  # type: ignore
    )
else:
    logging.info("Table already exists.")

# ================== Delete From Delta Table ==================
transform.delete_from_unity_table(
    spark,
    derivative_table_path,
    run_date,  # type: ignore
)

## Read curated tables

## DK Calculations

In [0]:
ExposureSales = spark.sql(f""" select
    ExposureKey,
    SourceSystem,
    BusinessDate
FROM `{env}-cdh`.primary.ExposureSales
WHERE BusinessDate = {run_date!r}
"""
)

ExposureSales = ExposureSales.withColumnRenamed("ExposureKey","ExposureKey_es")

 
collateral = spark.sql(f""" select
 
    AgreementKey,
    CategoryCode,
    CollateralStatusCode,
    RealEstateCollateralStatusCode,
    SourceSystem,
    PortfolioName,
    BusinessDate

FROM  `{env}-cdh`.primary.Collateral
WHERE BusinessDate = {run_date!r}
"""
)
 
collateral.createOrReplaceTempView("collateral") 

PaymentSchedule = spark.sql(f""" select
 
    InstallmentStatus,
    InstallmentAmount,
    InstallmentNumber,
    ExposureKey,
    SettlementKey,
    BusinessDate,
    SourceSystem,
    InterestAmount,
    InstallmentType,
    PaymentScheduleType,
    InstallmentDate,
    CapitalAmount,
    PortfolioKey,
    PortfolioName,
    InstallmentAmountPaid,
    ABS(OverdueDays) as OverdueDays,
    BusinessDate

FROM  `{env}-cdh`.derivative.DPaymentSchedule
WHERE BusinessDate = {run_date!r}
"""
) 

PaymentSchedule.createOrReplaceTempView("PaymentSchedule")

SettlementExposure = spark.sql(f""" select
 
    Bucket,
    ExposureKey,
    SettlementKey,
    BusinessDate,
    SettlementStatus,
    isPrimarySettlementFlag,
    IsActiveFlag,
    SettlementType,
    SubmissionDate,
    SourceSystem

FROM `{env}-cdh`.primary.SettlementExposure
WHERE BusinessDate = {run_date!r}
"""
) 
SettlementExposure.createOrReplaceTempView("SettlementExposure") 


AccountTransaction = spark.sql(f""" select
    ExposureKey,
    TransactionDate,
    PenaltyInterestAmount,
    InterestAmount,
    CommisionAmount,
    ExpenseAmount,
    CapitalAmount,
    JustificationCode,
    ReversedFlag,
    TransactionComments,
    SourceSystem,
    AccountType,
    AccountTransactionKey

FROM `{env}-cdh`.primary.AccountTransaction
WHERE BusinessDate = {run_date!r}
"""
)  


Product = spark.sql(f""" select
 
    ProductCode,
    ProductKey,
    ProductDescription,
    NextValeurDate,
    BusinessDate,
    SourceSystem

FROM `{env}-cdh`.primary.Product
WHERE BusinessDate = {run_date!r}
"""
)  

  
# Legal Actions QC Decommission to Arotron
LegalAction = spark.sql(f""" select
    LegalActionTypeCode,
    LegalActionType,
    ExposureKey,
    LegalActionKey,
    LegalActionStatus,
    LegalActionDate,
    SourceSystem,
    CDI,
    LegalActionStepType,
    LegalActionStepTypeCode,
    CompletionResultCode,
    CompletionResultDate,
    CompletionResult,
    AgreementKey
FROM `{env}-cdh`.primary.legalaction
WHERE BusinessDate = {run_date!r}
-- AND SourceSystem = 'arotron'
"""
)   
LegalAction.createOrReplaceTempView("LegalAction") 



params_loan_product_mod = spark.sql(f""" select
 
    prft_product_code,
    Modification_type 

FROM `{env}-cdh`.landed.params_loan_product_mod
WHERE BusinessDate = {run_date!r}
"""
)    
  
      

Restructuring = spark.sql(f""" select
 
    OldExposureKey,
    NewExposureKey,
    SourceSystem

FROM `{env}-cdh`.primary.Restructuring
WHERE BusinessDate = {run_date!r}
"""
)    
  
exposure_ever = spark.sql(f""" select
    ExposureKey,
    ExposureSourceSystemKey,
    SourceSystem,
    PortfolioKey,
    PortfolioName,
    IsActiveFlag,
    ExposureStatusCode,
    BusinessDate

FROM `{env}-cdh`.primary.Exposure
WHERE BusinessDate <= {run_date!r}
"""
)      

exposure_ever_extra = spark.sql(f""" select
    e.ExposureKey,
    dlps_agr.OperationalLegalStatusLevel3 as LPS,
    cast(dlps_agr.OperationalLegalStatusLevel3Code as string) as LPSCode,
    cast(dlps_agr.LegalProcessStatusLastChangeDate as string) as LPSDate,
    cast(NULL as string) as LPSEng,
    e.SourceSystem,
    e.BusinessDate

FROM `{env}-cdh`.primary.Exposure e
LEFT JOIN `{env}-cdh`.derivativeactive.dlegalprocessstatusagreementdebtor dlps_agr
    ON dlps_agr.AgreementKey = e.agreementkey
    AND dlps_agr.CDI = e.CDI
    AND dlps_agr.BusinessDate = e.BusinessDate
WHERE e.BusinessDate <= {run_date!r}
""")

party_sub_servicer = spark.sql(f""" SELECT 
    CDI,
    SubServicer,
    sourcesystem,
    BusinessDate

FROM `{env}-cdh`.primary.party                               
WHERE BusinessDate = {run_date!r}                             
""")

party_sub_servicer.createOrReplaceTempView("party_sub_servicer") 

assignment = spark.sql(f"""
SELECT *
FROM `{env}-cdh`.primaryactive.`assignment`
WHERE BusinessDate = {run_date!r}  
""")
assignment.createOrReplaceTempView("assignment")

extunionfile = spark.sql(f""" SELECT 
    ExposureNumber,
    IFRSGrossCarryingAmount

FROM `{env}-cdh`.primary.extunionfile                               
WHERE BusinessDate = {run_date!r}                             
""")

extunionfile.createOrReplaceTempView("extunionfile") 

In [0]:
exposure_loan_party_qca = spark.sql(f'''
                    SELECT *
                    FROM `{env}-cdh`.primary.Exposure
                    WHERE BusinessDate = {run_date!r}
                    and SourceSystem ='qca' 
''')
exposure_loan_party_qca.createOrReplaceTempView("exposure_loan_party_qca")


exposure_loan_party_profits = spark.sql(f'''
                    SELECT * 
                    FROM `{env}-cdh`.primary.Exposure
                    WHERE BusinessDate = {run_date!r} 
                    and SourceSystem ='profits' 
''')
exposure_loan_party_profits.createOrReplaceTempView("exposure_loan_party_profits")


legalaction_qca = spark.sql(f""" select
    ExposureKey,
    LegalActionDate,          
    BusinessDate,
    LegalActionTypeCode
FROM `{env}-cdh`.primary.legalaction
WHERE BusinessDate = {run_date!r} and SourceSystem ='qca'
""") 
legalaction_qca.createOrReplaceTempView("legalaction_qca")
 

exposure_loan_profits = spark.sql(f'''
                    SELECT * 
                    FROM `{env}-cdh`.primary.Exposure
                    WHERE BusinessDate = {run_date!r} 
                    and SourceSystem ='profits'
                    and partyrelationshiptype = 'Primary Owner' 
''')
exposure_loan_profits.createOrReplaceTempView("exposure_loan_profits")


exposure_ever_profits = spark.sql(f""" 
                    select *
                    FROM `{env}-cdh`.primary.Exposure
                    WHERE BusinessDate <= {run_date!r} 
                    and SourceSystem ='profits'
                    and partyrelationshiptype = 'Primary Owner'
""") 
exposure_ever_profits = (exposure_ever_profits.withColumn(
                "DueAmountCalc",
                coalesce(exposure_ever_profits.DuePrincipal, lit(0).cast("decimal(19,2)"))
                + coalesce(exposure_ever_profits.Dueinterest, lit(0).cast("decimal(19,2)"))
                + coalesce(
                    exposure_ever_profits.DueUnrealizedinterest, lit(0).cast("decimal(19,2)")
                )
                + coalesce(exposure_ever_profits.Penaltyinterest, lit(0).cast("decimal(19,2)"))
                + coalesce(
                    exposure_ever_profits.PenaltyUnrealizedinterest, lit(0).cast("decimal(19,2)")
                )
                + coalesce(exposure_ever_profits.DueCommission, lit(0).cast("decimal(19,2)"))
                + coalesce(exposure_ever_profits.DueExpense, lit(0).cast("decimal(19,2)"))
                + coalesce(exposure_ever_profits.DueAccrued, lit(0).cast("decimal(19,2)")),
            )
)
exposure_ever_profits.createOrReplaceTempView("exposure_ever_profits")
 
 
restructuring_profits = spark.sql(f""" select
    OldExposureKey,
    NewExposureKey,
    SourceSystem,
    RestructuringDate,
    BusinessDate 
FROM `{env}-cdh`.primary.Restructuring
WHERE BusinessDate = {run_date!r} and SourceSystem ='profits'
""")
restructuring_profits.createOrReplaceTempView("restructuring_profits")
 

dunderlaw = spark.sql(f""" select
    CalcLawStatusCode,
    CalcLawStatus,
    ApplicationStatusDate,
    ApplicationStatus,
    CourtDate,
    CDI,
    UnderLawTypeCode,
    UnderLawType,
    InLawFlag,
    LawStatus,
    LawSystem,
    CompletionReason,
    CompletionReasonCode,
    SubmissionDate,
    TemporaryCourtDecisionDate,
    CountyCourt,
    InterbankStatus3869
FROM `{env}-cdh`.derivative.DUnderLaw
WHERE BusinessDate = {run_date!r}
""") 
dunderlaw.createOrReplaceTempView("dunderlaw")
 
 
product_profits = spark.sql(f""" select
    ProductCode,
    SourceSystem,
    ProductKey,
    NextValeurDate,
    ProductDescription,
    BusinessDate 
FROM `{env}-cdh`.primary.Product
WHERE BusinessDate = {run_date!r} and SourceSystem ='profits'
""") 
product_profits.createOrReplaceTempView("product_profits")
 

portfolio = spark.sql(f""" select
    PortfolioKey,
    PortfolioName,
    unit_code_profits,
    AssignmentDate 
FROM  `{env}-cdh`.primary.Portfolio
WHERE BusinessDate = {run_date!r}
""") 
portfolio.createOrReplaceTempView("portfolio")
 
 
staticlegalactionmigration = spark.sql(f""" select * 
FROM `{env}-cdh`.reference.staticlegalactionmigration 
""") 
staticlegalactionmigration.createOrReplaceTempView("staticlegalactionmigration")
 
 
staticreglookupvalue = spark.sql(f""" select * 
FROM `{env}-cdh`.reference.staticreglookupvalue 
""") 
staticreglookupvalue.createOrReplaceTempView("staticreglookupvalue")
 

dimpeerescheduling = spark.sql(f""" select * 
FROM `{env}-cdh`.reference.dimpeerescheduling 
""") 
dimpeerescheduling.createOrReplaceTempView("dimpeerescheduling")


initializationexposureinfo = spark.sql(f""" select * 
FROM `{env}-cdh`.reference.initializationexposureinfo 
""") 
initializationexposureinfo.createOrReplaceTempView("initializationexposureinfo")

dlegalprocessstatusagreement= spark.sql(f""" select 
                                        OperationalLegalStatusLevel3,
                                        OperationalLegalStatusLevel3Code,
                                        LegalProcessStatusLastChangeDate,
                                        AgreementKey
FROM `{env}-cdh`.derivative.dlegalprocessstatusagreement 
""") 
dlegalprocessstatusagreement.createOrReplaceTempView("dlegalprocessstatusagreement")

# p-cdh.derivative.dlegalprocessstatusagreementdebtor

lps_agreement_party = spark.sql(f"""
SELECT 
    AgreementKey,
    CDI,
    OperationalLegalStatusLevel3Code AS LPSCode,
    OperationalLegalStatusLevel3 AS LPS,
    LegalProcessStatusLastChangeDate AS LPSDate
FROM `{env}-cdh`.derivativeactive.dlegalprocessstatusagreementdebtor
WHERE BusinessDate = {run_date!r}
""")

lps_agreement_party.createOrReplaceTempView("lps_agreement_party")

# LPS per ExposureKey — clean DataFrame for direct join to target_table_profits
lps_by_exposure = spark.sql(f"""
    SELECT e.ExposureKey AS lps_ExposureKey,
           dlps.OperationalLegalStatusLevel3Code AS LPSCode,
           dlps.OperationalLegalStatusLevel3 AS LPS,
           CAST(dlps.LegalProcessStatusLastChangeDate AS STRING) AS LPSDate
    FROM `{env}-cdh`.primary.Exposure e
    INNER JOIN `{env}-cdh`.derivativeactive.dlegalprocessstatusagreementdebtor dlps
        ON dlps.AgreementKey = e.AgreementKey AND dlps.CDI = e.CDI AND dlps.BusinessDate = e.BusinessDate
    WHERE e.BusinessDate = {run_date!r}
      AND e.SourceSystem = 'profits'
      AND e.PartyRelationshipType = 'Primary Owner'
""")

#high_balance_by_exposure 

hb_high_balance_cdis = set(
    r["CDI"]
    for r in spark.sql(f"""
        SELECT DISTINCT CDI
        FROM `{env}-cdh`.externalfiles.high_balance_aggrement_haps
        WHERE PartyRelationshipType = 'Primary Owner'
          AND CDI IS NOT NULL
    """).collect()
)
hb_high_balance_cdis_list = list(hb_high_balance_cdis)



In [0]:
dk_df = spark.sql(f"""
WITH exp_prod AS (
  select distinct 
     ex.BusinessDate, 
     ex.PortfolioName, 
     ex.ExposureKey, 
     ex.ExposureStatus, 
     ex.IsActiveFlag, 
     ex.Opendate,
     ex.SlaveFlag, 
     ex.SourceSystem, 
     pr.ProductKey, 
     pr.ProductDescription
  from exposure_loan_profits ex
  join product_profits pr 
  on ex.ProductKey = pr.ProductKey                   
), 
restr_1 AS (
  select distinct NewExposureKey, 
     max(RestructuringDate) RestructuringDate
  from restructuring_profits
  group by NewExposureKey                     
),
exp_portf_pat_restr AS (
  select distinct 
    ex.ExposureKey, 
    try_cast(ex.OpenDate as string) RestructuringDate
  from exposure_loan_profits ex 
  inner join portfolio pf on ex.PortfolioKey = pf.unit_code_profits 
  inner join (select distinct ExposureKey from PaymentSchedule) ps on ex.ExposureKey = ps.ExposureKey
  left join (select distinct NewExposureKey from restructuring_profits) rs on ex.ExposureKey = rs.NewExposureKey
  where ex.Opendate <= pf.AssignmentDate
),
pay_portf AS (
  select distinct 
    ExposureKey 
  from PaymentSchedule p
  inner join portfolio pf on p.Portfolioname = pf.PortfolioName
  where try_cast(p.InstallmentDate as date) > last_day(try_cast(pf.AssignmentDate as date))
),
pay_portf_2 AS (
  select 
    ExposureKey, 
    count(distinct last_day(InstallmentDate)) MonthsOfCleanPayHistory  
  from PaymentSchedule p
  inner join portfolio pf on p.Portfolioname = pf.PortfolioName
  where try_cast(InstallmentDate as date) > last_day(try_cast(pf.AssignmentDate as date))
    and InstallmentDate <= BusinessDate
    and InstallmentAmountPaid > 0
  group by ExposureKey
  ),
  pay_portf_3 AS (
    select ExposureKey
        ,sum(InstallmentAmount) TotalInstallmentAmount
			  ,sum(case when InstallmentAmountPaid > 0 and dateadd(day, OverdueDays, InstallmentDate) <= BusinessDate then InstallmentAmountPaid end) TotalInstallmentAmountPaid
  from PaymentSchedule p
  inner join portfolio pf on p.Portfolioname = pf.PortfolioName 
  where try_cast(InstallmentDate as date) > last_day(try_cast(pf.AssignmentDate as date))
    and InstallmentDate <= BusinessDate
  group by ExposureKey
  )
 
 
 
select distinct
ex.ExposureKey as ExposureKey_Dk_df
,rc.PaymentArrangementDate ActiveExposureRestructureStartDate_Dk
,mc.MonthsOfCleanPayHistory MonthsOfCleanPayHistory_Dk
,i.TotalInstallmentAmountPaid TotalPaymentAgreementPaid_Dk
,i.TotalInstallmentAmount TotalPaymentAgreementDue_Dk
,rc.PaymentArrangementOrigination ExposureRestructureOrigination_Dk
,rc.PaymentArrangementType ExposureRestructureType_Dk
,rc.PaymentArrangementStatus ExposureRestructureStatus_Dk
---,rc.PaymentArrangementDate ExposureRestructureDate_Dk
from exposure_loan_profits ex
inner join (
select e.BusinessDate
  ,e.ExposureKey AccountNo
  ,coalesce(lr.RestructuringDate,mr.RestructuringDate) PaymentArrangementDate
  ,case when e.ExposureStatus not in ('Define Delay','Writen Off','ΕΠΙΣΦΑΛΕΙΑ','ΟΡΙΣΤΙΚΗ ΚΑΘΥΣΤΕΡΗΣΗ') and e.IsActiveFlag = '1'-- KS FIX /DATA2-421 21/7/2025  
        and coalesce(lr.RestructuringDate,mr.RestructuringDate) = try_cast(e.Opendate as string) then 'Active'
      when e.ExposureStatus not in ('Define Delay','Writen Off','ΕΠΙΣΦΑΛΕΙΑ','ΟΡΙΣΤΙΚΗ ΚΑΘΥΣΤΕΡΗΣΗ') and e.IsActiveFlag = '1'-- KS FIX /DATA2-421 21/7/2025   
        and coalesce(lr.RestructuringDate,mr.RestructuringDate) > try_cast(e.Opendate as string) then 'Inactive'
      when e.ExposureStatus in ('Define Delay','Writen Off','ΕΠΙΣΦΑΛΕΙΑ','ΟΡΙΣΤΙΚΗ ΚΑΘΥΣΤΕΡΗΣΗ') or e.IsActiveFlag  = '0' --KS FIX /DATA2-421 21/7/2025   
        then 'Inactive'
  end PaymentArrangementStatus
  ,case when e.ProductDescription like '%ΥΠΟ ΑΙΡΕΣΗ%' then 'Conditional Woff' else 'Restructure' end PaymentArrangementType 
  ,case when right(e.Productkey,4) in ('6078', '6079', '4015', '4016', '7362', '7363') and lr.RestructuringDate is not null     then 'Cepal L3869'
      when right(e.Productkey,4) not in ('6078', '6079', '4015', '4016', '7362', '7363') and lr.RestructuringDate is not null then 'Cepal Restructure'
      when right(e.Productkey,4) in ('6078', '6079', '4015', '4016', '7362', '7363') and mr.RestructuringDate is not null     then 'Migrated L3869'
      when right(e.Productkey,4) not in ('6078', '6079', '4015', '4016', '7362', '7363') and mr.RestructuringDate is not null then 'Migrated Restructure'
      when  e.ProductDescription like '%OCW%' and  e.ProductDescription not like '%ΑΝΑΣΤΟΛΗ%' and e.SlaveFlag=false and e.Opendate >= '2022-01-01' then 'OCW' 
  end PaymentArrangementOrigination
from exp_prod e 
left join restr_1 lr
on lr.NewExposureKey = e.ExposureKey
left join exp_portf_pat_restr mr
on mr.ExposureKey = e.ExposureKey
and (lr.RestructuringDate is not null or mr.RestructuringDate is not null)
) rc 
on rc.AccountNo = ex.ExposureKey
inner join pay_portf ps 
on ps.ExposureKey = ex.ExposureKey
left join pay_portf_2 mc 
on mc.ExposureKey = ex.ExposureKey
left join pay_portf_3 i 
on i.ExposureKey = ex.ExposureKey
--where rc.PaymentArrangementType <> 'Conditional Woff'
--and rc.PaymentArrangementStatus =  'Active'
""")
 
dk_df.createOrReplaceTempView("dk_df")


In [0]:
dk_df3 =( exposure_loan_profits.withColumn(
                "Principal_Dk",
                (coalesce(exposure_loan_profits.Principal,lit(0))  +  coalesce(exposure_loan_profits.DuePrincipal,lit(0)))
                ).withColumn(
                "Interest_Dk",
                (coalesce(exposure_loan_profits.interest,lit(0)) + 
                coalesce(exposure_loan_profits.Dueinterest,lit(0)) + 
                coalesce(exposure_loan_profits.Penaltyinterest,lit(0)) + 
                coalesce(exposure_loan_profits.PenaltyUnrealizedinterest,lit(0)) + 
                coalesce(exposure_loan_profits.DueUnrealizedinterest,lit(0)) + 
                coalesce(exposure_loan_profits.Unrealizedinterest,lit(0)))
                ).withColumn(
                "Accruals_Dk",
                (coalesce(exposure_loan_profits.Accrued,lit(0)) + coalesce(exposure_loan_profits.DueAccrued,lit(0)))
                ).withColumn(
                "Expense_Dk",
                ((coalesce(exposure_loan_profits.Expense,lit(0))) + (coalesce(exposure_loan_profits.DueExpense,lit(0))))
                ).withColumn(
                "DenouncementIndicator_Dk",
                when((exposure_loan_profits.ExposureStatus == 'Written Off') 
                     | (exposure_loan_profits.ExposureStatus == 'ΕΠΙΣΦΑΛΕΙΑ')
                     | (exposure_loan_profits.ExposureStatus == 'Define Delay')
                     | (exposure_loan_profits.ExposureStatus == 'ΟΡΙΣΤΙΚΗ ΚΑΘΥΣΤΕΡΗΣΗ'),"true"
                ).otherwise("false")
                )
          )
 
dk_df3.createOrReplaceTempView("dk_df3")
 
dk_df3 = dk_df3.withColumnRenamed("ExposureKey", "ExposureKey_Dk")
 
all_columns = dk_df3.columns
 
columns_to_keep = [col for col in all_columns if col.endswith("_Dk")]
 
dk_df3 = dk_df3.select(columns_to_keep)
 
dk_df3 = dk_df3.withColumnRenamed("ExposureKey_Dk", "ExposureKey_Dk_df3")


In [0]:
dk_df4_pre = spark.sql(f"""
        SELECT  
        ex.ExposureKey, max(exqca2.CodeofConductStage_Qca_Dk) AS CodeofConductStage_Qca_Dk
        FROM exposure_loan_party_qca  exqca
        LEFT JOIN (select exposurekey,
                    CASE WHEN ex.ExposureStatus IN ('Define Delay','ΟΡΙΣΤΙΚΗ ΚΑΘΥΣΤΕΡΗΣΗ','ΕΠΙΣΦΑΛΕΙΑ', 'Writen Off') THEN '5'
                     WHEN COUNT(ex.ExposureKey ) OVER(PARTITION BY ex.ExposureKey) > 0
                     AND COUNT(ex.ExposureKey ) OVER(PARTITION BY ex.ExposureKey) =
                     SUM (CASE WHEN ex.ThirdLetterDate IS NOT NULL THEN 1 END) OVER(PARTITION BY ex.ExposureKey) THEN '4'      
                     WHEN ex.ThirdLetterDate IS NOT NULL THEN '3'  
                     WHEN ex.SecondLetterDate IS NOT NULL THEN '2'
                     WHEN ex.FirstLetterDate IS NOT NULL THEN '1'                      
                    END CodeofConductStage_Qca_Dk
                    from exposure_loan_party_qca ex
                    ) exqca2
        ON exqca.exposurekey =exqca2.exposurekey
        INNER JOIN exposure_loan_profits ex
        ON exqca.Exposurekey = substring(ex.Exposurekey,4,50)
        GROUP BY all
""")
 
dk_df4_pre.createOrReplaceTempView("dk_df4_pre")
 
dk_df4 = spark.sql(f'''
                 select ExposureKey,
                 case when CodeofConductStage_Qca_Dk ='5' then 'Denounced'
                        when CodeofConductStage_Qca_Dk ='4' then 'Non-Cooperative'
                        when CodeofConductStage_Qca_Dk ='3' then '3rd letter'
                        when CodeofConductStage_Qca_Dk ='2' then '2nd letter'
                        when CodeofConductStage_Qca_Dk ='1' then '1st letter'
                        end as CodeofConductStage_Qca_Dk
                from dk_df4_pre
                 ''')

dk_df4.createOrReplaceTempView("dk_df4")

dk_df4 = dk_df4.withColumnRenamed("ExposureKey", "ExposureKey_Dk_df4")


In [0]:
# Legal Actions QC Decommission to Arotron
dk_df5 = spark.sql(f"""
    SELECT  
    coalesce(ex_others.ExposureKey, ex_arotron.ExposureKey) ExposureKey, -- Legal Actions QC Decommission to Arotron
    MAX(
        CASE 
            WHEN coalesce(dl_others.OperationalLegalStatusLevel3Code, dl_arotron.OperationalLegalStatusLevel3Code) >= 211 THEN 1 
            WHEN LegalActionTypeCode IN ('5104', '2403', '5103', '5105', '2404', '2405', '2414') OR(LegalActionTypeCode LIKE '%SYS_BA_LEG_ACT_CPL_LG_02%' 
                    AND LegalActionStepType IN ('ΕΠΙΔΟΣΗ', 'ΘΥΡΟΚΟΛΛΗΣΗ', 'ΟΛΟΚΛΗΡΩΣΗ ΔΙΑΔΙΚΑΣΙΑΣ ΕΞΩΔΙΚΟΥ ΚΑΤΑΓΓΕΛΙΑΣ', 'ΠΑΡΑΛΑΒΗ ΕΚΘΕΣΗ ΕΠΙΔΟΣΗΣ') -- Legal Actions QC Decommission to Arotron
                    AND CompletionResultCode IN ('SYS_ACTION_CPL_IACT_032', 'SYS_ACTION_CPL_IACT_033', 'SYS_ACTION_CPL_IACT_034'))  -- Legal Actions QC Decommission to Arotron
            THEN 1 
            ELSE 0 
        END
    ) AS ExtrajudicialLetterEverFlag_Dk,
    MAX(
        CASE 
            WHEN coalesce(dl_others.OperationalLegalStatusLevel3Code, dl_arotron.OperationalLegalStatusLevel3Code) = 211 THEN coalesce(dl_others.LegalProcessStatusLastChangeDate, dl_arotron.LegalProcessStatusLastChangeDate)
            WHEN LegalActionTypeCode IN ('5104', '2403', '5103', '5105', '2404', '2405', '2414') OR     (LegalActionTypeCode LIKE '%SYS_BA_LEG_ACT_CPL_LG_02%' 
                     AND LegalActionStepType IN ('ΕΠΙΔΟΣΗ', 'ΘΥΡΟΚΟΛΛΗΣΗ', 'ΟΛΟΚΛΗΡΩΣΗ ΔΙΑΔΙΚΑΣΙΑΣ ΕΞΩΔΙΚΟΥ ΚΑΤΑΓΓΕΛΙΑΣ', 'ΠΑΡΑΛΑΒΗ ΕΚΘΕΣΗ ΕΠΙΔΟΣΗΣ') -- Legal Actions QC Decommission to Arotron
                     AND CompletionResultCode IN ('SYS_ACTION_CPL_IACT_032', 'SYS_ACTION_CPL_IACT_033', 'SYS_ACTION_CPL_IACT_034')) -- Legal Actions QC Decommission to Arotron
            THEN CompletionResultDate -- Legal Actions QC Decommission to Arotron
            ELSE NULL 
        END
    ) AS ExtrajudicialLetterLastDate_Dk
FROM 
    legalaction la
LEFT JOIN -- Legal Actions QC Decommission to Arotron
    exposure_loan_party_profits ex_others
    ON SUBSTRING(ex_others.ExposureKey, 4) = la.Exposurekey
    and la.SourceSystem <> 'arotron'
LEFT JOIN -- Legal Actions QC Decommission to Arotron
    exposure_loan_party_profits ex_arotron
    ON la.CDI = ex_arotron.CDI
    AND la.SourceSystem = 'arotron'
LEFT JOIN 
    dlegalprocessstatusagreement dl_others
    ON dl_others.AgreementKey = ex_others.AgreementKey
LEFT JOIN 
    dlegalprocessstatusagreement dl_arotron
    ON dl_arotron.AgreementKey = ex_arotron.AgreementKey
GROUP BY 
    coalesce(ex_others.ExposureKey, ex_arotron.ExposureKey); -- Legal Actions QC Decommission to Arotron
""")

dk_df5.createOrReplaceTempView("dk_df5")

dk_df5 = dk_df5.withColumnRenamed("ExposureKey", "ExposureKey_Dk_df5")

In [0]:
dk_df6 = spark.sql(f"""
    SELECT  
    coalesce(ex_others.ExposureKey, ex_arotron.ExposureKey) ExposureKey, -- Legal Actions QC Decommission to Arotron
    MAX(
        CASE 
            WHEN coalesce(dl_others.OperationalLegalStatusLevel3Code, dl_arotron.OperationalLegalStatusLevel3Code) >=311 THEN 1 
            WHEN LegalActionTypeCode IN ('1230','1235','5045','5144','5175','5659','6161','6272','6278','6284','6310') 
                 OR (LegalActionType = 'Διαταγή Πληρωμής' And LegalActionStepType = 'Διαταγή Πληρωμής' and CompletionResult = 'ΕΠΙΔΟΣΗ ΣΕ ΜΕΡΟΣ ΤΩΝ ΕΝΕΧΟΜΕΝΩΝ' Or
                     LegalActionType = 'Β Επίδοση' And LegalActionStepType = 'ΘΥΡΟΚΟΛΛΗΣΗ' and CompletionResult = '' Or
                     LegalActionType = 'Β Επίδοση' And LegalActionStepType = 'ΕΠΙΔΟΣΗ ΠΡΟΣ ΓΝΩΣΗ' and CompletionResult = '' Or
                     LegalActionType = 'Β Επίδοση' And LegalActionStepType = 'ΘΥΡΟΚΟΛΛΗΣΗ ΠΡΟΣ ΓΝΩΣΗ' and CompletionResult = 'ΕΠΙΔΟΣΗ ΣΕ ΜΕΡΟΣ ΤΩΝ ΕΝΕΧΟΜΕΝΩΝ' Or
                     LegalActionType = 'Β Επίδοση' And LegalActionStepType = 'ΕΠΙΔΟΣΗ' and CompletionResult = '' Or
                     LegalActionType = 'Β Επίδοση' And LegalActionStepType = 'ΕΠΙΔΟΣΗ ΠΡΟΣ ΓΝΩΣΗ' and CompletionResult = 'ΕΠΙΔΟΣΗ ΣΕ ΜΕΡΟΣ ΤΩΝ ΕΝΕΧΟΜΕΝΩΝ' Or
                     LegalActionType = 'Β Επίδοση' And LegalActionStepType = 'ΕΠΙΔΟΣΗ' and CompletionResult = 'ΕΠΙΔΟΣΗ ΠΡΟΣ ΟΛΟΥΣ' Or
                     LegalActionType = 'Β Επίδοση' And LegalActionStepType = 'ΘΥΡΟΚΟΛΛΗΣΗ' and CompletionResult = 'ΕΠΙΔΟΣΗ ΠΡΟΣ ΟΛΟΥΣ' Or
                     LegalActionType = 'Β Επίδοση' And LegalActionStepType = 'ΕΠΙΔΟΣΗ' and CompletionResult = 'ΕΠΙΔΟΣΗ ΣΕ ΜΕΡΟΣ ΤΩΝ ΕΝΕΧΟΜΕΝΩΝ' Or
                     LegalActionType = 'Β Επίδοση' And LegalActionStepType = 'ΕΠΙΔΟΣΗ ΠΡΟΣ ΓΝΩΣΗ' and CompletionResult = 'ΕΠΙΔΟΣΗ ΠΡΟΣ ΟΛΟΥΣ' Or
                     LegalActionType = 'Β Επίδοση' And LegalActionStepType = 'ΕΠΙΔΟΣΗ' and CompletionResult = '' Or
                     LegalActionType = 'Αγωγή' And LegalActionStepType = 'ΕΠΙΔΟΣΗ ΑΠΟΦΑΣΗΣ' and CompletionResult = '' Or
                     LegalActionType = 'Επίδοση εγγράφου' And LegalActionStepType = 'ΑΠΟΣΤΟΛΗ ΠΡΟΣ ΕΠΙΔΟΣΗ' and CompletionResult = 'ΘΥΡΟΚΟΛΛΗΣΗ' Or
                     LegalActionType = 'Επίδοση εγγράφου' And LegalActionStepType = 'ΑΠΟΣΤΟΛΗ ΠΡΟΣ ΕΠΙΔΟΣΗ' and CompletionResult = 'ΕΠΙΔΟΣΗ' Or
                     LegalActionType = 'Επίδοση εγγράφου' And LegalActionStepType = 'ΠΑΡΑΛΑΒΗ ΕΚΘΕΣΗ ΕΠΙΔΟΣΗΣ' and CompletionResult = 'ΕΠΙΔΟΣΗ' Or
                     LegalActionType = 'Επίδοση εγγράφου' And LegalActionStepType = 'ΑΠΟΣΤΟΛΗ ΠΡΟΣ ΕΠΙΔΟΣΗ' and CompletionResult = 'ΕΠΙΔΟΣΗ' Or
                     LegalActionType = 'Επίδοση εγγράφου' And LegalActionStepType = 'ΠΑΡΑΛΑΒΗ ΕΚΘΕΣΗ ΕΠΙΔΟΣΗΣ' and CompletionResult = 'ΕΠΙΔΟΣΗ' Or
                     LegalActionType = 'Επίδοση εγγράφου' And LegalActionStepType = '' and CompletionResult = 'ΕΠΙΔΟΣΗ' Or
                     LegalActionType = 'Επίδοση εγγράφου' And LegalActionStepType = 'ΠΑΡΑΛΑΒΗ ΕΚΘΕΣΗ ΕΠΙΔΟΣΗΣ' and CompletionResult = 'ΕΠΙΔΟΣΗ' Or
                     LegalActionType = 'Επίδοση εγγράφου' And LegalActionStepType = 'ΑΠΟΣΤΟΛΗ ΠΡΟΣ ΕΠΙΔΟΣΗ' and CompletionResult = 'ΕΠΙΔΟΣΗ') THEN 1
			WHEN (LegalActionTypeCode = 'SYS_BA_LEG_ACT_CPL_LG_05' 
				  and LegalActionStatus = 'Completed'
				  and LegalActionStepTypeCode in ('SYS_BA_LEG_PHASE_CPL_PH_191','SYS_BA_LEG_PHASE_CPL_PH_217','SYS_BA_LEG_PHASE_CPL_PH_230','SYS_BA_LEG_PHASE_CPL_PH_231')
				  and CompletionResultCode in ('SYS_ACTION_CPL_IACT_033','SYS_ACTION_CPL_IACT_034')) 
            THEN 1
			ELSE 0
		 END
    ) AS PaymentOrderFlag_Dk, -- Legal Actions QC Decommission to Arotron
    MAX(
        CASE 
            WHEN coalesce(dl_others.OperationalLegalStatusLevel3Code, dl_arotron.OperationalLegalStatusLevel3Code) >=311 THEN coalesce(dl_others.LegalProcessStatusLastChangeDate, dl_arotron.LegalProcessStatusLastChangeDate)
            WHEN LegalActionTypeCode IN ('1230','1235','5045','5144','5175','5659','6161','6272','6278','6284','6310')
                 OR (LegalActionType = 'Διαταγή Πληρωμής' And LegalActionStepType = 'Διαταγή Πληρωμής' and CompletionResult = 'ΕΠΙΔΟΣΗ ΣΕ ΜΕΡΟΣ ΤΩΝ ΕΝΕΧΟΜΕΝΩΝ' Or
                     LegalActionType = 'Β Επίδοση' And LegalActionStepType = 'ΘΥΡΟΚΟΛΛΗΣΗ'and CompletionResult = '' Or
                     LegalActionType = 'Β Επίδοση' And LegalActionStepType = 'ΕΠΙΔΟΣΗ ΠΡΟΣ ΓΝΩΣΗ' and  CompletionResult = '' Or
                     LegalActionType = 'Β Επίδοση' And LegalActionStepType = 'ΘΥΡΟΚΟΛΛΗΣΗ ΠΡΟΣ ΓΝΩΣΗ' and CompletionResult = 'ΕΠΙΔΟΣΗ ΣΕ ΜΕΡΟΣ ΤΩΝ ΕΝΕΧΟΜΕΝΩΝ' Or
                     LegalActionType = 'Β Επίδοση' And LegalActionStepType = 'ΕΠΙΔΟΣΗ' and CompletionResult = '' Or
                     LegalActionType = 'Β Επίδοση' And LegalActionStepType = 'ΕΠΙΔΟΣΗ ΠΡΟΣ ΓΝΩΣΗ' and CompletionResult = 'ΕΠΙΔΟΣΗ ΣΕ ΜΕΡΟΣ ΤΩΝ ΕΝΕΧΟΜΕΝΩΝ' Or
                     LegalActionType = 'Β Επίδοση' And LegalActionStepType = 'ΕΠΙΔΟΣΗ' and CompletionResult = 'ΕΠΙΔΟΣΗ ΠΡΟΣ ΟΛΟΥΣ' Or
                     LegalActionType = 'Β Επίδοση' And LegalActionStepType = 'ΘΥΡΟΚΟΛΛΗΣΗ' and CompletionResult = 'ΕΠΙΔΟΣΗ ΠΡΟΣ ΟΛΟΥΣ' Or
                     LegalActionType = 'Β Επίδοση' And LegalActionStepType = 'ΕΠΙΔΟΣΗ' and CompletionResult = 'ΕΠΙΔΟΣΗ ΣΕ ΜΕΡΟΣ ΤΩΝ ΕΝΕΧΟΜΕΝΩΝ' Or
                     LegalActionType = 'Β Επίδοση' And LegalActionStepType = 'ΕΠΙΔΟΣΗ ΠΡΟΣ ΓΝΩΣΗ' and CompletionResult = 'ΕΠΙΔΟΣΗ ΠΡΟΣ ΟΛΟΥΣ' Or
                     LegalActionType = 'Β Επίδοση' And LegalActionStepType = 'ΕΠΙΔΟΣΗ' and CompletionResult = '' Or
                     LegalActionType = 'Αγωγή' And LegalActionStepType = 'ΕΠΙΔΟΣΗ ΑΠΟΦΑΣΗΣ' and CompletionResult = '' Or
                     LegalActionType = 'Επίδοση εγγράφου' And LegalActionStepType = 'ΑΠΟΣΤΟΛΗ ΠΡΟΣ ΕΠΙΔΟΣΗ' and CompletionResult = 'ΘΥΡΟΚΟΛΛΗΣΗ' Or
                     LegalActionType = 'Επίδοση εγγράφου' And LegalActionStepType = 'ΑΠΟΣΤΟΛΗ ΠΡΟΣ ΕΠΙΔΟΣΗ' and CompletionResult = 'ΕΠΙΔΟΣΗ' Or
                     LegalActionType = 'Επίδοση εγγράφου' And LegalActionStepType = 'ΠΑΡΑΛΑΒΗ ΕΚΘΕΣΗ ΕΠΙΔΟΣΗΣ' and CompletionResult = 'ΕΠΙΔΟΣΗ' Or
                     LegalActionType = 'Επίδοση εγγράφου' And LegalActionStepType = 'ΑΠΟΣΤΟΛΗ ΠΡΟΣ ΕΠΙΔΟΣΗ' and CompletionResult = 'ΕΠΙΔΟΣΗ' Or
                     LegalActionType = 'Επίδοση εγγράφου' And LegalActionStepType = 'ΠΑΡΑΛΑΒΗ ΕΚΘΕΣΗ ΕΠΙΔΟΣΗΣ' and CompletionResult = 'ΕΠΙΔΟΣΗ' Or
                     LegalActionType = 'Επίδοση εγγράφου' And LegalActionStepType = '' and CompletionResult = 'ΕΠΙΔΟΣΗ' Or
                     LegalActionType = 'Επίδοση εγγράφου' And LegalActionStepType = 'ΠΑΡΑΛΑΒΗ ΕΚΘΕΣΗ ΕΠΙΔΟΣΗΣ' and CompletionResult = 'ΕΠΙΔΟΣΗ' Or
                     LegalActionType = 'Επίδοση εγγράφου' And LegalActionStepType = 'ΑΠΟΣΤΟΛΗ ΠΡΟΣ ΕΠΙΔΟΣΗ' and CompletionResult = 'ΕΠΙΔΟΣΗ') 
            THEN coalesce(dl_others.LegalProcessStatusLastChangeDate, dl_arotron.LegalProcessStatusLastChangeDate)
			WHEN (LegalActionTypeCode = 'SYS_BA_LEG_ACT_CPL_LG_05'
			 and LegalActionStatus = 'Completed'
			 and LegalActionStepTypeCode in ('SYS_BA_LEG_PHASE_CPL_PH_191','SYS_BA_LEG_PHASE_CPL_PH_217','SYS_BA_LEG_PHASE_CPL_PH_230','SYS_BA_LEG_PHASE_CPL_PH_231') 
			 and CompletionResultCode in ('SYS_ACTION_CPL_IACT_033','SYS_ACTION_CPL_IACT_034')) 
            THEN CompletionResultDate
		    ELSE NULL 
		END
    ) AS PaymentOrderDate_Dk    -- Legal Actions QC Decommission to Arotron
FROM 
    legalaction la
LEFT JOIN -- Legal Actions QC Decommission to Arotron
    exposure_loan_party_profits ex_others
    ON SUBSTRING(ex_others.ExposureKey, 4) = la.Exposurekey
    and la.SourceSystem <> 'arotron'
LEFT JOIN -- Legal Actions QC Decommission to Arotron
    exposure_loan_party_profits ex_arotron
    ON la.CDI = ex_arotron.CDI
    AND la.SourceSystem = 'arotron'
LEFT JOIN 
    dlegalprocessstatusagreement dl_others
    ON dl_others.AgreementKey = ex_others.AgreementKey
LEFT JOIN 
    dlegalprocessstatusagreement dl_arotron
    ON dl_arotron.AgreementKey = ex_arotron.AgreementKey
GROUP BY 
    coalesce(ex_others.ExposureKey, ex_arotron.ExposureKey); -- Legal Actions QC Decommission to Arotron

""")

dk_df6.createOrReplaceTempView("dk_df6")

dk_df6 = dk_df6.withColumnRenamed("ExposureKey", "ExposureKey_Dk_df6")

In [0]:
hasactivesettlement= spark.sql("""
WITH 
SettlementExposure_Bucket_profits AS (
  SELECT
    se.Bucket AS BucketQC,
    se.ExposureKey,
    se.SettlementStatus
  FROM SettlementExposure se
  WHERE se.SourceSystem = 'profits'
    AND se.isPrimarySettlementFlag = TRUE
),
SettlementExposure_Bucket_qca AS (
  SELECT
    BucketQCA,
    ExposureKey AS ExposureKey_seb
  FROM (
    SELECT
      se.Bucket AS BucketQCA,
      e.ExposureKey,
      ROW_NUMBER() OVER (PARTITION BY e.ExposureKey ORDER BY se.SubmissionDate DESC) AS rn
    FROM SettlementExposure se
    INNER JOIN exposure_loan_profits e
      ON substring(e.ExposureKey, 4, 50) = se.ExposureKey
    WHERE se.SourceSystem = 'qca'
      AND se.SettlementType != 'Νόμος 3869 – Προσωρινή Διαταγή'
      AND se.SettlementStatus IN ('Running','Partially Fulfilled')
  ) a
  WHERE rn = 1
)
SELECT
  e.ExposureKey,
  CASE
    WHEN qca.ExposureKey_seb IS NOT NULL
    THEN 1
    ELSE 0
  END AS HasActiveSettlementFlag3869
FROM exposure_loan_profits e
LEFT JOIN SettlementExposure_Bucket_qca qca
  ON qca.ExposureKey_seb = e.ExposureKey;
""")
hasactivesettlement.createOrReplaceTempView("hasactivesettlement")

In [0]:
dragged = spark.sql(f"""
WITH base AS (
  SELECT DISTINCT
      ex.ExposureKey,
      ex.CDI,
      ex.PortfolioName,
      CASE 
        WHEN ex.ProductCode IN ('6078','6079','4015','4016','7362','7363')
          OR ex.ProductKey IN ('40568-40001','40568-40000','40568;40001','40568;40000') 
        THEN 1 ELSE 0 
      END AS L3869ProductFlag,
      COALESCE(stl.HasActiveSettlementFlag3869, 0) AS HasActiveSettlementFlag3869,
      ex.BusinessDate
  FROM exposure_loan_party_profits ex
  LEFT JOIN hasactivesettlement stl
    ON ex.ExposureKey = stl.ExposureKey
  WHERE ex.PartyRelationshipType = 'Primary Owner'
),
tagged AS (
  SELECT
      b.*,
      MAX(CASE WHEN b.L3869ProductFlag = 1 OR b.HasActiveSettlementFlag3869 = 1 THEN 1 ELSE 0 END)
        OVER (PARTITION BY b.CDI, b.PortfolioName, b.BusinessDate)
        AS grp_has_impl_3869
  FROM base b
)
SELECT
    t.*,
    CASE 
      WHEN t.grp_has_impl_3869 = 1 
       AND t.L3869ProductFlag = 0 
       AND t.HasActiveSettlementFlag3869 = 0
      THEN 1 ELSE 0 
    END AS L3869DraggedFlag
FROM tagged t
""")

dragged.createOrReplaceTempView("dragged")

In [0]:
dk_df7 = spark.sql(
    f"""
    select  ex.ExposureKey,
            max(case when du.InLawFlag=1 and du.UnderLawType='N3869' then 1 else  0 end) L3869Flag_Dk,
            max(case when du.InLawFlag=1 and du.UnderLawType='N3869' then du.CalcLawStatus else  Null end) L3869Status_Dk,
            max(case when du.InLawFlag=1 and du.UnderLawType='N3869' then du.SubmissionDate else  Null end) L3869SubmissionDate_Dk,
            max(case when du.InLawFlag=1 and du.UnderLawType='N3869' then du.TemporaryCourtDecisionDate else  Null end) L3869ProvisionalCourtDecisionDate_Dk,
            max(case when du.InLawFlag=1 and du.UnderLawType='N3869' then du.CourtDate else  Null end) L3869CourtDate_Dk,
            MAX(CASE
            WHEN du.InLawFlag = 1 AND du.CalcLawStatus = 'Positive Court Decision' THEN 1
            WHEN du.InLawFlag = 1 AND du.CalcLawStatus = 'Pending Court Decision' THEN 2
            WHEN du.InLawFlag = 0 AND du.LawStatus = 'Εκτός ν3869 - Απορριπτική Δικαστ. Απόφαση' THEN 11
            WHEN du.InLawFlag = 0 AND du.LawStatus = 'Εκτός ν3869 - Παραίτηση' THEN 12
            WHEN du.InLawFlag = 0 AND du.LawStatus = 'Εκτός ν3869 - Ματαίωση' THEN 13
            WHEN du.InLawFlag = 0 AND du.LawStatus = 'Εκτός ν3869 - Έκπτωση' THEN 14
            WHEN du.InLawFlag = 0 AND du.LawStatus = 'Εκτός Ν3869' THEN 15
        END
    ) AS L3869StatusMainBorrowerCode_Dk,

    MAX(CASE
            WHEN du.InLawFlag = 1
             AND du.UnderLawType = 'N3869'
             AND du.CourtDate < a.BusinessDate
                THEN 'Past Court Date'
            WHEN du.InLawFlag = 1
             AND du.UnderLawType = 'N3869'
             AND du.CourtDate >= a.BusinessDate
                THEN 'Not Past Court Date'
            ELSE NULL
        END
    ) AS L3869CourtDateCategoryMainBorrower_Dk,

    MAX(CASE
            WHEN du.InLawFlag = 1
             AND du.UnderLawType = 'N3869'
             AND du.InterbankStatus3869 IS NOT NULL
             AND du.InterbankStatus3869 NOT IN (
                    'N/A',
                    'Πελάτης Διατραπεζικής Ν3869 Follower',
                    'Πελάτης Διατραπεζικής Ν3869 Leader',
                    'Τειρεσίας - Παύση Εκκρεμοδικίας',
                    'Έξοδος από Διατραπεζική ν.3869'
                )
                THEN 1
            ELSE 0
        END
    ) AS L3869InterbankMainBorrower_Dk,

    MAX(a.ChannelCode) AS Assignmentchannel_Dk,
    MAX(d.L3869DraggedFlag) AS L3869DraggedFlag_Dk
    from exposure_loan_party_profits ex
    inner join dunderlaw du
      on du.CDI = ex.CDI
    LEFT JOIN assignment AS a
      ON  ex.PartyKey     = a.PartyKey
      AND ex.ExposureKey  = a.ExposureKey
      AND a.BusinessDate  = ex.BusinessDate
    LEFT JOIN dragged d
      on d.ExposureKey = ex.ExposureKey
      and ex.cdi = d.cdi

    group by Ex.ExposureKey

          """
)
dk_df7.createOrReplaceTempView("dk_df7")

dk_df7 = dk_df7.withColumnRenamed("ExposureKey", "ExposureKey_Dk_df7")

In [0]:
dk_df8 = spark.sql(f"""
            select distinct
            ex.ExposureKey
            ,rs.OldExposureKey RestructuredAccountNo_Dk
            ,case when pr.ProductDescription like '%ΥΠΟ ΑΙΡΕΣΗ%' then 'Conditional Woff' else 'Restructure' end as RestructureType_Dk
            from exposure_loan_profits ex
            inner join product_profits pr 
            on ex.ProductKey = pr.ProductKey 
            inner join restructuring_profits rs 
            on ex.ExposureKey = rs.NewExposureKey 
""")

dk_df8.createOrReplaceTempView("dk_df8")

dk_df8 = dk_df8.withColumnRenamed("ExposureKey", "ExposureKey_Dk_df8")




In [0]:
dk_df9 = spark.sql(f"""
            select ex1.ExposureKey,
            ex1.DuePrincipal PrincipalAtDefault_Dk,
            ex1.DueInterest InterestAtDefault_Dk,
            ex1.DueAmountCalc AmountDueAtDefault_Dk
            from exposure_ever_profits ex1
            inner join exposure_loan_profits ex2 
            on ex1.BusinessDate = ex2.Defaultdate
            and ex1.ExposureKey = ex2.ExposureKey
""")

dk_df9.createOrReplaceTempView("dk_df9")

dk_df9 = dk_df9.withColumnRenamed("ExposureKey", "ExposureKey_Dk_df9")


In [0]:
#We created this dataframe in order to calculate some fields that will be needed on the df12 & df13, so we won't join with this df in the end
dk_df10 = spark.sql(f"""
                SELECT 
                    ex.ExposureKey,
                    ex.CDI,
                    ex.LastBillingdate,
                    CASE WHEN (ex.Opendate >= DATE_FORMAT(date_trunc('MONTH', port.AssignmentDate), 'yyyy-MM-dd')
                    AND ex.Opendate <= DATE_FORMAT(last_day(port.AssignmentDate), 'yyyy-MM-dd'))
                    AND NULLIF(ex.LastRestructuringDate, DATE('1753-01-01')) < DATE_FORMAT(date_trunc('MONTH', port.AssignmentDate), 'yyyy-MM-dd')
                    THEN 'Unknown'
                    ELSE ex.LastRestructuringTypeCode 
                    END ModificationType_Dk,
                    CASE WHEN ex.Opendate >= DATE_FORMAT(date_trunc('MONTH', port.AssignmentDate), 'yyyy-MM-dd')
                    AND ex.Opendate <= DATE_FORMAT(last_day(port.AssignmentDate), 'yyyy-MM-dd')
                    AND NULLIF(ex.LastRestructuringDate, DATE('1753-01-01'))  < DATE_FORMAT(date_trunc('MONTH', port.AssignmentDate), 'yyyy-MM-dd')
                    THEN ex.Opendate
                    ELSE NULLIF(ex.LastRestructuringDate, DATE('1753-01-01'))  
                    END LastModDate_Dk
                FROM exposure_loan_party_profits ex
                left join portfolio port
                on ex.PortfolioKey = port.unit_code_profits
 
            
""")
 
dk_df10.createOrReplaceTempView("dk_df10")
dk_df10 = dk_df10.withColumnRenamed("ExposureKey", "ExposureKey_Dk_df10")

In [0]:
#We created this dataframe in order to calculate some fields that will be needed on the df12, so we won't join with this df in the end
dk_df11 = spark.sql(f"""
WITH StatusQuery AS (
    SELECT ex.exposurekey,
           du.CalcLawStatus,
           CASE WHEN du.CalcLawStatus = 'Positive Court Decision' THEN 1 ELSE 0 END AS L3869StatusFlag
      FROM exposure_loan_party_profits ex
           LEFT JOIN dunderlaw du ON du.CDI = ex.CDI
           where du.underlawtype ='N3869'
)
SELECT ExposureKey,
       MAX(L3869StatusFlag) AS L3869StatusFlag
  FROM StatusQuery
GROUP BY ExposureKey
""")                  
 
dk_df11.createOrReplaceTempView("dk_df11")
dk_df11 = dk_df11.withColumnRenamed("ExposureKey", "ExposureKey_Dk_df11")

In [0]:
#We created this dataframe in order to calculate some fields that will be needed on the df14, so we won't join with this df in the end
dk_df12 = spark.sql(f"""
            select 
            distinct 
            df10.ExposureKey,
 
                    case when df11.L3869StatusFlag = '1'
                    and coalesce(st.COR26Segment,df10.ModificationType_Dk,'') 
                    not in ('ΕΘΕΛΟΝΤΙΚΗ ΠΑΡΑΔΟΣΗ ΕΝΥΠΟΘΗΚΟΥ ΑΚΙΝΗΤΟΥ',
                    'Mortgage to Rent/Lease', 'Voluntary Sale of Property')
                    then 'ΔΙΑΧΕΙΡΙΣΗ ΜΕΣΩ ΔΙΑΔΙΚΑΣΙΑΣ ΠΤΩΧΕΥΣΗΣ'	
                    else coalesce(st.COR26Segment, df10.ModificationType_Dk) 
                    END AS LastModificationActionType1_Dk
            from dk_df10 df10
            left join dk_df11 df11
            on df11.ExposureKey = df10.ExposureKey
            left join staticreglookupvalue st
            on df10.ModificationType_Dk = st.LookupCategory
            and st.lookuptype ='COR_ModificationType'
""")
dk_df12.createOrReplaceTempView("dk_df12")  
dk_df12 = dk_df12.withColumnRenamed("ExposureKey", "ExposureKey_Dk_df12")

In [0]:
dk_df13 = spark.sql(f"""
                    select distinct df10.ExposureKey,
                    df10.LastModDate_Dk as LastModificationDate_Dk
                    from dk_df10 df10
""")
 
dk_df13.createOrReplaceTempView("dk_df13")
dk_df13 = dk_df13.withColumnRenamed("ExposureKey", "ExposureKey_Dk_df13")

In [0]:
dk_df14 = spark.sql(f"""
            select 
            distinct 
            df12.ExposureKey,
 
                    CASE WHEN df12.LastModificationActionType1_Dk = 'Unknown' 
                    THEN df12.LastModificationActionType1_Dk 
                    ELSE dim.ReschedulingFacilities
                    END LastModificationActionType_Dk
            from dk_df12 df12
            left join dimpeerescheduling dim
            on dim.PEEModificationType = df12.LastModificationActionType1_Dk
""")
dk_df14.createOrReplaceTempView("dk_df14")
dk_df14 = dk_df14.withColumnRenamed("ExposureKey", "ExposureKey_Dk_df14")

In [0]:
dk_df16 = spark.sql(f"""
        select  ex.ExposureKey,
            max(OperationalLegalStatusLevel3 ) LegalProcessStatus_Dk,
            max(LegalProcessStatusLastChangeDate ) LegalProcessStatusDate_Dk
        from exposure_loan_party_profits ex
        inner join dlegalprocessstatusagreement dl
        on dl.AgreementKey = ex.AgreementKey
        group by Ex.ExposureKey
          """)
dk_df16.createOrReplaceTempView("dk_df16")

dk_df16 = dk_df16.withColumnRenamed("ExposureKey", "ExposureKey_Dk_df16")

In [0]:
dk_df17 = spark.sql(f"""
        select  ex.ExposureKey,
            max(CountyCourt) as CourtRegion_Dk
        from exposure_loan_profits ex
        inner join dunderlaw du
        on du.CDI = ex.CDI
        group by ex.ExposureKey
          """)
dk_df17.createOrReplaceTempView("dk_df17")

dk_df17 = dk_df17.withColumnRenamed("ExposureKey", "ExposureKey_Dk_df17")

In [0]:
exposure_loan_profits2 = exposure_loan_profits.withColumnRenamed("ExposureKey", "ExposureKey_profits")

Dk_Final = exposure_loan_profits2.join(
    dk_df, exposure_loan_profits2["ExposureKey_profits"] == dk_df["ExposureKey_Dk_df"],
    "left"
).join(
    dk_df3, exposure_loan_profits2["ExposureKey_profits"] == dk_df3["ExposureKey_Dk_df3"],
    "left"
).join(
    dk_df4, exposure_loan_profits2["ExposureKey_profits"] == dk_df4["ExposureKey_Dk_df4"],
    "left"
).join(
    dk_df5, exposure_loan_profits2["ExposureKey_profits"] == dk_df5["ExposureKey_Dk_df5"],
    "left"
).join(
    dk_df6, exposure_loan_profits2["ExposureKey_profits"] == dk_df6["ExposureKey_Dk_df6"],
    "left"
).join(
    dk_df7, exposure_loan_profits2["ExposureKey_profits"] == dk_df7["ExposureKey_Dk_df7"],
    "left"
).join(
    dk_df9, exposure_loan_profits2["ExposureKey_profits"] == dk_df9["ExposureKey_Dk_df9"],
    "left"
).join(
    dk_df14, exposure_loan_profits2["ExposureKey_profits"] == dk_df14["ExposureKey_Dk_df14"],
    "left"
).join(
    dk_df13, exposure_loan_profits2["ExposureKey_profits"] == dk_df13["ExposureKey_Dk_df13"],
    "left"
).join(
    dk_df16, exposure_loan_profits2["ExposureKey_profits"] == dk_df16["ExposureKey_Dk_df16"],
    "left"
).join(
    dk_df17, exposure_loan_profits2["ExposureKey_profits"] == dk_df17["ExposureKey_Dk_df17"],
    "left"
)
 
Dk_Final =Dk_Final.withColumn(
                "ActiveModificationProgramStatus_Dk",
                when((~dk_df14.LastModificationActionType_Dk.isNull())
& (~exposure_loan_profits2.ExposureStatus.isin ('Define Delay','Writen Off','ΕΠΙΣΦΑΛΕΙΑ','ΟΡΙΣΤΙΚΗ ΚΑΘΥΣΤΕΡΗΣΗ'))
& exposure_loan_profits2.IsActiveFlag == True,'true'
                ).when((dk_df14.LastModificationActionType_Dk.isNotNull())
& ((exposure_loan_profits2.ExposureStatus.isin ('Define Delay','Writen Off','ΕΠΙΣΦΑΛΕΙΑ','ΟΡΙΣΤΙΚΗ ΚΑΘΥΣΤΕΡΗΣΗ'))
                     | (exposure_loan_profits2.IsActiveFlag == False)),'false'
                ).otherwise(lit(None))
)
 
 
Dk_Final = Dk_Final.withColumnRenamed("ExposureKey_profits", "ExposureKey_Dk_Final")

 
all_columns = Dk_Final.columns
 
columns_to_keep = [col for col in all_columns if "_Dk" in col]
 
Dk_Final = Dk_Final.select(columns_to_keep)
 
final_columns=Dk_Final.columns
 
columns_to_keep2=[col for col in final_columns if "_df" not in col]
 
Dk_Final = Dk_Final.select(columns_to_keep2)
 
Dk_Final.createOrReplaceTempView("Dk_Final")

## First transformations (profits)

In [0]:
exposure = spark.sql(f'''
    SELECT e.*,
        dlps_agr.LPSCode,
        dlps_agr.LPS,
        dlps_agr.LPSDate,
        cast(NULL as string) as LPSEng
    FROM `{env}-cdh`.primary.Exposure e
    LEFT JOIN (
        SELECT DISTINCT AgreementKey, CDI, BusinessDate,
            cast(OperationalLegalStatusLevel3Code as string) as LPSCode,
            OperationalLegalStatusLevel3 as LPS,
            cast(LegalProcessStatusLastChangeDate as string) as LPSDate
        FROM `{env}-cdh`.derivativeactive.dlegalprocessstatusagreementdebtor
        WHERE BusinessDate = {run_date!r}
    ) dlps_agr
        ON dlps_agr.AgreementKey = e.AgreementKey
        AND dlps_agr.CDI = e.CDI
        AND dlps_agr.BusinessDate = e.BusinessDate
    WHERE e.BusinessDate = {run_date!r}
''')

exposure.createOrReplaceTempView("exposure")



exposure_ever.createOrReplaceTempView("exposure_ever")

exposure_ever_max_active  = spark.sql("SELECT ExposureKey, SourceSystem, max(BusinessDate) as MaxActiveDate "
                                      "FROM exposure_ever where IsActiveFlag = '1' " 
                                      "group by ExposureKey, SourceSystem ")

exposure_ever_max_active.createOrReplaceTempView("exposure_ever_max_active")
exposure_ever_min_nonactive  = spark.sql("SELECT exposure_ever.ExposureKey as ExposureKeyNonActive, exposure_ever.SourceSystem,"
                                         " min(exposure_ever.BusinessDate) AS ClosedDate FROM exposure_ever " 
                                         "INNER JOIN  exposure_ever_max_active ON exposure_ever_max_active.ExposureKey = exposure_ever.ExposureKey "
                                         "AND exposure_ever_max_active.SourceSystem = exposure_ever.SourceSystem "
                                         "where exposure_ever.IsActiveFlag = '0' and exposure_ever.BusinessDate> exposure_ever_max_active"
                                         ".MaxActiveDate "
                                         "group by exposure_ever.ExposureKey, exposure_ever.SourceSystem")  




exposure_profits = exposure.filter((exposure.SourceSystem == 'profits') & (exposure.PrimaryPartyFlag == True))
exposure_profits = (exposure_profits.withColumn(
                "InterestRateCalc",
                (coalesce(exposure_profits.BaseRate,lit(0)) +
                coalesce(exposure_profits.SpreadRate,lit(0)) +
                coalesce(exposure_profits.N128Rate,lit(0)))
                ).withColumn(
                "DueAmountCalc",
                coalesce(exposure_profits.DuePrincipal, lit(0).cast("decimal(19,2)"))
                + coalesce(exposure_profits.Dueinterest, lit(0).cast("decimal(19,2)"))
                + coalesce(
                    exposure_profits.DueUnrealizedinterest, lit(0).cast("decimal(19,2)")
                )
                + coalesce(exposure_profits.Penaltyinterest, lit(0).cast("decimal(19,2)"))
                + coalesce(
                    exposure_profits.PenaltyUnrealizedinterest, lit(0).cast("decimal(19,2)")
                )
                + coalesce(exposure_profits.DueCommission, lit(0).cast("decimal(19,2)"))
                + coalesce(exposure_profits.DueExpense, lit(0).cast("decimal(19,2)"))
                + coalesce(exposure_profits.DueAccrued, lit(0).cast("decimal(19,2)")),
            )
)
exposure_profits.createOrReplaceTempView("exposure_profits") 
product_profits = Product.filter(Product.SourceSystem == 'profits')    


PaymentSchedule_min = spark.sql(
    "select ExposureKey, min(InstallmentDate) as InstallmentDateMin "
    "from PaymentSchedule "
    "where 1=1 "
    "and InstallmentDate > BusinessDate "
    "and InterestAmount > 0 "
    "and InstallmentStatus = 'Active' "
    "and ((SourceSystem = 'profits' and InstallmentType in ('Installment', 'Interest Request')) "
    "or (SourceSystem = 'qca' or SourceSystem = 'process')) "
    "group by ExposureKey, BusinessDate"
)

SettlementExposure.createOrReplaceTempView("SettlementExposure")

SettlementExposure_Bucket_qca = spark.sql("select BucketQCA, ExposureKey as ExposureKey_seb "
                              "from "
                              "    ("
                              "	select SettlementExposure.Bucket as BucketQCA, exposure.ExposureKey, "
                              "	row_number() over (partition by exposure.ExposureKey order by SubmissionDate desc) as rown "
                              "	from SettlementExposure "
                              " inner join  exposure_profits exposure "
                              " on substring(exposure.exposurekey,4,50) = SettlementExposure.exposurekey "
                              "	where 1=1 "
                              "	and SettlementExposure.SourceSystem='qca' "
                              "	and SettlementType!='Νόμος 3869 – Προσωρινή Διαταγή' "
                              "	and SettlementStatus IN ('Running','Partially Fulfilled') "
                              "    )a "
                              "    where rown=1")

SettlementExposure_Bucket_profits = spark.sql(" select SettlementExposure.Bucket as BucketProfits, ExposureKey, SettlementStatus "
                                    "  from SettlementExposure "
                                    "  where 1=1 "
                                    "  and SourceSystem='profits'  "
                                    "  and IsActiveFlag=1 "
                                    "  and isPrimarySettlementFlag = True ")

AccountTransaction.createOrReplaceTempView("AccountTransaction")

AccountTransaction_Write_Off = spark.sql(
                        "        SELECT "
                        "        ExposureKey"
                        "        ,TransactionDate as WriteOffDate"
                        "        ,SUM (coalesce(PenaltyInterestAmount,0) + coalesce(InterestAmount,0)"
                        "         + coalesce(CommisionAmount,0) + coalesce(ExpenseAmount,0) +coalesce( CapitalAmount,0))  AS WriteOffAmount"
                        "    FROM AccountTransaction acc"
                        "    WHERE JustificationCode IN ('43006','43031')"
                        "    AND   ReversedFlag = '0'"
                        "    and TransactionComments  not like '%TUN%'  "
                        "    and SourceSystem='profits'"
                        "    AND AccountType='exposure' "
                        "    AND   (coalesce(PenaltyInterestAmount,0) + coalesce(InterestAmount,0)"
                        "    + coalesce(CommisionAmount,0) + coalesce(ExpenseAmount,0) +coalesce( CapitalAmount,0)) > 0"
                        "    GROUP BY ExposureKey,TransactionDate"
                        )




AccountTransaction_Write_Off.createOrReplaceTempView("AccountTransaction_Write_Off")

AccountTransaction_Write_Off_non_cond = spark.sql(
                        " SELECT ExposureKey"
                        " ,SUM(WriteOffAmount) AS WriteOffAmountSum"
                        " ,min(WriteOffDate) AS TransactionDateMin"
                        " FROM AccountTransaction_Write_Off wroff"
                        " GROUP BY ExposureKey"
)

exposure_ever_min_status_4  = spark.sql(
                                    "    SELECT ExposureKey ExposureKey_min_status_4 "
                                    "            ,min(BusinessDate) AS MinDate"
                                    "        FROM exposure_ever ex"
                                    "        where ex.ExposureStatusCode='4' "
                                    "        and SourceSystem = 'profits' "
                                    "        GROUP BY exposurekey "
                                    )		

Product.createOrReplaceTempView("Product")
product_profits.createOrReplaceTempView("product_profits")

Restructuring.createOrReplaceTempView("Restructuring")
params_loan_product_mod.createOrReplaceTempView("params_loan_product_mod")


Conditional_Write_off_Date = spark.sql (
                            "      select  min(Opendate) as CondWriteOffDate "
                            "          ,OldExposureKey as ExposureKey_Wof_D"
                            "  from product_profits prod"
                            "  inner join ( select distinct ex.ProductCode,ex.ExposureKey,ex.AgreementKey,ex.Opendate,re.OldExposureKey"
                            "              from Restructuring re"
                            "              inner join exposure_profits ex"
                            "              on re.NewExposureKey=ex.ExposureKey"
                            "              where re.sourcesystem='profits'"
                            "          ) rest"
                            "  on prod.ProductCode=rest.ProductCode"
                            "  left join params_loan_product_mod mod"
                            "     on      prod.ProductCode=mod.prft_product_code"
                            "  and mod.Modification_type='CL-woff'"
                            "  group by OldExposureKey"
 
                    )
 
Conditional_Write_off_Amount = spark.sql (    
                        "    select  sum(exOpendate.OutstandingTotalBalance) as CondWriteOffAmount"
                        "    ,OldExposureKey as ExposureKey_Wof_A"
                        "    from product_profits prod"
                        "    INNER JOIN ( select distinct ex.ProductCode,ex.ExposureKey,ex.AgreementKey,ex.Opendate,re.OldExposureKey"
                        "                from Restructuring re"
                        "                inner join exposure_profits ex"
                        "                on re.NewExposureKey=ex.ExposureKey"
                        "                where re.sourcesystem='profits'"
                        "                and ex.SourceSystem='profits'"
                        "            ) rest"
                        "    on prod.ProductCode=rest.ProductCode"
                        "    LEFT JOIN params_loan_product_mod mod"
                        "    on      prod.ProductCode=mod.prft_product_code"
                        "    LEFT JOIN exposure_profits exOpendate"
                        "    on exOpendate.ExposureKey=rest.ExposureKey"
                        "    AND exOpendate.BusinessDate=rest.Opendate"
                        "    and mod.Modification_type='CL-woff'"
                        "    group by OldExposureKey"
                        )

Conditional_Writeoff_Product_Keys = spark.sql("""
                                                            SELECT DISTINCT ProductKey, ProductDescription
                                                            FROM product_profits
                                                            WHERE 1=1
                                                            AND ProductDescription ILIKE '%ΥΠΟ ΑΙΡΕΣΗ%'
                                                        """)

Conditional_Writeoff_Product_Keys.createOrReplaceTempView("Conditional_Writeoff_Product_Keys")


Payment_Schedule_Loan_Next_Install = spark.sql ( 
                "    select ExposureKey as ExposureKeyNextInstall"
                "    ,min(CASE WHEN InstallmentAmount >0 THEN InstallmentDate END) AS NextInstallmentLoanDate" 
                "    ,SUM(InstallmentAmount) AS NextInstallmentAmount" 
                "    from ("
                "        select InstallmentAmount, InstallmentDate, Exposurekey, Businessdate," 
                "        ROW_NUMBER() OVER (PARTITION BY ExposureKey ORDER BY InstallmentDate asc) as InstallmentRank" 
                "    from PaymentSchedule" 
                "    where InstallmentStatus='Active'" 
                "    and InstallmentDate > BusinessDate"
                "    ) a" 
                "    where InstallmentRank=1" 
                "    group by ExposureKey, BusinessDate"
                )


Payment_Schedule_Loan_Last_Install = spark.sql ( 
                "    select ExposureKey as ExposureKeyLastInstall"
                "    ,max(CASE WHEN InstallmentAmount >0 THEN InstallmentDate END) AS LastInstallmentLoanDate"
                "    ,SUM(InstallmentAmount) AS LastInstallmentAmount"
                "    from (select InstallmentAmount, InstallmentDate, Exposurekey, Businessdate, "
                "    ROW_NUMBER() OVER (PARTITION BY ExposureKey ORDER BY InstallmentDate asc) as InstallmentRank "
                "    from PaymentSchedule "
                "    WHERE InstallmentStatus = 'Active' "
                "    AND InstallmentDate < BusinessDate"
                "    ) a "
                "    where InstallmentRank=1 "
                "    group by ExposureKey,BusinessDate "
)


Payment_Schedule_Loan_Next_Install.createOrReplaceTempView("Payment_Schedule_Loan_Next_Install")
Payment_Schedule_Loan_Last_Install.createOrReplaceTempView("Payment_Schedule_Loan_Last_Install")
 
                                       
Payment_Schedule_Loan_Capital = spark.sql ( 
                    "    select ExposureKey"
                    "    ,MIN(CASE WHEN CapitalAmount>0 THEN InstallmentDate END)	AS NextInstallmentCapitalLoanDate"
                    "    ,SUM(CASE WHEN CapitalAmount>0 THEN CapitalAmount END)  AS NextInstallmentCapitalLoanAmnt"
                    "    from PaymentSchedule"
                    "    where 1=1"
                    "    and InstallmentStatus='Active'"
                    "    and InstallmentDate>BusinessDate"
                    "    group by ExposureKey"
                )
Payment_Schedule_Loan_Capital.createOrReplaceTempView("Payment_Schedule_Loan_Capital")

Payment_schedule_next_capital_installment = spark.sql (
                " select distinct ex.ExposureKey as ExposureKey_nci,"
                " cast(pmnt.NextInstallmentCapitalLoanDate as date)"
                " AS NextInstallmentCapitalDate,"
                " pmnt.NextInstallmentCapitalLoanAmnt "
                " AS NextInstallmentCapitalAmount"
                " from exposure_profits ex"
                " left join Payment_Schedule_Loan_Capital pmnt"
                "     on pmnt.ExposureKey=ex.ExposureKey"
                )    
 
Payment_Schedule_Loan_Capital_Interest = spark.sql('''
                    select ExposureKey 
    					,MIN(CASE WHEN InterestAmount>0 THEN InstallmentDate END)	AS NextInstallmentInterestLoanDate 
    					,MIN(CASE WHEN CapitalAmount>0	THEN InstallmentDate END)	AS NextInstallmentCapitalLoanDate 
     	            from PaymentSchedule	 
    	            where InstallmentStatus='Active'
                    and InstallmentDate>BusinessDate
    	            group by ExposureKey
                                                   '''
                )

Payment_Schedule_Loan_Capital_Interest.createOrReplaceTempView("Payment_Schedule_Loan_Capital_Interest")

Payment_schedule_next_capital_interest_installment = spark.sql(
                "    select  distinct ex.ExposureKey as ExposureKey_ncii,"
                "            CASE  WHEN pmnt.NextInstallmentInterestLoanDate IS NOT NULL THEN CAST(pmnt.NextInstallmentInterestLoanDate AS DATE)"
                "            ELSE  "
                "            CASE  WHEN MechanismCode='4'  THEN CAST(pmnt.NextInstallmentCapitalLoanDate AS DATE) " # Amortised Loan
                "                  WHEN ex.InterestRateCalc>0 AND MechanismCode IN ('2','3','5') AND prd.NextValeurDate>=prd.BusinessDate " #Revolving Loan,Revolving Loan with Promisory Notes,Amortised Loan (Fixed Capital)
                "                  THEN prd.NextValeurDate "
                "            ELSE NULL END"
                "    END AS NextInstallmentInterestDate"
                "    from exposure_profits ex"
                "    left join Payment_Schedule_Loan_Capital_Interest pmnt"
                "        on pmnt.ExposureKey=ex.ExposureKey"
                "    left join product_profits prd"
                "        on prd.ProductKey=ex.ProductKey"
                )



LegalAction.createOrReplaceTempView("LegalAction")

LegalAction_po = spark.sql( '''
                    select * 
                    from (            	
                        select ExposureKey as ExposureKey_lapo, LegalActionKey
                         , ROW_NUMBER() over(partition by ExposureKey order by LegalActionDate ASC, LegalActionKey) rnk
                         ,LegalActionDate
                     from LegalAction
                         where LegalActionTypeCode IN ('1205','1235','905') 
                            and LegalAction.SourceSystem = 'qca'
                    )
                    where rnk = 1
                    '''
)

In [0]:
print(Restructuring.columns)

In [0]:
# Special calculation of the IsSecuredFlag for EUCLID
LastBusinessDateOfPreviousQuarter = spark.sql(f'''
    SELECT CASE
      WHEN IsBusinessDay = 1 THEN CalendarDate
      ELSE PreviousBusinessDay
    END AS LastBusinessDateOfPreviousQuarter
    FROM `{env}-cdh`.reference.ref_calendar
    WHERE CalendarDate = date_add({run_date!r}, -1 * cast((
        SELECT DayOfQuarter
        FROM `{env}-cdh`.reference.ref_calendar
        WHERE CalendarDate = {run_date!r}
        ) as int) -- Subtract the number of days in this quarter to reach the last day of the previous quarter
    )
    ''').first()[0]

danalyticsmetricsdate = spark.sql(f'''
    SELECT distinct BusinessDate
    FROM `{env}-cdh`.derivative.danalyticsmetrics
    WHERE PortfolioName='EUCLID'  
''')

danalyticsmetricsdate.createOrReplaceTempView("danalyticsmetricsdate")

IsSecuredFlagEuclidMinDate = spark.sql(f'''
    SELECT min(BusinessDate) IsSecuredFlagEuclidMinDate
    FROM danalyticsmetricsdate
    WHERE BusinessDate > '{LastBusinessDateOfPreviousQuarter}'
    ''').first()[0]

danalyticsmetricsEndOfPreviousQuarter = danalyticsmetricsdate.filter(
    danalyticsmetricsdate.BusinessDate == LastBusinessDateOfPreviousQuarter
)

def get_secured_agreements_query(date):
    return spark.sql(f'''
        SELECT distinct AgreementKey
        FROM `{env}-cdh`.derivative.danalyticsmetrics
        WHERE businessdate = '{date}'
        AND AgreementKey IN (
            SELECT distinct AgreementKey
            FROM `{env}-cdh`.primary.collateral 
            WHERE sourcesystem = 'thesis' 
            AND collateralstatus = 'Ενεργή'
            AND businessdate = '{date}'
        )
        GROUP BY AgreementKey
        HAVING SUM(CREVBaseTLC) > 20000
    ''')

if not danalyticsmetricsEndOfPreviousQuarter.isEmpty():
    SecuredAgreements = get_secured_agreements_query(LastBusinessDateOfPreviousQuarter)
else:    
    SecuredAgreements = get_secured_agreements_query(IsSecuredFlagEuclidMinDate)

SecuredAgreements.createOrReplaceTempView("SecuredAgreements")

IsSecuredFlagEuclid = spark.sql(f'''
    SELECT AgreementKey,
    CASE
        WHEN AgreementKey IN (SELECT AgreementKey FROM SecuredAgreements) THEN True
        ELSE False
    END AS is_secured_flag
FROM exposure
WHERE PortfolioName='EUCLID'
''')
IsSecuredFlagEuclid = IsSecuredFlagEuclid.filter(IsSecuredFlagEuclid.is_secured_flag == True).distinct()

In [0]:
# Special calculation of the IsSecuredFlag for HAPS Retail portfolios

#Agreements with no active collateral
dcollateral_notactive= spark.sql(f'''
select agreementkey
from collateral
where 0=(
select count(1)
from  collateral dc
where dc.SourceSystem=collateral.SourceSystem and dc.RealEstateCollateralStatusCode='Ενεργή'
and dc.businessdate=collateral.BusinessDate and dc.AgreementKey=collateral.AgreementKey
)and collateral.SourceSystem='thesis' and collateral.portfolioname in ('Orion X Securitisation DAC' , 'Galaxy II Funding DAC', 'Cosmos Retail', 'GAIA II PORTFOLIO WHOLESALE', 'DOMUS PORTFOLIO - RETAIL','DOMUS PORTFOLIO - WHOLESALE', 'RHODIUM PORTFOLIO - RETAIL', 'RHODIUM PORTFOLIO - WHOLESALE', 'GAIA I PORTFOLIO', 'GAIA II PORTFOLIO RETAIL')
''')

#Agreements with zero LV 
dmetrics_zeroLV= spark.sql(f'''
select AgreementKey
from  `{env}-cdh`.derivative.danalyticsmetrics
where businessdate={run_date!r} and portfolioname in ('Orion X Securitisation DAC' , 'Galaxy II Funding DAC', 'Cosmos Retail', 'GAIA II PORTFOLIO WHOLESALE', 'DOMUS PORTFOLIO - RETAIL','DOMUS PORTFOLIO - WHOLESALE', 'RHODIUM PORTFOLIO - RETAIL', 'RHODIUM PORTFOLIO - WHOLESALE', 'GAIA I PORTFOLIO', 'GAIA II PORTFOLIO RETAIL')
group by danalyticsmetrics.AgreementKey
having sum(coalesce(LVNetBase,0))=0                              
''')

# Agreements with CREV < 10K and Crev Coverage <10%
exposure_TLC= spark.sql(f'''
select AgreementKey, sum(try_cast(OutstandingTotalBalance as DECIMAL(20, 2))) as TLC
from exposure_profits ex
where portfolioname in ('Orion X Securitisation DAC' , 'Galaxy II Funding DAC', 'Cosmos Retail', 'GAIA II PORTFOLIO WHOLESALE', 'DOMUS PORTFOLIO - RETAIL','DOMUS PORTFOLIO - WHOLESALE', 'RHODIUM PORTFOLIO - RETAIL', 'RHODIUM PORTFOLIO - WHOLESALE', 'GAIA I PORTFOLIO', 'GAIA II PORTFOLIO RETAIL')
group by AgreementKey                             
''')

dmetrics_CREVBaseTLC= spark.sql(f'''
select AgreementKey, sum(coalesce(CREVBaseTLC,0)) as TotalCREVBaseTLC
from `{env}-cdh`.derivative.danalyticsmetrics
where businessdate={run_date!r} and portfolioname in ('Orion X Securitisation DAC' , 'Galaxy II Funding DAC', 'Cosmos Retail', 'GAIA II PORTFOLIO WHOLESALE', 'DOMUS PORTFOLIO - RETAIL','DOMUS PORTFOLIO - WHOLESALE', 'RHODIUM PORTFOLIO - RETAIL', 'RHODIUM PORTFOLIO - WHOLESALE', 'GAIA I PORTFOLIO', 'GAIA II PORTFOLIO RETAIL')
group by AgreementKey
having sum(coalesce(CREVBaseTLC,0))<10000
''')

dmetrics_join = exposure_TLC.join(dmetrics_CREVBaseTLC, on='AgreementKey', how='left') \
    .withColumn('CrevCoverage', try_divide(dmetrics_CREVBaseTLC.TotalCREVBaseTLC, exposure_TLC.TLC)) \
    .select(exposure_TLC.AgreementKey, exposure_TLC.TLC, dmetrics_CREVBaseTLC.TotalCREVBaseTLC, 'CrevCoverage')
dmetrics_CrevCoverage=dmetrics_join.filter(dmetrics_join.CrevCoverage<0.10).select(dmetrics_join.AgreementKey)

# Perform the union of the three DataFrames
union_df = dcollateral_notactive.union(dmetrics_zeroLV).union(dmetrics_CrevCoverage)

Haps_UnsecuredAgreements = union_df.select('agreementkey').distinct().withColumn('is_secured_flag', lit(False))

dcollateral_active= spark.sql(f'''
select distinct ex.agreementkey
from exposure_profits ex
inner join collateral on collateral.AgreementKey=ex.AgreementKey
where RealEstateCollateralStatusCode='Ενεργή'
and collateral.SourceSystem='thesis' and ex.portfolioname in ('Orion X Securitisation DAC' , 'Galaxy II Funding DAC', 'Cosmos Retail', 'GAIA II PORTFOLIO WHOLESALE', 'DOMUS PORTFOLIO - RETAIL','DOMUS PORTFOLIO - WHOLESALE', 'RHODIUM PORTFOLIO - RETAIL', 'RHODIUM PORTFOLIO - WHOLESALE', 'GAIA I PORTFOLIO', 'GAIA II PORTFOLIO RETAIL')
''')

# Step 1: Perform a left anti join to find agreementkeys in dcollateral_active but not in Haps_UnsecuredAgreements

secured_agreements = dcollateral_active.join(
    Haps_UnsecuredAgreements,
    on='agreementkey',
    how='left_anti'
)

# Step 2: Set is_secured_flag = True for those agreements
secured_agreements_with_flag = secured_agreements.withColumn('is_secured_flag', lit(True))

# Step 3: Combine with the original Haps_UnsecuredAgreements (which already has is_secured_flag = False)
Haps_Final_UnsecuredAgreements = Haps_UnsecuredAgreements.union(secured_agreements_with_flag)


In [0]:
# Special calculation of the IsSecuredFlag for MONZA

is_secured_flag_Monza = spark.sql(f'''
-- Retrieve the latest reference date from the servicer_report_securedperimeter table
WITH latest_servicer_report_securedperimeter as (
    SELECT max(ReferenceDate) as MaxReferenceDate
    FROM `{env}-reportinputfiles`.investor.servicer_report_securedperimeter
    WHERE ReferenceDate <= {run_date!r}
)

        SELECT
            DebtorCDI,
            CASE 
                WHEN SecUnsec = 'Unsecured D' THEN FALSE
                WHEN SecUnsec = 'Secured D' THEN TRUE
            END AS IsSecuredFlag,
            ReferenceDate
        FROM `{env}-reportinputfiles`.investor.servicer_report_securedperimeter sec_per
        JOIN latest_servicer_report_securedperimeter lsr 
            ON sec_per.ReferenceDate = lsr.MaxReferenceDate
        WHERE PortfolioName ILIKE '%monza%'

''')

In [0]:
Law_profits = spark.sql('''  select   ExposureKey as ExposureKey_law ,CalcLawStatus as LawStatus,CalcLawStatusCode as LawStatusCode,UnderLawType,UnderLawTypeCode, 
                             cast(ApplicationStatusDate as date) as LawApplicationStatusDate,cast(LawProvisionalCourtDecisionDate as date),cast(LawCourtDate as date)
                    from (
                    
                    select temp1.*,temp2.CDI,temp2.PartyRelationshipType
                    ,rank() over (partition by temp1.ExposureKey,temp1.PortfolioName,temp1.ApplicationStatusDate order by 
                            CASE    WHEN temp2.PartyRelationshipType='Primary Owner' then 1
                                    WHEN temp2.PartyRelationshipType='Guarantor' then 2
                                    WHEN temp2.PartyRelationshipType='Co-Owner' then 3
                                    ELSE 4 END
                                    ,CDI) as rnk
                    from (
                        select distinct temp1.ExposureKey
                        ,temp1.PortfolioName
                        ,temp1.CalcLawStatus
                        ,temp1.CalcLawStatusCode
                        ,temp1.UnderLawType
                        ,temp1.UnderLawTypeCode
                        ,min(temp1.ApplicationStatusDate) as ApplicationStatusDate
                        ,case when temp1.CalcLawStatus='Pending Court Decision'   then max(temp1.CourtDate) end as LawProvisionalCourtDecisionDate
                        ,case when temp1.CalcLawStatus='Positive Court Decision'  then max(temp1.CourtDate) end as LawCourtDate
                        from (
                        
                            select distinct ex.ExposureKey
                            ,ex.CDI
                            ,ex.PortfolioName
                            ,ex.PartyRelationshipType
                            ,row_number() OVER (PARTITION BY ex.ExposureKey,ex.CDI ORDER BY 
                            CASE    WHEN ex.PartyRelationshipType='Primary Owner' then 1
                                    WHEN ex.PartyRelationshipType='Guarantor' then 2
                                    WHEN ex.PartyRelationshipType='Co-Owner' then 3
                                    ELSE 4 END ) rn
                            ,law.CalcLawStatus
                            ,law.CalcLawStatusCode
                            ,law.UnderLawType
                            ,law.UnderLawTypeCode
                            ,law.ApplicationStatusDate
                            ,law.CourtDate
                            ,rank() over (partition by ex.ExposureKey order by coalesce(CalcLawStatus,'a') desc) as rnk
                            from exposure ex
                            left join dunderlaw law
                            on law.CDI=ex.CDI
                            where 1=1
                            and law.CalcLawStatus IS NOT NULL
                            and ex.SourceSystem='profits'
                        )temp1
                        where 1=1
                        and temp1.rnk=1
                        and temp1.rn=1
                        group by temp1.ExposureKey,temp1.PortfolioName,temp1.CalcLawStatus,temp1.UnderLawType,temp1.CalcLawStatusCode,temp1.UnderLawTypeCode
                    )temp1
                    
                        inner join (
                            select distinct ex.ExposureKey
                            ,ex.CDI
                            ,ex.PortfolioName
                            ,ex.PartyRelationshipType
                            ,law.ApplicationStatusDate
                            ,case when law.CalcLawStatus='Pending Court Decision'   then law.CourtDate end as LawProvisionalCourtDecisionDate
                            ,case when law.CalcLawStatus='Positive Court Decision'  then law.CourtDate end as LawCourtDate
                            ,row_number() OVER (PARTITION BY ex.ExposureKey,ex.CDI ORDER BY 
                            CASE    WHEN ex.PartyRelationshipType='Primary Owner' then 1
                                    WHEN ex.PartyRelationshipType='Guarantor' then 2
                                    WHEN ex.PartyRelationshipType='Co-Owner' then 3
                                    ELSE 4 END ) rn
                            from exposure ex
                            left join dunderlaw law
                            on law.CDI=ex.CDI
                            where 1=1
                            and law.CalcLawStatus IS NOT NULL
                            and ex.SourceSystem='profits'
                        )temp2
                        on temp2.ExposureKey=temp1.ExposureKey
                        and temp2.ApplicationStatusDate=temp1.ApplicationStatusDate
                        and temp2.PortfolioName=temp1.PortfolioName
                        and temp2.rn=1
                    
                    )a
                    where a.rnk=1'''
)

exposure_ever_extra.createOrReplaceTempView("exposure_ever_extra")

exposure_ever_extra_judicial = spark.sql( '''
        select * 
        FROM (
        select ex.ExposureKey as ExposureKey_ee_judicial ,ex.LPS,ex.LPSCode,ex.LPSDate,ex.LPSEng, row_number() over(partition by ex.ExposureKey order by ex.BusinessDate asc) rnk
    	from exposure_ever_extra ex
    	    where ex.SourceSystem='qc'
    	    and (ex.LPSEng like 'Auction%' or ex.LPSEng like 'PMT Order%' or ex.LPSEng in ('Foreclosure', 'Extrajudicial Delivered', 'Forced Prenotation'))
        )
        WHERE rnk = 1
'''
)

LegalAction_judicial =spark.sql (   '''
        select distinct LegalActionKey,ExposureKey as ExposureKey_la_judicial ,LegalActionDate,ExtraJudicialLetterFlag,SourceSystem
    	from (
    	    select distinct LegalActionKey,ExposureKey,SourceSystem
    	    ,ROW_NUMBER() over(partition by ExposureKey order by LegalActionDate ASC, LegalActionKey) rnk
    	    ,cast(LegalActionDate as date)
    	    ,1 as ExtraJudicialLetterFlag  
    	    
            from LegalAction
    	        where LegalActionTypeCode IN ('1051','2401','2402', '2403','2404','2405', '2407','2408','2410', '2414','2415','5002407','5103', '5104','5105','5564','5654')
    	        and LegalAction.SourceSystem = 'qca'
        )a 
    	where a.rnk=1
        
    '''
)
exposure_non_active_profits = exposure_ever_min_nonactive.filter(exposure_ever_min_nonactive.SourceSystem == 'profits').distinct()
exposure_non_active_arotron = exposure_ever_min_nonactive.filter(exposure_ever_min_nonactive.SourceSystem == 'arotron').distinct()
exposure_non_active_qca = exposure_ever_min_nonactive.filter(exposure_ever_min_nonactive.SourceSystem == 'qca').distinct()

##Play Profits

In [0]:


exposure_adjustmentstepup=spark.sql(f'''
                    SELECT * 
                    FROM `{env}-cdh`.primary.exposureadjustmentstepup
                    WHERE BusinessDate = {run_date!r} and SourceSystem = "profits"

''')
exposure_adjustmentstepup.createOrReplaceTempView("exposure_adjustmentstepup")

exposure_adjustmentperiod = spark.sql(f'''
                    SELECT * 
                    FROM `{env}-cdh`.primary.ExposureAdjustmentPeriod
                    WHERE BusinessDate = {run_date!r} and SourceSystem = "profits"

''')
exposure_adjustmentperiod.createOrReplaceTempView("exposure_adjustmentperiod")


StepUp = spark.sql(f"""
WITH joined AS (
  SELECT
      exa.exposurekey                                AS stepexposurekey,
      exa.AdjustedFixedInstallmentAmount,
      exa.zerointerest,
      exa.isactiveflag                               AS stepisactiveflag,
      ex.mechanismcode                               AS stepmechanismcode,
      ex.exposurestatuscode                          AS stepexposurestatuscode,
      ex.firstinstallmentdate,
      exa.AdjustedRequestedInstallmentFrom,
      exa.AdjustedRequestedInstallmentTo
  FROM exposure_adjustmentperiod exa
  INNER JOIN exposure_profits ex
    ON ex.exposurekey = exa.exposurekey
),
dedup AS (
  SELECT DISTINCT
      stepexposurekey,
      AdjustedFixedInstallmentAmount,
      zerointerest,
      stepisactiveflag,
      stepmechanismcode,
      stepexposurestatuscode,
      firstinstallmentdate,
      AdjustedRequestedInstallmentFrom,
      AdjustedRequestedInstallmentTo
  FROM joined
),
annotated AS (
  SELECT
      d.*,
      add_months(firstinstallmentdate, AdjustedRequestedInstallmentFrom - 1) AS period_from_date,
      add_months(firstinstallmentdate, AdjustedRequestedInstallmentTo   - 1) AS period_to_date
  FROM dedup d
),
ranked AS (
  SELECT
      stepexposurekey,
      AdjustedFixedInstallmentAmount,
      zerointerest,
      stepisactiveflag,
      stepmechanismcode,
      stepexposurestatuscode,
      firstinstallmentdate,
      RANK() OVER (
        PARTITION BY stepexposurekey
        ORDER BY
          period_from_date,
          period_to_date,
          AdjustedFixedInstallmentAmount DESC
      ) AS rnk,
      AdjustedRequestedInstallmentFrom,
      AdjustedRequestedInstallmentTo
  FROM annotated
)
SELECT DISTINCT
    stepexposurekey,
    AdjustedFixedInstallmentAmount,
    zerointerest,
    stepisactiveflag,
    stepmechanismcode,
    stepexposurestatuscode,
    firstinstallmentdate,
    AdjustedRequestedInstallmentFrom,
    AdjustedRequestedInstallmentTo,
    rnk
FROM ranked
WHERE stepisactiveflag = '1'
  AND stepmechanismcode = '4'
  AND stepexposurestatuscode IN ('1','2')
  AND firstinstallmentdate IS NOT NULL
""")

StepUp.createOrReplaceTempView("StepUp")


SplitBalance=spark.sql(f'''
                       SELECT * 
                       FROM `{env}-cdh`.primary.SplitBalance
                       WHERE BusinessDate = {run_date!r} and SourceSystem = "profits"
                       ''')
SplitBalance.createOrReplaceTempView("splitbalance")

ExposureInterestFutureSpread=spark.sql(f'''
                    SELECT * 
                    FROM `{env}-cdh`.primary.ExposureInterestFutureSpread
                    WHERE BusinessDate = {run_date!r} and SourceSystem = "profits"
                                       ''')
ExposureInterestFutureSpread.createOrReplaceTempView("exposureinterestfuturespread")

FutureSpread = spark.sql(f'''
    select 
        ex.exposurekey as ExposureKeyFutureSpread,
        ex.businessdate as businessdate1,
        (fs1.DebitSpreadRate + fs1.CreditSpread) as FutureMargin,
        rank() over(partition by fs1.exposurekey order by fs1.FutureMarginApplyDate) rnk,
        fs1.FutureMarginApplyDate,
        ex.IsActiveFlag,
        ex.Maturitydate,
        ex.PrimaryPartyFlag,
        ex.sourcesystem
    from exposure_profits ex
    inner join exposureinterestfuturespread fs1
        on fs1.exposurekey = ex.exposurekey
    where ex.IsActiveFlag = 1
      and fs1.futuremarginapplydate < ex.maturitydate
      and fs1.futuremarginapplydate > {run_date!r}
      and ex.PrimaryPartyFlag = True
''')

FutureSpread.createOrReplaceTempView("futurespread")

bulletpool=spark.sql(f'''
                    select ex.exposurekey from exposure_profits ex
                    where  ex.mechanismcode in ('1','2','3','6')
                    and ex.ExposureStatusCode in ('1','2') and ex.IsActiveFlag="1" and ex.SlaveFlag="0"
                      ''')
bulletpool.createOrReplaceTempView("bulletpool")

bulletloans=spark.sql(f'''
                    select s.exposurekey from paymentschedule s
                    inner join bulletpool b
                    on s.exposurekey=b.exposurekey and s.installmentdate > s.businessdate
                    group by s.exposurekey
                    having count(1)=1
                      ''')
bulletloans.createOrReplaceTempView("bulletloans")

bullet=spark.sql(f'''
                    select s.exposurekey,s.installmentamount,s.installmentdate from paymentschedule s
                    inner join bulletloans b
                    on s.exposurekey=b.exposurekey   and s.installmentdate>s.businessdate
                 ''')
bullet.createOrReplaceTempView("bullet")
        
nextInterest = spark.sql(f'''
            WITH nextInterest AS (
              SELECT
                  ex.exposurekey                                           AS exposurekey2,
                  COALESCE(stepup.fixinterestCode, 0)                      AS fixcode,
                  COALESCE(stepup.floatinterestCode, 0)                    AS floatcode,
                  stepup.*,
                  ex.currency,
                  CASE WHEN COALESCE(stepup.fixinterestCode, 0) <> 0
                       THEN 'Future Fixed' ELSE 'Future Variable' END      AS interest_type,
                  CASE WHEN COALESCE(stepup.fixinterestCode, 0) <> 0
                       THEN COALESCE(stepup.fixinterestCode, 0)
                       ELSE COALESCE(stepup.floatinterestCode, 0) END      AS inter_code,
                  ROW_NUMBER() OVER (
                    PARTITION BY ex.exposurekey
                    ORDER BY stepup.ValidDate NULLS LAST, stepup.BusinessDate, ex.exposurekey
                  )                                                        AS rnk
              FROM exposure_profits ex
              INNER JOIN exposure_adjustmentstepup stepup
                      ON ex.exposurekey = stepup.exposurekey
                     AND stepup.IsActiveFlag = '1'
                     AND ex.PrimaryPartyFlag = TRUE
                     AND stepup.ValidDate > stepup.BusinessDate
                     AND stepup.ValidDate < DATE '2090-01-01'
              WHERE COALESCE(stepup.fixinterestCode, 0) <> 0
                 OR COALESCE(stepup.floatinterestCode, 0) <> 0
            )
            SELECT *
            FROM nextInterest
''')

nextInterest.createOrReplaceTempView("nextInterest")

OCWLoan=spark.sql(f'''
                  with ocwloan as (
            select ex.exposurekey,
            u.underlawtype,
            u.applicationstatus,
            case when u.applicationstatus="99 - Ολοκλήρωση αίτησης" then u.CompletionReason else null end as NewOCWSubStatus,
            case when u.applicationstatus="99 - Ολοκλήρωση αίτησης" then u.ApplicationStatusDate else null end as NewOCWSubStatusDate,
            ex.businessdate
            from exposure_profits ex
            inner join dunderlaw u
            on u.CDI=ex.CDI  and u.Underlawtype ="N4738/2020" and lawsystem="Εξωδικαστικός μηχανισμός")
            select distinct * from ocwloan
                  ''')
OCWLoan.createOrReplaceTempView("OCWLoan")




perimeter3869=spark.sql(f'''
                  select distinct ex.exposurekey,
                  ex.CDI,
                  p.ProductDescription,
                  p.ProductKey,
                  ex.exposurestatus,
                  ex.mechanism,
                  ex.BaseRate,
                  ex.N128Rate,
                  ex.SpreadRate,
                  ex.businessdate as businessdate
                  from exposure_profits ex
                  inner join  product_profits p 
                  on p.ProductKey = ex.ProductKey 
                  inner join  dunderlaw law
                  on law.CDI = ex.CDI and law.InLawFlag=1 and law.UnderLawType in ("N3869")
                  where ex.IsActiveFlag="1" and ex.PrimaryPartyFlag=True  and ex.ExposureStatusCode <> "4"
                  and p.ProductKey in ('40567;4016' ,'40567;4015' ,'40568;4015' ,'40567;6079' ,'40567;6078' ,'40568;4016' ,'40517;7362' ,'40517;7363' ,
                  '40568;6079' ,'40568;6078' ,'40568;7363' ,'40324;4015' ,'40568;40000','40568;40001','713')
                          ''')

perimeter3869.createOrReplaceTempView("perimeter3869")

n3869interest=spark.sql(f'''
                          select distinct p.exposurekey,productdescription,productkey,exposurestatus,N128Rate,BaseRate,SpreadRate,
                          case when p.N128Rate + p.BaseRate + p.SpreadRate =0 then 1 else 0 end HasZeroInterestToday
                          from perimeter3869 p
                          left join nextInterest nextInt on nextInt.exposurekey2 = p.exposurekey and nextInt.rnk=1
                          left join FutureSpread fs on fs.ExposureKeyFutureSpread = p.exposurekey and fs.rnk=1 
                          and fs.FutureMarginApplyDate > p.businessdate and fs.FutureMargin > 0
                          where p.baserate + p.n128rate + p.spreadrate = 0

                          ''')
n3869interest.createOrReplaceTempView("n3869interest")



exposure_product=spark.sql('''
                        select distinct  ex.exposurekey as exposurekey,p.ProductDescription as description
                        from exposure_profits ex
                        inner  join product_profits p
                        on p.productkey=ex.productkey
                        ''')

exposure_product.createOrReplaceTempView("exposure_product")



In [0]:
play_profits = spark.sql(f'''
            SELECT 
                ex.exposurekey as ExposureKey,
                CASE WHEN ex.ExposureStatusCode IN ("1","2")
                    AND ex.MechanismCode IN ("4","5","6","7")
                    AND ex.MaturityDate > ex.BusinessDate
                    AND ex.Principal>0
                    AND prd.description not like ("%ΑΙΡΕΣΗ%")
                    THEN 1 ELSE 0 END HasPaymentPlanFlag,
                CASE
                    WHEN COALESCE(BaloonAmount, 0) <> 0 THEN 1
                    ELSE 0
                END HasBalloonFlag,
                CASE 
                    WHEN ex.N128Rate+ex.BaseRate+ex.SpreadRate=0 THEN 1
                    ELSE 0
                END HasZeroInterestTodayFlag,
                CASE 
                    WHEN ex.InterestRateType='Fixed' and ex.InterestDuration!=999 THEN '4' 
                    ELSE 
                        CASE 
                            WHEN ex.InterestRateType='Fixed' THEN '3' 
                            ELSE '1' 
                        END 
                END InterestRateTypeTaxonomy,
                CASE 
                    WHEN ex.InterestRateType='Variable' OR (ex.InterestRateType='Fixed' AND ex.InterestDuration=999) THEN NULL ELSE    
                        CASE 
                        WHEN ex.InterestDuration=999 OR ex.InterestDuration IS NULL THEN NULL 
                        ELSE  
                            CASE 
                            WHEN ex.InterestFrequency=1 THEN concat(ex.InterestDuration, 'D') 
                            ELSE concat(ex.InterestDuration, 'M') END END END InterestRateResetInterval,
                CASE 
                    WHEN splt.exposurekey IS NOT NULL THEN 1 ELSE 0 END HasSplitBalanceFlag,
                CASE 
                    WHEN (ex.InterestRateType='Fixed' AND ex.InterestDuration=999) or (ex.InterestRateType='Variable') THEN NULL 
                    ELSE   ex.InterestFixedDate 
                    END FixInterestEndDate,
                CASE
                    WHEN ex.ExposureStatusCode in ('3','4') THEN 1 
                    WHEN COALESCE(DaysPastDueWithMateriality,0)>0  THEN 1 
	            ELSE 0 END InArrearsFlag,
                CASE WHEN f3869.HasZeroInterestToday = 1 THEN
                    CASE 
                        WHEN COALESCE(nextInt.ValidDate, ex.MaturityDate) <= COALESCE(fs1.FutureMarginApplyDate, ex.MaturityDate) 
                            THEN COALESCE(nextInt.ValidDate, ex.MaturityDate) 
                        ELSE COALESCE(fs1.FutureMarginApplyDate, ex.MaturityDate)
                        END 
                END L3869InterestFreePeriodEndDate,
                CASE 
                    WHEN ex.MechanismCode in ('4', '5', '6') THEN '2' ELSE '9'
                END RepaymentMethod,
                CASE
                    WHEN (ex.InstalmentFrequency='0' OR ex.Mechanism='Amortised Loan (Unequal Installments)' OR ex.ExposureStatus in ('3','4')
                        OR ex.MaturityDate<ex.BusinessDate ) 
                        THEN NULL 
                    ELSE ex.InstalmentFrequency
                END RepaymentFrequency,
                CASE  step.zerointerest WHEN '0' THEN 'Interest Capitalization on Period expiration'
                       WHEN '1' THEN 'Zero Interest'
					   WHEN '2' THEN 'Interest posting on Loan Maturity Date'
					   WHEN '3' THEN 'Interest Capitalization per Instalment '
					   WHEN '4' THEN 'Interest Request' END UnpaidInterestHandling,
                CASE WHEN step.stepexposurekey IS NOT NULL THEN 1 ELSE 0 END HasStepUpInstallmentFlag,
                CASE WHEN fs1.ExposureKeyFutureSpread IS NOT NULL THEN 1 ELSE 0 END HasFutureSpreadFlag,
                CASE WHEN b.exposurekey IS NOT NULL THEN 1 ELSE 0 END IsBulletFlag,
                CASE WHEN nextInt.exposurekey2 IS NOT NULL THEN 1 ELSE 0 END HasFutureInterestChangesFlag,
                CASE WHEN ocw.underlawtype IS NOT NULL THEN 1 ELSE 0 END OCWFlag,
                CASE WHEN ocw.NewOCWSubStatusDate IS NOT NULL THEN 'Pending' ELSE 'Approved' END OCWApplicationStatus,
                nextint.ValidDate,
                fs1.FutureMargin,
                fs1.FutureMarginApplyDate,
                splt.FirstPeriodNumberOfInstallements,
                step.AdjustedRequestedInstallmentFrom as InstalmentSNFrom,
                step.AdjustedRequestedInstallmentTo as InstalmentSNTo,
                step.AdjustedFixedInstallmentAmount as newInstalmentAmount,
                ex.businessdate AS BusinessDate
            FROM exposure_profits ex
            LEFT JOIN n3869interest f3869
            on f3869.exposurekey = ex.exposurekey
            LEFT JOIN OCWLoan ocw 
            on ocw.exposurekey=ex.exposurekey
            LEFT JOIN nextInterest nextInt
            ON ex.exposurekey = nextInt.exposurekey2 and nextInt.rnk=1
            LEFT JOIN bullet b ON ex.exposurekey=b.exposurekey
            LEFT JOIN futurespread fs1
            ON ex.Exposurekey = fs1.ExposureKeyFutureSpread and fs1.rnk=1
            LEFT JOIN splitbalance splt 
            ON splt.ExposureKey=ex.ExposureKey
            LEFT JOIN StepUp step
            ON step.stepexposurekey=ex.exposurekey and step.rnk=1
            LEFT JOIN exposure_product prd 
            ON prd.exposurekey=ex.exposurekey
            
''')


play_profits=play_profits.select(
    play_profits.ExposureKey.alias("ExposureKeyPlayProfits"),
    play_profits.HasBalloonFlag.cast("boolean"),
    play_profits.HasZeroInterestTodayFlag.cast("boolean"),
    play_profits.InterestRateTypeTaxonomy,
    play_profits.InterestRateResetInterval,
    play_profits.HasSplitBalanceFlag.cast("boolean"),
    play_profits.FixInterestEndDate,
    play_profits.RepaymentMethod,
    play_profits.RepaymentFrequency,
    play_profits.HasStepUpInstallmentFlag.cast("boolean"),
    play_profits.UnpaidInterestHandling,
    play_profits.HasFutureSpreadFlag.cast("boolean"),
    play_profits.IsBulletFlag.cast("boolean"),
    play_profits.HasFutureInterestChangesFlag.cast("boolean"),
    play_profits.OCWFlag.cast("boolean"),
    play_profits.OCWApplicationStatus,
    play_profits.L3869InterestFreePeriodEndDate,
    play_profits.InArrearsFlag.cast("boolean"),
    play_profits.HasPaymentPlanFlag.cast("boolean"),
    play_profits.ValidDate,
    play_profits.FutureMargin,
    play_profits.FutureMarginApplyDate,
    play_profits.FirstPeriodNumberOfInstallements,
    play_profits.InstalmentSNFrom,
    play_profits.InstalmentSNTo,
    play_profits.newInstalmentAmount,
    play_profits.BusinessDate
)

play_profits.createOrReplaceTempView("play_profits")

eom=spark.sql(f'''
              select  distinct try_cast(CurrentMonthLastWorkingDate as string) from `{env}-cdh`.reference.ref_calendar
where calendardate between "2021-01-01" and current_date()
              ''')
### this date was used during the construction of Play Data Tape
eom.createOrReplaceTempView("eom")

historical_monthly=spark.sql(f'''
        select  ex.exposurekey,
        ex.portfolioname,
        ex.IsActiveFlag,
        ex.OutstandingTotalBalance,
        eom.CurrentMonthLastWorkingDate,
        coalesce(ex.DaysPastDueWithMateriality,0) as SourcePastDueDays,
        ex.exposurestatuscode,
        ex.BusinessDate
        from exposure_profits ex
        inner join eom on eom.CurrentMonthLastWorkingDate = ex.businessdate and ex.sourcesystem="profits" and ex.PrimaryPartyFlag=True
                            ''' )
historical_monthly.createOrReplaceTempView("historical_monthly")

inArrears = spark.sql(f'''
                      select p.exposurekeyplayprofits as ExposureKey,r.DateLastCurrent
                      from play_profits p 
                      left join (
                        select a.exposurekey,max(a.BusinessDate) as DateLastCurrent from historical_monthly a
                        inner join play_profits b on b.exposurekeyplayprofits = a.exposurekey
                        where b.inarrearsflag='1'
                        and a.outstandingtotalbalance>0
                        and a.isactiveflag=True
                        and a.SourcePastDueDays<=0
                        and a.businessdate < {run_date!r}
                        group by a.exposurekey )
                        r on r.exposurekey=p.exposurekeyplayprofits
                        where p.inarrearsflag='1'
                      
                ''')

inArrears.createOrReplaceTempView("inArrears")

notInArrears=spark.sql(f'''
                       select p.exposurekeyplayprofits as ExposureKey,p.BusinessDate as DateLastCurrent
                       from play_profits p 
                        where p.inarrearsflag='0'
                       ''')
notInArrears.createOrReplaceTempView("notInArrears")

play_profits_final=spark.sql('''
                            select p.*,
                            case when p.InArrearsFlag='1' then arr.DateLastCurrent else narr.DateLastCurrent end DateLastCurrent
                            from play_profits p
                            left join inArrears arr
                            on arr.exposurekey = p.ExposureKeyPlayProfits
                            left join notInArrears narr
                            on narr.exposurekey = p.ExposureKeyPlayProfits
                        ''')
play_profits_final=play_profits_final.distinct()
play_profits_final.createOrReplaceTempView("play_profits_final")

exposurerequest = spark.sql(f'''
        select ExposureKey, RequestExpiryDate, ExposureStatusCode, RequestTypeCode, SourceSystem, BusinessDate
        from  `{env}-cdh`.primary.exposurerequest
        where BusinessDate = {run_date!r}
        and SourceSystem = 'profits'
        ''' )
exposurerequest.createOrReplaceTempView("exposurerequest")

exposure_request_final = spark.sql("""
        select  ExposureKey, 
                min(RequestExpiryDate) FirstArrDate, 
                max(RequestExpiryDate) DateLastInArrears
        from exposurerequest
        where RequestTypeCode in ('1', '2', '4')
        and ExposureStatusCode != '1'
        and RequestExpiryDate <= BusinessDate
        group by ExposureKey
""")

exposure_request_final.createOrReplaceTempView("exposure_request_final")

## First Calculations (profits)

In [0]:
exposure_profits = (
            exposure_profits.withColumn(
                "ExposureStatusCalc",
                when(
                    exposure_profits.ExposureStatusCode == "1","Normal"
                ).when(
                    exposure_profits.ExposureStatusCode == "2","Overdue"
                ).when(
                    exposure_profits.ExposureStatusCode == "3","Define Delay"
                ).when(
                    exposure_profits.ExposureStatusCode == "4","Writen Off"
                )
            ).withColumn( 
                "OffBalanceSymbolPre",
                when(
                    exposure_profits.ExposureStatusCode =='4' , 
                    coalesce(exposure_profits.OutstandingTotalBalance.cast("decimal(19,2)"),lit(0)) - (coalesce(exposure_profits.Accrued.cast("decimal(19,2)"),lit(0))
                    + coalesce(exposure_profits.DueAccrued.cast("decimal(19,2)"),lit(0)))  
                ).otherwise((coalesce(exposure_profits.DueUnrealizedinterest,lit(0)) + coalesce(exposure_profits.PenaltyUnrealizedinterest,lit(0)) + coalesce(exposure_profits.Unrealizedinterest,lit(0))) )
            ).withColumn( 
                "OffBalancePre",
                coalesce(exposure_profits.DueUnrealizedinterest,lit(0)) + coalesce(exposure_profits.PenaltyUnrealizedinterest,lit(0)) +coalesce(exposure_profits.Unrealizedinterest,lit(0))
            ).withColumn( 
                "OnBalancePre",
                (coalesce(exposure_profits.Principal,lit(0)) + 
                 coalesce(exposure_profits.interest,lit(0))  + 
                 coalesce(exposure_profits.Expense,lit(0))  + 
                 coalesce(exposure_profits.DuePrincipal,lit(0))  + 
                 coalesce(exposure_profits.Dueinterest,lit(0)) + 
                 coalesce(exposure_profits.Penaltyinterest,lit(0))  + 
                 coalesce(exposure_profits.DueExpense,lit(0))  + 
                 coalesce(exposure_profits.Commission,lit(0)) + 
                 coalesce(exposure_profits.DueCommission,lit(0)))
            ).withColumn( 
                "OnBalanceSymbolPre",
                when(
                    exposure_profits.ExposureStatusCode =='4' , lit(0)  
                ).otherwise(
                    coalesce(exposure_profits.Principal,lit(0)) + 
                    coalesce(exposure_profits.interest,lit(0)) + 
                    coalesce(exposure_profits.Expense,lit(0)) + 
                    coalesce(exposure_profits.DuePrincipal,lit(0)) + 
                    coalesce(exposure_profits.Dueinterest,lit(0)) + 
                    coalesce(exposure_profits.Penaltyinterest,lit(0)) + 
                    coalesce(exposure_profits.DueExpense,lit(0)) + 
                    coalesce(exposure_profits.Commission,lit(0)) + 
                    coalesce(exposure_profits.DueCommission,lit(0))       

                )
            ).withColumn( 
                "MLTPre",
                coalesce(exposure_profits.Unrealizedinterest,lit(0)) +  
                coalesce(exposure_profits.DueUnrealizedinterest,lit(0)) +  
                coalesce(exposure_profits.PenaltyUnrealizedinterest,lit(0))
            )
            .withColumn( 
                "GBVGalPre",
 		        coalesce(exposure_profits.Principal,lit(0)) + 
                coalesce(exposure_profits.interest,lit(0))  + 
                coalesce(exposure_profits.Unrealizedinterest,lit(0)) + 
                coalesce(exposure_profits.Expense,lit(0)) + 
                coalesce(exposure_profits.DuePrincipal,lit(0)) + 
                coalesce(exposure_profits.Dueinterest,lit(0)) + 
                coalesce(exposure_profits.Penaltyinterest,lit(0))+ 
                coalesce(exposure_profits.DueUnrealizedinterest,lit(0)) + 
                coalesce(exposure_profits.PenaltyUnrealizedinterest,lit(0)) + 
                coalesce(exposure_profits.DueExpense,lit(0))  + 
                coalesce(exposure_profits.Accrued,lit(0)) +
                coalesce(exposure_profits.DueAccrued,lit(0))
            )
            .withColumn( 
                "GBVMonzaPre",
 		        coalesce(exposure_profits.Principal,lit(0)) + 
                coalesce(exposure_profits.interest,lit(0))  + 
                coalesce(exposure_profits.Expense,lit(0)) + 
                coalesce(exposure_profits.DuePrincipal,lit(0)) + 
                coalesce(exposure_profits.Dueinterest,lit(0)) + 
                coalesce(exposure_profits.Penaltyinterest,lit(0))+ 
                coalesce(exposure_profits.DueExpense,lit(0))  + 
                coalesce(exposure_profits.Accrued,lit(0)) +
                coalesce(exposure_profits.DueAccrued,lit(0)) +
                coalesce(exposure_profits.Commission,lit(0)) +
                coalesce(exposure_profits.DueCommission,lit(0))
            )
            .withColumn( 
                "GBVRhodiumPre",
 		        coalesce(exposure_profits.Principal,lit(0)) + 
                coalesce(exposure_profits.interest,lit(0))  + 
                coalesce(exposure_profits.Expense,lit(0)) + 
                coalesce(exposure_profits.DuePrincipal,lit(0)) + 
                coalesce(exposure_profits.Dueinterest,lit(0)) + 
                coalesce(exposure_profits.Penaltyinterest,lit(0))+ 
                coalesce(exposure_profits.DueExpense,lit(0))  + 
                coalesce(exposure_profits.Accrued,lit(0)) +
                coalesce(exposure_profits.DueAccrued,lit(0))
            )
            .withColumn( 
                "GBVKYMGLNPre",
 		        coalesce(exposure_profits.Principal,lit(0)) + 
                coalesce(exposure_profits.DuePrincipal,lit(0)) +
                coalesce(exposure_profits.interest,lit(0))  + 
                coalesce(exposure_profits.Dueinterest,lit(0)) + 
                coalesce(exposure_profits.Expense,lit(0)) + 
                coalesce(exposure_profits.DueExpense,lit(0))  + 
                coalesce(exposure_profits.Commission,lit(0)) +
                coalesce(exposure_profits.DueCommission,lit(0)) +
                coalesce(exposure_profits.Accrued,lit(0)) +
                coalesce(exposure_profits.DueAccrued,lit(0)) 
            )
            .withColumn(
                "GBVNeptPre",
                when(
                    (exposure_profits.ExposureStatusCode.isin('3','4')) | ((exposure_profits.ExposureStatusCode == '2') & (exposure_profits.DaysPastDue>90)) , 
                     coalesce(exposure_profits.OutstandingTotalBalance.cast("decimal(19,2)"),lit(0)) - 
                    ((coalesce(exposure_profits.DueUnrealizedinterest,lit(0))
                     +coalesce(exposure_profits.PenaltyUnrealizedinterest,lit(0))
                     +coalesce(exposure_profits.Unrealizedinterest,lit(0))) 
                    +(coalesce(exposure_profits.Accrued,lit(0))
                     +coalesce(exposure_profits.DueAccrued,lit(0))))  
                )
                .when(
                    (exposure_profits.ExposureStatusCode == '1') | ((exposure_profits.ExposureStatusCode == '2') & (exposure_profits.DaysPastDue<=90)) , 
                    coalesce(exposure_profits.OutstandingTotalBalance.cast("decimal(19,2)"),lit(0)) -
                    (coalesce(exposure_profits.DueUnrealizedinterest,lit(0))
                     +coalesce(exposure_profits.PenaltyUnrealizedinterest,lit(0))
                     +coalesce(exposure_profits.Unrealizedinterest,lit(0)))
                )
            )           
        
        )

exposure_profits = (
            exposure_profits.withColumn( 
                "OnBalancePreFinal",
                when (exposure_profits.PortfolioKey.isin("15","18","19"),
                    coalesce(exposure_profits.OutstandingTotalBalance,lit(0))-coalesce(exposure_profits.OffBalancePre,lit(0))           
                ).when (exposure_profits.PortfolioKey.isin("3","25"), 
                        exposure_profits.OnBalanceSymbolPre                       
                    ).otherwise(exposure_profits.OnBalancePre)
            )
			.withColumn( 
                "GBVPreFinal",
                when((exposure_profits.PortfolioKey.isin("2","101","9","8","1","113")) & (exposure_profits.OnBalancePre == 0), 
					  lit(0)
                ).when((exposure_profits.PortfolioKey.isin("2","101","9","8","1","113")) & (exposure_profits.OnBalancePre != 0), 
					  coalesce(exposure_profits.OutstandingTotalBalance,lit(0))-coalesce(exposure_profits.OffBalancePre,lit(0))
                ).when(exposure_profits.PortfolioKey == '4', 
					   exposure_profits.GBVNeptPre
                ).when((exposure_profits.PortfolioKey.isin("3","25")) & (exposure_profits.OnBalanceSymbolPre == 0), 
					  lit(0)
                ).when((exposure_profits.PortfolioKey.isin("3","25")) & (exposure_profits.OnBalanceSymbolPre != 0), 
					  coalesce(exposure_profits.OutstandingTotalBalance,lit(0))-coalesce(exposure_profits.OffBalanceSymbolPre,lit(0))
                ).when(exposure_profits.PortfolioKey.isin('15','18','19'), 
					   coalesce(exposure_profits.OutstandingTotalBalance,lit(0))-coalesce(exposure_profits.OffBalancePre,lit(0))
                ).when(exposure_profits.PortfolioKey.isin('5','6','7','10','11','12','14','114','115','116','117','121','23','120','119','122','24','124','125','126','32','33','34','3334','38','41','44'),
                       when(
                             (exposure_profits.ExposureStatusCode == 4)
                            |(exposure_profits.BankingUnitCode.isin(3120, 3127, 3121) )
                            |(exposure_profits.ProductCode.isin(40584, 40784, 40006, 40284, 40294)), lit(0)
                      ).when(
                           exposure_profits.MechanismCode == '7', coalesce(exposure_profits.GBVGalPre,lit(0)) - coalesce(exposure_profits.MLTPre,lit(0))
                      ).otherwise(exposure_profits.GBVGalPre)
                 ).when(exposure_profits.PortfolioKey.isin('26','27'),
                      when(
                             (exposure_profits.ExposureStatusCode == 4), lit(0)  
                      ).when(
                              (concat(exposure_profits.ProductCode,lit('-'), exposure_profits.BankingUnitCode).isin('42616-42616', '42618-42618')),lit(0)
                      ).when(
                              (exposure_profits.ExposureKey.isin('MZW0010316300000000338','MZW0010329900000000176','MZW0010329900000000559','MZW0010329900000000560','MZW0010329900000000561','MZW0010329900000000925')), lit(0) 
                      ).otherwise(exposure_profits.GBVMonzaPre)
                 ).when(exposure_profits.PortfolioKey.isin('30','31','3031','28','29','2829'),
                      when(
                              (exposure_profits.ProductCode.isin('40294', '40584', '40784')),lit(0)
                      ).when(
                             (exposure_profits.ExposureStatusCode == 3), (coalesce(exposure_profits.GBVRhodiumPre,lit(0)) - coalesce(exposure_profits.Accrued,lit(0)) - coalesce(exposure_profits.DueAccrued,lit(0)))
                      ).otherwise(exposure_profits.GBVRhodiumPre)
                ).when(exposure_profits.PortfolioKey.isin('35','36','21','37'), # Imola, Solar Migrations MC , SOLAR ONE VK
                       when(
                             (exposure_profits.ExposureStatusCode == 4), lit(0)
                        ).otherwise(coalesce(exposure_profits.Principal,lit(0)) + coalesce(exposure_profits.DuePrincipal,lit(0)))     
                ).when(exposure_profits.PortfolioKey.isin('39','40'), # GALANE (GLN), KYMA (KYM)
                       when(
                             (exposure_profits.ExposureStatusCode == 4)
                            |(exposure_profits.BankingUnitCode.isin(3120, 3127, 3121))
                            |(exposure_profits.ProductCode.isin(40584, 40784, 40006, 40284, 40294)), lit(0)
                       ).otherwise(coalesce(exposure_profits.GBVKYMGLNPre,lit(0)))
                ).otherwise(
                       when(exposure_profits.OnBalancePre == 0,lit(0)
                        ).otherwise(coalesce(exposure_profits.OutstandingTotalBalance,lit(0))-coalesce(exposure_profits.OffBalancePre,lit(0))) 
                )
            )
            .withColumn( 
                "Deliquent",
                coalesce(exposure_profits.DuePrincipal,lit(0))+
                coalesce(exposure_profits.Dueinterest,lit(0))+
                coalesce(exposure_profits.DueUnrealizedinterest,lit(0))+
                coalesce(exposure_profits.Penaltyinterest,lit(0))+
                coalesce(exposure_profits.PenaltyUnrealizedinterest,lit(0))+
                coalesce(exposure_profits.DueCommission,lit(0))+
                coalesce(exposure_profits.DueExpense,lit(0))+
                coalesce(exposure_profits.DueAccrued,lit(0)) 
            )
        )

 
 
 

exposure_arotron = exposure.filter(exposure.SourceSystem == 'arotron')
exposure_arotron = exposure_arotron.filter(exposure_arotron.PrimaryPartyFlag == True)
exposure_qca = exposure.filter(exposure.SourceSystem == 'qca').distinct()
#exposure_qc = exposure.filter(exposure.SourceSystem == 'qc').distinct()
        
exposure_sec = exposure.alias("exposure_sec")
exposure_sec.createOrReplaceTempView("exposure_sec")

        
exposure_qca = exposure_qca.filter((exposure_qca.PartyRelationshipTypeCode == '5000') & (exposure_qca.SourceSystem == 'qca')) 


collateral_thesis_guarantee = collateral.filter(collateral.SourceSystem == 'thesis')
collateral_thesis_secured = collateral.filter(collateral.SourceSystem == 'thesis')


collateral_thesis_guarantee = collateral_thesis_guarantee.withColumnRenamed("RealEstateCollateralStatusCode","RealEstateCollateralStatusCode_guarantee"
).withColumnRenamed("AgreementKey","AgreementKey_guarantee"
).withColumnRenamed("CategoryCode","CategoryCode_guarantee")

collateral_thesis_guarantee = collateral_thesis_guarantee.filter(collateral_thesis_guarantee.CategoryCode_guarantee == 'Εγγύηση Ελληνικού Δημοσίου')
                  
collateral_thesis_secured = collateral_thesis_secured.withColumnRenamed("RealEstateCollateralStatusCode","RealEstateCollateralStatusCode_secured"
        ).withColumnRenamed("AgreementKey","AgreementKey_secured"
        ).withColumnRenamed("CategoryCode","CategoryCode_secured")
        
collateral_thesis_secured = collateral_thesis_secured.filter(
    (col('RealEstateCollateralStatusCode_secured') == 'Ενεργή') &
    (col('PortfolioName') != 'EUCLID') &
    (~col('PortfolioName').ilike('%MONZA%')) &
    (~col('PortfolioName').isin('Orion X Securitisation DAC' , 'Galaxy II Funding DAC', 'Cosmos Retail', 'GAIA II PORTFOLIO WHOLESALE', 'DOMUS PORTFOLIO - RETAIL','DOMUS PORTFOLIO - WHOLESALE', 'RHODIUM PORTFOLIO - RETAIL', 'RHODIUM PORTFOLIO - WHOLESALE', 'GAIA I PORTFOLIO', 'GAIA II PORTFOLIO RETAIL'))
    )                                                          

secured_flag_keys = collateral_thesis_secured.withColumn('is_secured_flag', lit(True)).select("AgreementKey_secured", "is_secured_flag").distinct()

secured_flag_keys = secured_flag_keys.union(IsSecuredFlagEuclid)


exposure_profits = exposure_profits.withColumnRenamed("ExposureKey","ExposureKeyProfits"
            ).withColumnRenamed("ExposureSourceSystemKey","ExposureSourceSystemKeyProfits"
            ).withColumnRenamed("AgreementKey","AgreementKeyProfits"
            ).withColumnRenamed("SourceSystem","SourceSystemProfits"
            ).withColumnRenamed("Denounceddate","DenouncedDateProfits"
            ).withColumnRenamed("EverDenouncedFlag","EverDenouncedFlagProfits"
            ).withColumnRenamed("IsSubsidyFlag","IsSubsidyFlagProfits"
            ).withColumnRenamed("LPS","LPSProfits"
            ).withColumnRenamed("LPSEng","LPSEngProfits"
            ).withColumnRenamed("LPSCode","LPSCodeProfits"
            ).withColumnRenamed("LPSDate","LPSDateProfits"
            ).withColumnRenamed("RestructuringPEETypeCode","RestructuringPEETypeCodeProfits")


        
        
exposure_arotron = exposure_arotron.withColumnRenamed("ExposureKey","ExposureKeyArotron"
            ).withColumnRenamed("ExposureSourceSystemKey","ExposureSourceSystemKeyArotron"
            ).withColumnRenamed("SourceSystem","SourceSystemyArotron"
            ).withColumnRenamed("Denounceddate","DenouncedDateArotron"
            ).withColumnRenamed("EverDenouncedFlag","EverDenouncedFlagArotron"
            ).withColumnRenamed("IsSubsidyFlag","IsSubsidyFlagArotron"
            ).withColumnRenamed("LPS","LPSArotron"
            ).withColumnRenamed("LPSEng","LPSEngArotron"
            ).withColumnRenamed("LPSCode","LPSCodeArotron"
            ).withColumnRenamed("LPSDate","LPSDateArotron"
            ).withColumnRenamed("RestructuringPEETypeCode","RestructuringPEETypeCodeArotron")

    
    
        
exposure_qca = exposure_qca.withColumnRenamed("ExposureKey","ExposureKeyQCA"
            ).withColumnRenamed("ExposureSourceSystemKey","ExposureSourceSystemKeyQCA"
            ).withColumnRenamed("SourceSystem","SourceSystemQCA"
            ).withColumnRenamed("DenouncedDate","DenouncedDateQCA"
            ).withColumnRenamed("EverDenouncedFlag","EverDenouncedFlagQCA"
            ).withColumnRenamed("EverDenouncedDate","EverDenouncedDateQCA"
            ).withColumnRenamed("IsSubsidyFlag","IsSubsidyFlagQCA"
            ).withColumnRenamed("LPS","LPSQCA"
            ).withColumnRenamed("LPSEng","LPSEngQCA"
            ).withColumnRenamed("LPSCode","LPSCodeQCA"
            ).withColumnRenamed("LPSDate","LPSDateQCA"
            ).withColumnRenamed("RestructuringPEETypeCode","RestructuringPEETypeCodeQCA"
            ).withColumnRenamed("TreatmentBranchCode","TreatmentBranchCodeQCA"
            ).withColumnRenamed("TreatmentBranch","TreatmentBranchQCA"
            ).withColumnRenamed("LetterStatusCode","LetterStatusCodeQCA"
            ).withColumnRenamed("LetterStatus","LetterStatusQCA"
            ).withColumnRenamed("LetterStatusDate","LetterStatusDateQCA"
            ).withColumnRenamed("CoCStatusCode","CoCStatusCodeQCA"
            ).withColumnRenamed("CoCStatus","CoCStatusQCA"
            ).withColumnRenamed("CoCAccountStatusDate","CoCAccountStatusDateQCA"
            ).withColumnRenamed("FirstLetterDate","FirstLetterDateQCA"
            ).withColumnRenamed("SecondLetterDate","SecondLetterDateQCA"
            ).withColumnRenamed("ThirdLetterDate","ThirdLetterDateQCA"
            ).withColumnRenamed("FirstLetterReminderDate","FirstLetterReminderDateQCA"
            ).withColumnRenamed("FirstLetterFlag","FirstLetterFlagQCA"
            ).withColumnRenamed("FirstLetterReminderFlag","FirstLetterReminderFlagQCA"
            ).withColumnRenamed("SecondLetterFlag","SecondLetterFlagQCA"
            ).withColumnRenamed("ThirdLetterFlag","ThirdLetterFlagQCA"
            ).withColumnRenamed("ExclusionDate","ExclusionDateQCA"
            ).withColumnRenamed("ExclusionReasonCode","ExclusionReasonCodeQCA"
            ).withColumnRenamed("ExclusionReason","ExclusionReasonQCA"
            ).withColumnRenamed("LegalActionPriority","LegalActionPriorityQCA"
            ).withColumnRenamed("DueDebtAmount", "DueDebtAmountQCA"
            ).withColumnRenamed("AdministrationStatus", "AdministrationStatusQCA" #new
            ).withColumnRenamed("SayFlag", "SayFlagQCA"#new
            ).withColumnRenamed("CaseCode", "CaseCodeQCA"#new
            ).withColumnRenamed("AllBorrowersDeceasedFlag", "AllBorrowersDeceasedFlagQCA"#new
            ).withColumnRenamed("InsuranceCompany", "InsuranceCompanyQCA"#new
            ).withColumnRenamed("BranchCode", "BranchCodeQCA"#new
            ).withColumnRenamed("InstalmentFrequency", "InstalmentFrequencyQCA"#new
            ).withColumnRenamed("Branch", "BranchQCA"#new
            ).withColumnRenamed("CourtDate", "CourtDateQCA"#new
            ).withColumnRenamed("PhysicalNumber", "PhysicalNumberQCA"#new
            ).withColumnRenamed("SPVCode", "SPVCodeQCA"#new
            ).withColumnRenamed("OriginationBank", "OriginationBankQCA"#new
            ).withColumnRenamed("ObjectiveStatus", "ObjectiveStatusQCA"#new
            ).withColumnRenamed("AdministrationStatusCode", "AdministrationStatusCodeQCA"#new
            ).withColumnRenamed("SPV", "SPVQCA"#new
            ).withColumnRenamed("AccountNticeBUKey", "AccountNticeBUKeyQCA"#new
            ).withColumnRenamed("PreCourtDate", "PreCourtDateQCA"#new
            ).withColumnRenamed("ObjectiveStatusdate", "ObjectiveStatusdateQCA"#new
            ).withColumnRenamed("ObjectiveStatusCode", "ObjectiveStatusCodeQCA"#new
            ).withColumnRenamed("ContractSigningDate", "ContractSigningDateQCA"#new
            ).withColumnRenamed("Tenor", "TenorQCA")#new
    
Restructuring = Restructuring.withColumnRenamed("OldExposureKey","OldExposureKey_restr")
    


#Below code is responsible for finding ProductDescription from qc/qca when Exposure has main source system profits
# Use 'exposure_qca_all' to avoid conflict with later Core Retail filter that overwrites 'exposure_qca' temp view
exposure_qca.createOrReplaceTempView("exposure_qca_all")

product_qca = spark.sql(f""" select
    ProductCode,
    productkey,
    SourceSystem,
    ProductDescription,
    BusinessDate 
FROM `{env}-cdh`.primary.Product
WHERE BusinessDate = {run_date!r} and SourceSystem ='qca'
""") 
product_qca.createOrReplaceTempView("product_qca")

exposure_product_qca=spark.sql('''
                        select distinct  ex.exposurekeyqca as exposurekeyqcqca,p.ProductDescription as SubProductDescription
                        from exposure_qca_all ex
                        inner  join product_qca p
                        on p.productkey=ex.productkey
                        ''')

exposure_product_qca.createOrReplaceTempView("exposure_product_qca")

## Join conditions profits

In [0]:

exposure_profits_exposure_qca_join_condition = (
    substring(exposure_profits["ExposureKeyProfits"],4,50) == exposure_qca["ExposureKeyQCA"]
)

exposure_profits_exposure_non_active_profits_join_condition = (
    exposure_profits["ExposureKeyProfits"] == exposure_non_active_profits["ExposureKeyNonActive"]
)

exposure_profits_ExposureSales_join_condition = (
    exposure_profits["ExposureKeyProfits"] == ExposureSales["ExposureKey_es"]
)

exposure_profits_Restructuring_join_condition = (
    exposure_profits["ExposureKeyProfits"] == Restructuring["OldExposureKey_restr"]
)


exposure_profits_collateral_guarantee_join_condition = (
    (exposure_profits["AgreementKeyProfits"] == collateral_thesis_guarantee["AgreementKey_guarantee"])
)

exposure_profits_collateral_secured_join_condition = (
    (exposure_profits["AgreementKeyProfits"] == secured_flag_keys["AgreementKey_secured"])
)

exposure_profits_collateral_secured_join_condition_Haps = (
    (exposure_profits["AgreementKeyProfits"] == Haps_Final_UnsecuredAgreements["agreementkey"])
)

exposure_profits_is_secured_flag_Monza = (
    (exposure_profits["CDI"] == is_secured_flag_Monza["DebtorCDI"])
)

exposure_profits_PaymentSchedule_min_join_condition = (
    exposure_profits["ExposureKeyProfits"] == PaymentSchedule_min["ExposureKey"]
)

exposure_profits_Settlement_Bucket_profits_join_condition = (
    substring(exposure_profits["ExposureKeyProfits"], 4, 100) == SettlementExposure_Bucket_profits["ExposureKey"]) #fix: regexp_extract double-escaping

exposure_profits_Settlement_Bucket_qca_join_condition = (
    exposure_profits["ExposureKeyProfits"] == SettlementExposure_Bucket_qca["ExposureKey_seb"]
)

exposure_profits_exposure_ever_min_status_4_join_condition = (
    exposure_profits["ExposureKeyProfits"] == exposure_ever_min_status_4["ExposureKey_min_status_4"]
)

exposure_profits_AccountTransaction_Write_Off_non_cond_join_condition = (
    exposure_profits["ExposureKeyProfits"] == AccountTransaction_Write_Off_non_cond["ExposureKey"]
)

exposure_profits_Conditional_Write_off_join_condition = (
    exposure_profits["productkey"] == Conditional_Writeoff_Product_Keys["ProductKey"]
)

exposure_profits_Conditional_Write_off_Date_join_condition = (
    exposure_profits["ExposureKeyProfits"] == Conditional_Write_off_Date["ExposureKey_Wof_D"]
)#new
 
exposure_profits_Conditional_Write_off_Amount_join_condition = (
    exposure_profits["ExposureKeyProfits"] == Conditional_Write_off_Amount["ExposureKey_Wof_A"]
)#new

exposure_profits_Payment_schedule_next_installment_join_condition = (
    exposure_profits["ExposureKeyProfits"] == Payment_Schedule_Loan_Next_Install["ExposureKeyNextInstall"]
)

exposure_profits_Payment_schedule_next_capital_installment_join_condition = (
    exposure_profits["ExposureKeyProfits"] == Payment_schedule_next_capital_installment["ExposureKey_nci"]
)

exposure_profits_Payment_schedule_next_capital_interest_installment_join_condition = (
    exposure_profits["ExposureKeyProfits"] == Payment_schedule_next_capital_interest_installment["ExposureKey_ncii"]
)

exposure_profits_LegalAction_po_join_condition = (
    (exposure_profits["ExposureKeyProfits"] == LegalAction_po["ExposureKey_lapo"])
			 
)

exposure_profits_Law_profits_join_condition = (
    (exposure_profits["ExposureKeyProfits"] == Law_profits["ExposureKey_law"])
)

exposure_profits_exposure_ever_extra_judicial_join_condition = (
    (exposure_profits["ExposureKeyProfits"] == exposure_ever_extra_judicial["ExposureKey_ee_judicial"])

)
exposure_profits_LegalAction_judicial_join_condition = (
    substring(exposure_profits.ExposureKeyProfits,4,100) ==  LegalAction_judicial["ExposureKey_la_judicial"]
)

exposure_profits_play_profits_final_join_condition= (
    (exposure_profits["ExposureKeyProfits"] == play_profits_final["ExposureKeyPlayProfits"])
)

exposure_profits_exposure_request_final_join_condition= (
    (exposure_profits["ExposureKeyProfits"] == exposure_request_final["ExposureKey"])
)

exposure_profits_Payment_schedule_max_installment_join_condition = (
    exposure_profits["ExposureKeyProfits"] == Payment_Schedule_Loan_Last_Install["ExposureKeyLastInstall"]
)

exposure_profits_exposure_product_qca_join_condition = (
    substring(exposure_profits["ExposureKeyProfits"], 4, 50) == exposure_product_qca["exposurekeyqcqca"]
)

exposure_profits_party_CDI_join_condition = (
    (exposure_profits['CDI'] == party_sub_servicer['CDI']) &
    (party_sub_servicer['SubServicer'] == 'EUCLID DV01 - ASSIGN FOR SERVICING')
)

# Joins
target_table_profits = (
    exposure_profits.join(
        exposure_qca,
        exposure_profits_exposure_qca_join_condition,
        "left",
    ).join(
        exposure_non_active_profits,
        exposure_profits_exposure_non_active_profits_join_condition,
        "left",
    ).join(
        ExposureSales,
        exposure_profits_ExposureSales_join_condition,
        "left",
    ).join(
        collateral_thesis_guarantee,
        exposure_profits_collateral_guarantee_join_condition,
        "left",
    ).join(
        secured_flag_keys,
        exposure_profits_collateral_secured_join_condition,
        "left",
    ).join(
        Haps_Final_UnsecuredAgreements,
        exposure_profits_collateral_secured_join_condition_Haps,
        "left",
    ).join(
        is_secured_flag_Monza,
        exposure_profits_is_secured_flag_Monza,
        "left",    
    ).join(
        PaymentSchedule_min,
        exposure_profits_PaymentSchedule_min_join_condition,
        "left",
    ).join(
        SettlementExposure_Bucket_profits,
        exposure_profits_Settlement_Bucket_profits_join_condition,
        "left",
    ).join(
        SettlementExposure_Bucket_qca,
        exposure_profits_Settlement_Bucket_qca_join_condition,
        "left",
    ).join(
        exposure_ever_min_status_4,
        exposure_profits_exposure_ever_min_status_4_join_condition,
        "left",
    ).join(
        AccountTransaction_Write_Off_non_cond,
        exposure_profits_AccountTransaction_Write_Off_non_cond_join_condition,
        "left",
    ).join(
        Conditional_Write_off_Date,
        exposure_profits_Conditional_Write_off_Date_join_condition,
        "left",
    ).join(
        Conditional_Write_off_Amount,
        exposure_profits_Conditional_Write_off_Amount_join_condition,
        "left",
    ).join(
        Conditional_Writeoff_Product_Keys,
        exposure_profits_Conditional_Write_off_join_condition,
        "left",
    ).join(
        Payment_Schedule_Loan_Next_Install,
        exposure_profits_Payment_schedule_next_installment_join_condition,
        "left",
    ).join(
        Payment_schedule_next_capital_installment,
        exposure_profits_Payment_schedule_next_capital_installment_join_condition,
        "left",
    ).join(
        Payment_schedule_next_capital_interest_installment,
        exposure_profits_Payment_schedule_next_capital_interest_installment_join_condition,
        "left",
    ).join(
        LegalAction_po,
        exposure_profits_LegalAction_po_join_condition,
        "left",
    ).join(
        Law_profits,
        exposure_profits_Law_profits_join_condition,
        "left",
    ).join(
        LegalAction_judicial,
        exposure_profits_LegalAction_judicial_join_condition,
        "left",
    ).join(
        exposure_ever_extra_judicial,
        exposure_profits_exposure_ever_extra_judicial_join_condition,
        "left",
    ).join(
        Restructuring,
        exposure_profits_Restructuring_join_condition,
        "left",
    ).join(
        Dk_Final, 
        Dk_Final["ExposureKey_Dk_Final"] == exposure_profits["ExposureKeyProfits"],
        "left",
    ).join(
        play_profits_final,
        exposure_profits_play_profits_final_join_condition,
        "left",
    ).join(
        exposure_request_final,
        exposure_profits_exposure_request_final_join_condition,
        "left",
    ).join(
        initializationexposureinfo,
        initializationexposureinfo.ExposureKey == exposure_profits.ExposureKeyProfits,
        "left",
    ).join(
        Payment_Schedule_Loan_Last_Install,
        exposure_profits_Payment_schedule_max_installment_join_condition,
        "left",
    ).join(
        party_sub_servicer,
        exposure_profits_party_CDI_join_condition,
        "left",
    ).join(
        exposure_product_qca,
        exposure_profits_exposure_product_qca_join_condition,
        "left",#new
    ).join(
        lps_by_exposure,
        exposure_profits["ExposureKeyProfits"] == lps_by_exposure["lps_ExposureKey"],
        "left",
    # ).join(
    #     high_balance_by_exposure,
    #     exposure_profits["ExposureKeyProfits"] == high_balance_by_exposure["hb_ExposureKey"],
    #     "left",
    )	 				
)

## Second calculations (profits)

In [0]:
target_table_profits_ExposureId_primary_key_fields = [
    exposure_profits.ExposureKeyProfits
]
 
target_table_profits = target_table_profits.withColumn(
                "ExposureIdNew",
                sha2(
                    concat_ws(";", *target_table_profits_ExposureId_primary_key_fields),
                    256,
                ),
            ).withColumn("GBVCalc",
                when(
                    (exposure_profits.PortfolioKey.isin('15','18','19')) , coalesce(exposure_profits.OutstandingTotalBalance.cast("decimal(19,2)"),lit(0))-coalesce(exposure_profits.OffBalancePre,lit(0))
                ).when(
                    exposure_profits.PortfolioKey.isin('24') , 
                    coalesce(exposure_profits.Principal, lit(0)) +
                    coalesce(exposure_profits.interest, lit(0)) +
                    coalesce(exposure_profits.Expense, lit(0)) +
                    coalesce(exposure_profits.DuePrincipal, lit(0)) +
                    coalesce(exposure_profits.Dueinterest, lit(0)) +
                    coalesce(exposure_profits.DueExpense, lit(0)) +
                    coalesce(exposure_profits.Penaltyinterest, lit(0))
                ).otherwise(exposure_profits.GBVPreFinal)                                                                    
            ).withColumn(
                "LatestDefaultDateCalc",
                when(exposure_profits.ExposureStatusCode == '1', lit(None))
                .when((exposure_profits.ExposureStatusCode == '2')
                     &(coalesce(exposure_profits.DaysPastDueWithMateriality, lit(0)) <= 90), lit(None))
                .when((exposure_profits.ExposureStatusCode.isin('2','3','4'))
                     &(coalesce(exposure_profits.DaysPastDueWithMateriality, lit(0)) > 90)
                     &(exposure_profits.Overduedate.isNotNull())
                     &(exposure_profits.Overduedate > '1900-01-01')
                       , dateadd(exposure_profits.Overduedate, 90))
                .otherwise(exposure_profits.Defaultdate)
            ).withColumn(
                "DenouncedDateCalc",
                when(
                    (exposure_profits.ExposureStatusCode.isin('3','4')) & (exposure_qca.SourceSystemQCA == 'qca'), coalesce(col("LatestDefaultDateCalc"),exposure_qca.DenouncedDateQCA)
                )
                .when(
                    exposure_profits.ExposureStatusCode.isin('3','4') ,  col("LatestDefaultDateCalc")
                    )
            ).withColumn(
                "OffBalanceCalc",
                when(
                   (exposure_profits.PortfolioKey.isin("3","25")), exposure_profits.OffBalanceSymbolPre                
                ).otherwise(exposure_profits.OffBalancePre)
            ).withColumn(
                "BoGBalance",
                when(
                    exposure_profits.ExposureStatusCode.isin('1','2','3') ,
                    coalesce(exposure_profits.OutstandingTotalBalance,lit(0)) -
                    (coalesce(exposure_profits.Unrealizedinterest,lit(0)) +
                    coalesce(exposure_profits.DueUnrealizedinterest,lit(0))+
                    coalesce(exposure_profits.PenaltyUnrealizedinterest,lit(0)))  
                )
                .when(exposure_profits.ExposureStatusCode == '4', lit(0))              
                 
            ).withColumn(
                "EverDenouncedFlagCalc",
                when(exposure_qca.EverDenouncedFlagQCA.isNotNull(),exposure_qca.EverDenouncedFlagQCA
                ).otherwise(lit(0))    
            ).withColumn(
                "EverDenouncedDateCalc",
                when(exposure_qca.EverDenouncedDateQCA.cast("date") != '1753-01-01',exposure_qca.EverDenouncedDateQCA.cast("date")
                ).otherwise(lit(None))          
            ).withColumn(
                "OnBalanceCalc",
                when(exposure_profits.PortfolioKey.isin('26','27'),
                     coalesce(exposure_profits.GBVMonzaPre,lit(0)) - coalesce(exposure_profits.Accrued,lit(0))- coalesce(exposure_profits.DueAccrued,lit(0))
                ).when(exposure_profits.PortfolioKey.isin('15','18','19'),
                     coalesce(exposure_profits.OutstandingTotalBalance.cast("decimal(19,2)"),lit(0)) - coalesce(exposure_profits.OffBalancePre,lit(0))
                ).when(~exposure_profits.PortfolioKey.isin('5','6','7','10','11','12','14','114','115','116','117','121','23','120','119','122','24','124','125','126','30','31','3031','28','29','2829','35','36','21','37','38','41','44'), exposure_profits.OnBalancePreFinal # Imola, Solar Migrations MC , SOLAR ONE VK , ATHENA VK , JEWEL OS
                ).when(exposure_profits.ProductCode == '40584', exposure_profits.OnBalancePreFinal
                ).when(exposure_profits.GBVPreFinal == 0, coalesce(exposure_profits.Commission,lit(0)) + coalesce(exposure_profits.DueCommission,lit(0))
                ).otherwise(coalesce(exposure_profits.GBVPreFinal,lit(0)) - coalesce(exposure_profits.Accrued,lit(0))- coalesce(exposure_profits.DueAccrued,lit(0)) +
                            coalesce(exposure_profits.Commission,lit(0)) + coalesce(exposure_profits.DueCommission,lit(0)))
            ).withColumn(
                "IsSoldFlagCalc",
                when(ExposureSales.ExposureKey_es.isNotNull(), True
                ).otherwise(False)
            ).withColumn(
                "ExposureTenorCalc",
                when(exposure_profits.Maturitydate > '8900-01-01', lit(None)
                ).otherwise(months_between(exposure_profits.Maturitydate,exposure_profits.Opendate).cast("int"))
            ).withColumn(
                "IsSubsidyFlagCalc",
                 when(coalesce(exposure_profits.IsSubsidyFlagProfits, exposure_qca.IsSubsidyFlagQCA) == 'Y' , True)
                .when(coalesce(exposure_profits.IsSubsidyFlagProfits, exposure_qca.IsSubsidyFlagQCA) == 'N' , False
                ).otherwise(lit(None))
            ).withColumn(
                "DefaultdateCalc",
                when(exposure_profits.ExposureStatusCode.isin('3','4') , exposure_profits.Defaultdate)
                .when((exposure_profits.ExposureStatusCode == '2')
                     &(exposure_profits.DaysPastDue >90)
                     &(exposure_profits.Overduedate.isNotNull())
                     &(exposure_profits.Overduedate > '1900-01-01')                      
                       , dateadd(exposure_profits.Overduedate,90)
                ).otherwise(lit(None))
            ).withColumn(
                "IsGuaranteeFlagCalc",
                when(collateral_thesis_guarantee.AgreementKey_guarantee.isNotNull() , lit(True)
                ).otherwise(lit(False))
            ).withColumn(
                "IsSecuredFlagCalc",
                coalesce(secured_flag_keys.is_secured_flag,
                        Haps_Final_UnsecuredAgreements.is_secured_flag,
                        is_secured_flag_Monza.IsSecuredFlag,
                        lit(False))
            ).withColumn(
                "InterestNextDueDateCalc",
                when((exposure_profits.ExposureStatusCode.isin('3','4')) | (exposure_profits.Maturitydate<= exposure_profits.BusinessDate), lit(None)
                ).otherwise(coalesce(PaymentSchedule_min.InstallmentDateMin.cast("date"),exposure_profits.Maturitydate))
            ).withColumn(
                "NPLDefaultAmountCalc",
                when(exposure_profits.ExposureStatusCode.isin('2','3','4'), exposure_profits.OutstandingTotalBalance.cast("decimal(19,2)")
                ).otherwise(lit(None))
            ).withColumn(
                "NumberOfUnpaidBillsCalc",
                exposure_profits.TotalOverdueInsterest + exposure_profits.TotalOverdueInstallments
               
            ).withColumn(
                "DaysPastDueFinal",
                when((exposure_profits.IsActiveFlag == False) & (exposure_non_active_profits.ClosedDate.isNotNull()), lit(0))
                .when(exposure_profits.ExposureStatusCode == '1', lit(0))
 
                .when((exposure_profits.ExposureStatusCode == '2')
                     &(exposure_profits.OutstandingTotalBalance.cast("decimal(19,2)") == 0)
                     &(exposure_profits.DaysPastDueWithMateriality == 0), lit(0)
 
                ).when((exposure_profits.ExposureStatusCode == '2')
                     &(exposure_profits.Deliquent > 0)
                     &(exposure_profits.Deliquent == exposure_profits.DueAccrued)
                     &(exposure_profits.Overduedate =='1753-01-01'), exposure_profits.DaysPastDueWithMateriality
               
                ).when((exposure_profits.ExposureStatusCode.isin('2','3','4'))
                       &(coalesce(exposure_profits.DaysPastDueWithMateriality, lit(0)) == 0)
                       &(coalesce(exposure_profits.Overduedate, lit('1753-01-01')) != '1753-01-01'), datediff(exposure_profits.BusinessDate, exposure_profits.Overduedate)
         
                ).when((exposure_profits.ExposureStatusCode.isin('2','3','4'))
                       &(coalesce(exposure_profits.DaysPastDueWithMateriality, lit(0)) == 0)
                       &(coalesce(exposure_profits.Defaultdate, lit('1900-01-01')) != '1900-01-01'), datediff(exposure_profits.BusinessDate, exposure_profits.Defaultdate)
               
                ).when((exposure_profits.ExposureStatusCode.isin('2','3','4'))
                       &(coalesce(exposure_profits.DaysPastDueWithMateriality, lit(0)) == 0)
                       &(exposure_profits.ActiveExpiryDate.isNotNull()), datediff(exposure_profits.BusinessDate, exposure_profits.ActiveExpiryDate)
                         
                ).when((exposure_profits.ExposureStatusCode.isin('3','4'))
                       &(coalesce(exposure_profits.DaysPastDueWithMateriality, lit(0)) == 0)
                       &(exposure_profits.Opendate.isNotNull()), datediff(exposure_profits.BusinessDate, exposure_profits.Opendate)  
 
                ).when((exposure_profits.ExposureStatusCode == '2')
                       &(coalesce(exposure_profits.Defaultdate, lit('1900-01-01')) != '1900-01-01')
                       &(coalesce(exposure_profits.Overduedate, lit('1753-01-01')) != '1753-01-01'), exposure_profits.DaysPastDueWithMateriality
 
                ).otherwise(exposure_profits.DaysPastDueWithMateriality)
            ).withColumn(
                "SettlementBucketCalc",
                when(~coalesce(lower(SettlementExposure_Bucket_profits.SettlementStatus),lit('cancelled')).isin('client cancelled (inactive sor mgr)',
                                                                        'rejected (inactive sor mgr)',
                                                                        'cancelled',
                                                                        'internal cancelled (inactive sor mgr)',
                                                                        'cancelled - internal rejection'), SettlementExposure_Bucket_profits.BucketProfits
                ).when(SettlementExposure_Bucket_qca.ExposureKey_seb.isNotNull(),SettlementExposure_Bucket_qca.BucketQCA
                ).otherwise(lit(None))#new
            ).withColumn(
                "HasActiveSettlementFlagCalc",
                when(~coalesce(lower(SettlementExposure_Bucket_profits.SettlementStatus), lit('cancelled')).isin('client cancelled (inactive sor mgr)',
                                                                        'rejected (inactive sor mgr)',
                                                                        'cancelled',
                                                                        'internal cancelled (inactive sor mgr)',
                                                                        'cancelled - internal rejection') , True#new
 
                ).when(SettlementExposure_Bucket_qca.ExposureKey_seb.isNotNull(), True
                ).otherwise(False)
            ).withColumn(
                "NonConditionalWriteOffDateCalc",
                when(exposure_profits.ExposureStatusCode == '4', exposure_ever_min_status_4.MinDate
                ).otherwise(AccountTransaction_Write_Off_non_cond.TransactionDateMin)
            ).withColumn(
                "NonConditionalWriteOffAmountCalc",
                when(exposure_profits.ExposureStatusCode == '4', exposure_profits.OutstandingTotalBalance.cast("decimal(19,2)") + coalesce(AccountTransaction_Write_Off_non_cond.WriteOffAmountSum,lit(0))
                ).otherwise(AccountTransaction_Write_Off_non_cond.WriteOffAmountSum)
            ).withColumn(
                "ConditionalWriteOffAmountCalc",
                when(
                    Conditional_Writeoff_Product_Keys["ProductKey"].isNotNull(),
                    exposure_profits["OutstandingTotalBalance"].cast("decimal(19,2)")
                ).otherwise(lit(None))
            ).withColumn(
                "writeoffamountprofits",
                when(exposure_profits.ExposureStatusCode == '4', exposure_profits.OutstandingTotalBalance.cast("decimal(19,2)")
                ).otherwise(exposure_profits.url_nrm_int_amn)
            ).withColumn(
                "ConditionalWriteOffDateCalc",
                when(
                    Conditional_Writeoff_Product_Keys["ProductKey"].isNotNull(),
                    exposure_profits["OpenDate"].cast("date")
                ).otherwise(lit(None))
            ).withColumn(
                "NextInstallmentMinimumDateCalc",
                when(Payment_schedule_next_capital_interest_installment.NextInstallmentInterestDate <= Payment_schedule_next_capital_installment.NextInstallmentCapitalDate
                        , Payment_schedule_next_capital_interest_installment.NextInstallmentInterestDate
                ).otherwise(Payment_schedule_next_capital_installment.NextInstallmentCapitalDate)
            ).withColumn(
                "ExtraJudicialLetterDateCalc",
                coalesce(LegalAction_judicial.LegalActionDate,exposure_ever_extra_judicial.LPSDate)
            ).withColumn(
                "ExtraJudicialLetterFlagCalc",
                when((LegalAction_judicial.ExposureKey_la_judicial.isNotNull()) | (exposure_ever_extra_judicial.ExposureKey_ee_judicial.isNotNull())   ,lit(True)
                ).otherwise(lit(None))
           
            ).withColumn(
                "ClosedReasonCalc",
                when(((exposure_profits.IsActiveFlag == False ) & (Restructuring.OldExposureKey_restr.isNotNull())), lit('Restructuring')
              ).when(((exposure_profits.IsActiveFlag == False ) & (ExposureSales.ExposureKey_es.isNotNull())), lit('Sold')
              ).when(exposure_profits.IsActiveFlag == False , lit('paid-off')
              ).otherwise(lit(None))          
            ).withColumn(
                "AssetSegmentCodeCalc",
                when(initializationexposureinfo.AssetSegment.isNull(),exposure_profits.AssetSegmentCode).otherwise(lit(None))
            ).withColumn(
                "AssetSegmentCalc",
                coalesce(initializationexposureinfo.AssetSegment,exposure_profits.AssetSegment)
            ).withColumn("NextInterestChangeDateCalc",
                when(exposure_profits.InterestRateType == 'Fixed', exposure_profits.InterestFixedDate
                ).otherwise(lit(None))            
            ).withColumn("PrincipalNextDueDateCalc",
                when(exposure_profits.ExposureStatusCode.isin('3','4'), lit(None)
              ).when(exposure_profits.Maturitydate < exposure_profits.BusinessDate , lit(None)
              ).when(exposure_profits.MechanismCode.isin('4','5', '6') , exposure_profits.NextInstallmentdate                      
              ).otherwise(exposure_profits.Maturitydate)            
            ) 
 
 
target_table_profits = target_table_profits.withColumn(
                            "BucketCalc",
                              when(exposure_profits.IsActiveFlag != True,lit(0)
                            ).when(target_table_profits.SettlementBucketCalc.isNotNull(), target_table_profits.SettlementBucketCalc
                            ).when(exposure_profits.ExposureStatusCode == '3', lit(98)
                            ).when(exposure_profits.ExposureStatusCode == '4', lit(99)
                            ).otherwise(
                                coalesce(
                                    user_defined_functions.get_bucket_from_dpd_udf(target_table_profits.DaysPastDueFinal),
                                    user_defined_functions.get_bucket_from_dpd_udf(exposure_profits.DaysPastDueWithMateriality),
                                    exposure_profits.Bucket)
                        )
                        ).withColumn(
                            "PEERestructureTypeCodeCalc", when( target_table_profits.RestructuringPEETypeCodeProfits.isNotNull(), target_table_profits.RestructuringPEETypeCodeProfits.cast("string")
                            ).when(target_table_profits.HasActiveSettlementFlagCalc == True, lit('17')
                            ).otherwise(lit(None))
                         ).withColumn(
                            "Coupon_Dk",
                            when(
                                 (exposure_profits.ExposureStatus == 'Written Off')
                                 | (exposure_profits.ExposureStatus == 'ΕΠΙΣΦΑΛΕΙΑ')
                                 ,lit(0)
                            )
                            .when(
                                  ((exposure_profits.ExposureStatus != 'Written Off')
                                   & (exposure_profits.ExposureStatus != 'ΕΠΙΣΦΑΛΕΙΑ'))
                                   & (coalesce(target_table_profits.DaysPastDueFinal,exposure_profits.DaysPastDueWithMateriality,lit(0)) == lit(0))
                                   ,  
                            (exposure_profits.BaseRate + exposure_profits.SpreadRate)
                            )
                            .when(((exposure_profits.ExposureStatus != 'Written Off') & (exposure_profits.ExposureStatus != 'ΕΠΙΣΦΑΛΕΙΑ'))
                                   
                            & (coalesce(target_table_profits.DaysPastDueFinal, exposure_profits.DaysPastDueWithMateriality, lit(0) )> lit(0))
                            ,
                            (exposure_profits.BaseRate + exposure_profits.SpreadRate + exposure_profits.PenaltyRate)
                        )
                        ).withColumn(
                            "InitialAssignmentAmount",
                            coalesce(target_table_profits.InitialAssignmentBalance,exposure_profits.OriginationAmount
                        )
                        ).withColumn(
                            "GracePeriodEndDateCalc",
                            when(
                            (exposure_profits.GracePeriodEnddate < "1950-01-01") | 
                            (exposure_profits.GracePeriodEnddate > "2100-01-01"), 
                            lit(None)
                            ).otherwise(exposure_profits.GracePeriodEnddate)
                        ).withColumn(
                            "SettlementDaysPastDue",
                            when(
                                target_table_profits.HasActiveSettlementFlagCalc == True, 
                                when(
                                    coalesce(datediff(exposure_profits.Overduedate, exposure_profits.BusinessDate), lit(0)) < 0,
                                    -coalesce(datediff(exposure_profits.Overduedate, exposure_profits.BusinessDate), lit(0))
                                ).otherwise(
                                    coalesce(datediff(exposure_profits.Overduedate, exposure_profits.BusinessDate), lit(0))
                                )
                            ).otherwise(lit(None))
                        ).withColumn(
                            "TotalUnrealizedInterest",
                            coalesce(exposure_profits.Unrealizedinterest,lit(0)) + coalesce(exposure_profits.DueUnrealizedinterest,lit(0))
                        ).withColumn(
                            "NetDueAmount",
                            coalesce(exposure_profits.DuePrincipal,lit(0)) + coalesce(exposure_profits.Dueinterest,lit(0))
                            + coalesce(exposure_profits.DueUnrealizedinterest,lit(0)) + coalesce(exposure_profits.DueExpense,lit(0))
                        ).withColumn(
                            "TotalInterestAccruals",
                            coalesce(exposure_profits.interest, lit(0)) + coalesce(exposure_profits.Dueinterest, lit(0)) + 
                            coalesce(exposure_profits.DueUnrealizedinterest, lit(0)) + coalesce(exposure_profits.Penaltyinterest, lit(0)) + 
                            coalesce(exposure_profits.PenaltyUnrealizedinterest, lit(0)) + coalesce(exposure_profits.Unrealizedinterest, lit(0)) + coalesce(exposure_profits.Accrued, lit(0)) + coalesce(exposure_profits.DueAccrued, lit(0))
                        ).withColumn(
                            "TotalInterestL3869",
                            when(
                                (exposure_profits.ProductKey).rlike("4015|4016|6078|6079|7362|7363|40000|400001") & 
                                (exposure_profits.ExposureStatus == 'ΕΠΙΣΦΑΛΕΙΑ'),
                                target_table_profits.Interest_Dk
                            ).otherwise(lit(None))
                        )

In [0]:
target_table_profits = (
    target_table_profits
    .withColumn(
        "L3869WaterfallMainBorrower",
        when(
            (target_table_profits.L3869StatusMainBorrowerCode_Dk == 1) &
            (
                exposure_profits.ProductCode.isin("6078", "6079", "4015", "4016", "7362", "7363") |
                exposure_profits.ProductKey.isin("40568;40001", "40568;40000")
            ),
            lit("Positive Court Decision - Implemented")
        )
        .when(
            (target_table_profits.L3869StatusMainBorrowerCode_Dk == 1) &
            (
                ~exposure_profits.ProductCode.isin("6078", "6079", "4015", "4016", "7362", "7363") &
                ~exposure_profits.ProductKey.isin("40568;40001", "40568;40000")
            ) &
            (target_table_profits.HasActiveSettlementFlagCalc == 1),
            lit("Positive Court Decision - Implemented through Settlement")
        )
        .when(
            (target_table_profits.L3869StatusMainBorrowerCode_Dk == 1) &
            (target_table_profits.L3869DraggedFlag_Dk ==1 ),
            lit("Positive Court Decision - Dragged Implementation")
        )
        .when(
            target_table_profits.L3869StatusMainBorrowerCode_Dk == 1,
            lit("Positive Court Decision - Not Implemented")
        )
        .when(
            (target_table_profits.L3869StatusMainBorrowerCode_Dk == 2) &
            (target_table_profits.L3869CourtDateCategoryMainBorrower_Dk == "Past Court Date"),
            lit("Pending Court Decision - Past Court Date")
        )
        .when(
            (target_table_profits.L3869StatusMainBorrowerCode_Dk == 2) &
            (target_table_profits.L3869CourtDateCategoryMainBorrower_Dk.isNull()),
            lit("Pending Court Decision - N/A")
        )
        .when(
            (target_table_profits.L3869StatusMainBorrowerCode_Dk == 2) &
            (target_table_profits.L3869InterbankMainBorrower_Dk == 1) &
            (exposure_profits.ExposureStatus == 'Written Off') &
            (target_table_profits.GBVCalc == 0),
            lit("Pending Court Decision - Future court Date - Interbank - Out of Scope")
        )
        .when(
            (target_table_profits.L3869StatusMainBorrowerCode_Dk == 2) &
            (target_table_profits.L3869InterbankMainBorrower_Dk == 1) &
            (target_table_profits.Assignmentchannel_Dk == "L3869 ΔΤ - FAs"),
            lit("Pending Court Decision - Future court Date - Interbank - Assigned")
        )
        .when(
            (target_table_profits.L3869StatusMainBorrowerCode_Dk == 2) &
            (target_table_profits.L3869InterbankMainBorrower_Dk == 1),
            lit("Pending Court Decision - Future court Date - Interbank - Not Assigned")
        )
        .when(
            (target_table_profits.L3869StatusMainBorrowerCode_Dk == 2) &
            (target_table_profits.L3869InterbankMainBorrower_Dk == 0),
            lit("Pending Court Decision - Future court Date - Not Interbank")
        )
        .when(target_table_profits.L3869StatusMainBorrowerCode_Dk == 11, lit("Out of Law - Rejected"))
        .when(target_table_profits.L3869StatusMainBorrowerCode_Dk == 12, lit("Out of Law - Resigned"))
        .when(target_table_profits.L3869StatusMainBorrowerCode_Dk == 13, lit("Out of Law - Cancelled"))
        .when(target_table_profits.L3869StatusMainBorrowerCode_Dk == 14, lit("Out of Law - Expelled"))
        .when(target_table_profits.L3869StatusMainBorrowerCode_Dk == 15, lit("Out of Law - Other"))
        .otherwise(lit(None))
    )
)


### Third Calculations profits

### InstallmentIncreaseGroup, InitialInstallmentGroup, StepUpSegment calculation

In [0]:
target_table_profits.createOrReplaceTempView("target_table_profits")

Installment_End_Date_Future = '2200-01-01' 

PaymentSchedule_future = spark.sql(f'''
SELECT ExposureKey, InstallmentDate, abs(InstallmentAmount) as InstallmentAmount, InstallmentType, InstallmentNumber, SourceSystem
FROM PaymentSchedule 
WHERE BusinessDate = {run_date!r}                         
                            ''')

PaymentSchedule_future.createOrReplaceTempView("PaymentSchedule_future")

FutureInitialInstallmentDate = spark.sql(f'''
          SELECT   ExposureKey
              ,MIN(InstallmentDate) AS InitialInstallmentDate
          FROM PaymentSchedule_future
          WHERE InstallmentDate>{run_date!r}           
          AND InstallmentAmount>=10
          group by ExposureKey

          ''')
FutureInitialInstallmentDate.createOrReplaceTempView("FutureInitialInstallmentDate")
FutureAccount = spark.sql(f'''
          SELECT exposure_profits.ExposureKey,
                exposure_profits.PortfolioName 
          FROM target_table_profits
          INNER JOIN exposure_profits
          ON target_table_profits.ExposureKeyProfits = exposure_profits.ExposureKey
          WHERE target_table_profits.GBVCalc>0  AND exposure_profits.ExposureStatusCode <> 3
           ''')

FutureAccount.createOrReplaceTempView("FutureAccount") 

FutureInstallments = spark.sql ( f'''  
            SELECT
       c.PortfolioName
      ,a.ExposureKey
      ,a.InstallmentNumber    AS Initial_Installment_Number
      ,a.InstallmentDate      AS Initial_Installment_Date
      ,abs(a.InstallmentAmount)    AS Initial_Installment_Amount
  FROM PaymentSchedule_future               a
  INNER JOIN FutureInitialInstallmentDate     b
  ON 
        a.ExposureKey=b.ExposureKey 
    AND b.InitialInstallmentDate=a.InstallmentDate 
    AND InstallmentAmount >= 10 
  INNER JOIN FutureAccount c
  ON 
    a.ExposureKey=c.ExposureKey
  WHERE  1=1
  AND InstallmentDate > {run_date!r} 
  AND abs(InstallmentAmount) >= 10 
  AND ((SourceSystem='profits' and InstallmentType = 'Installment') 
        or (SourceSystem='qca' or SourceSystem='process'))
''')


 
FutureInstallments.createOrReplaceTempView("FutureInstallments")   

All_installments_Future = spark.sql(f'''
  SELECT 
       b.PortfolioName
      ,a.ExposureKey
      ,b.Initial_Installment_Number
      ,b.Initial_Installment_Date
      ,b.Initial_Installment_Amount
      ,InstallmentNumber
      ,InstallmentDate
      ,InstallmentAmount
  FROM PaymentSchedule_future       a
  INNER JOIN FutureInstallments   b
  ON 
    a.ExposureKey=b.ExposureKey
  WHERE 
  ((SourceSystem='profits' and InstallmentType = 'Installment') or (SourceSystem='qca' or SourceSystem='process'))    --'Installment' 
  AND InstallmentDate>b.Initial_Installment_Date AND InstallmentDate <= {Installment_End_Date_Future!r}
  AND InstallmentAmount > 1.15* Initial_Installment_Amount
  AND InstallmentAmount -  Initial_Installment_Amount > 30 
  AND Initial_Installment_Number IS NOT NULL

          ''')

All_installments_Future.createOrReplaceTempView("All_installments_Future")   

FutureMinIEventDate = spark.sql(''' 
  SELECT
      ExposureKey
      ,MIN(InstallmentDate) AS Event_Date
  FROM All_installments_Future
  GROUP BY ExposureKey     

''') 

FutureMinIEventDate.createOrReplaceTempView("FutureMinIEventDate")   

FutureInstallmentsFinal = spark.sql ('''
   SELECT         
      a.PortfolioName
      ,a.ExposureKey
      ,  CASE
          WHEN size(array_distinct(collect_list(a.Initial_Installment_Number))) > 1
            THEN sum(a.Initial_Installment_Amount)
          ELSE max(a.Initial_Installment_Amount)
          END AS Initial_Installment_Amount
      ,max(a.Initial_Installment_Date)   AS Initial_Installment_Date
      ,max(a.Initial_Installment_Number) AS Initial_Installment_Number
      ,CASE
        WHEN size(array_distinct(collect_list(a.InstallmentNumber))) > 1
          THEN sum(a.InstallmentAmount)
        ELSE max(a.InstallmentAmount)
        END AS Event_Amount
      ,max(a.InstallmentNumber) AS Event_InstallmentNumber
      ,max(b.Event_Date)             AS Event_Date
      ,DATEDIFF(month,a.Initial_Installment_Date,b.Event_Date)  AS Event_Months

      ,"Future"                          AS StepUpFlag
   FROM All_installments_Future      a
  INNER JOIN FutureMinIEventDate    b
  ON 
    a.ExposureKey=b.ExposureKey 
    AND a.InstallmentDate=b.Event_Date 
    AND Initial_Installment_Amount>=10 
    AND InstallmentAmount>=10
  WHERE InstallmentNumber IS NOT NULL
  group by 
  a.ExposureKey,
  a.PortfolioName,
  Initial_Installment_Date,
  Event_Date
 ''')
 
FutureInstallmentsFinal.createOrReplaceTempView("FutureInstallmentsFinal")   
 
FutureInstallmentsFinal = spark.sql ( """
                                     SELECT ExposureKey as ExposureKey_FutureInstallment ,  Event_Date   as FutureStepUpEventDate,Event_Amount as FutureStepUpEventAmount,
        CASE WHEN (Event_Amount -Initial_Installment_Amount)/Initial_Installment_Amount>=0.15 AND  (Event_Amount -Initial_Installment_Amount)/Initial_Installment_Amount<0.20  THEN '15%-20%'
        WHEN (Event_Amount -Initial_Installment_Amount)/Initial_Installment_Amount>=0.20 AND  (Event_Amount -Initial_Installment_Amount)/Initial_Installment_Amount<0.30  THEN '20%-30%'
        WHEN (Event_Amount -Initial_Installment_Amount)/Initial_Installment_Amount>=0.30 AND  (Event_Amount -Initial_Installment_Amount)/Initial_Installment_Amount<0.50  THEN '30%-50%'
        WHEN (Event_Amount -Initial_Installment_Amount)/Initial_Installment_Amount>=0.50 AND  (Event_Amount -Initial_Installment_Amount)/Initial_Installment_Amount<1.00  THEN '50%-100%'
        WHEN (Event_Amount -Initial_Installment_Amount)/Initial_Installment_Amount>=1.00 AND  (Event_Amount -Initial_Installment_Amount)/Initial_Installment_Amount<2.00  THEN '100%-200%'
        WHEN (Event_Amount -Initial_Installment_Amount)/Initial_Installment_Amount>=2.00                                            THEN '200+%'
    END AS InstallmentIncreaseGroup
    ,CASE WHEN Initial_Installment_Amount<=50    THEN '<=50'
        WHEN Initial_Installment_Amount<=100    THEN '50-100'
        WHEN Initial_Installment_Amount<=200    THEN '100-200'
        WHEN Initial_Installment_Amount<=400    THEN '200-400'
        WHEN Initial_Installment_Amount<=800    THEN '400-800'
        WHEN Initial_Installment_Amount>800    THEN '800+'
    END AS InitialInstallmentGroup
     ,CASE WHEN Initial_Installment_Amount>800 AND ((Event_Amount -Initial_Installment_Amount)/Initial_Installment_Amount>=0.30)                          THEN 'High Step Up'
    WHEN (Initial_Installment_Amount>400 AND Initial_Installment_Amount<=800) AND (Event_Amount -Initial_Installment_Amount)/Initial_Installment_Amount>=0.50        THEN 'High Step Up'
    WHEN (Initial_Installment_Amount>200 AND Initial_Installment_Amount<=400) AND (Event_Amount -Initial_Installment_Amount)/Initial_Installment_Amount>=1.00        THEN 'High Step Up'
    WHEN (Initial_Installment_Amount>100 AND Initial_Installment_Amount<=200) AND (Event_Amount -Initial_Installment_Amount)/Initial_Installment_Amount>=1.00        THEN 'High Step Up'
    WHEN (Initial_Installment_Amount>50 AND Initial_Installment_Amount<=100)  AND (Event_Amount -Initial_Installment_Amount)/Initial_Installment_Amount>=2.00        THEN 'High Step Up'
    WHEN Initial_Installment_Amount>800 AND (Event_Amount -Initial_Installment_Amount)/Initial_Installment_Amount<0.30                            THEN 'Medium Step up'
    WHEN (Initial_Installment_Amount>400 AND Initial_Installment_Amount<=800) AND ((Event_Amount -Initial_Installment_Amount)/Initial_Installment_Amount>=0.20 
       AND (Event_Amount -Initial_Installment_Amount)/Initial_Installment_Amount<0.50)                                            THEN 'Medium Step up'
    WHEN ((Initial_Installment_Amount>200 AND Initial_Installment_Amount<=400) OR (Initial_Installment_Amount>100 AND Initial_Installment_Amount<=200)) 
       AND ((Event_Amount -Initial_Installment_Amount)/Initial_Installment_Amount>=0.30 
       AND (Event_Amount -Initial_Installment_Amount)/Initial_Installment_Amount<1.00)                                            THEN 'Medium Step up'
    WHEN (Initial_Installment_Amount>50 AND Initial_Installment_Amount<=100) AND ((Event_Amount -Initial_Installment_Amount)/Initial_Installment_Amount>=0.50 
       AND (Event_Amount -Initial_Installment_Amount)/Initial_Installment_Amount<2.00)                                            THEN 'Medium Step up'
    WHEN Initial_Installment_Amount<=50  AND ((Event_Amount -Initial_Installment_Amount)/Initial_Installment_Amount>=1.00)                          THEN 'Medium Step up'
    WHEN (Initial_Installment_Amount>400 AND Initial_Installment_Amount<=800) AND (Event_Amount -Initial_Installment_Amount)/Initial_Installment_Amount<0.20        THEN 'Mild Step Up'
    WHEN ((Initial_Installment_Amount>100 AND Initial_Installment_Amount<=400) AND (Event_Amount -Initial_Installment_Amount)/Initial_Installment_Amount<0.30)        THEN 'Mild Step Up'
    WHEN (Initial_Installment_Amount>50 AND Initial_Installment_Amount<=100) AND ((Event_Amount -Initial_Installment_Amount)/Initial_Installment_Amount>=0.30 
       AND  (Event_Amount -Initial_Installment_Amount)/Initial_Installment_Amount<0.50)                                            THEN 'Mild Step Up'
    WHEN Initial_Installment_Amount<=50 AND ((Event_Amount -Initial_Installment_Amount)/Initial_Installment_Amount>=0.50 
    AND  (Event_Amount -Initial_Installment_Amount)/Initial_Installment_Amount<1.00)                                            THEN 'Mild Step Up'                          
     END AS StepUpSegment
     ,StepUpFlag
  FROM FutureInstallmentsFinal
 """)

In [0]:
PaymentSchedule_past = spark.sql(f'''
SELECT ExposureKey, InstallmentDate, abs(InstallmentAmount) as InstallmentAmount, InstallmentType, InstallmentNumber, SourceSystem
FROM PaymentSchedule 
WHERE BusinessDate = {run_date!r}                           
                            ''')

PaymentSchedule_past.createOrReplaceTempView("PaymentSchedule_past")

PastInitialInstallmentDate = spark.sql(f'''
          SELECT   ExposureKey
              ,MIN(InstallmentDate) AS InitialInstallmentDate
          FROM PaymentSchedule_past
          WHERE InstallmentDate>cast(DATEADD(year, -1, {run_date!r}) as date) -- Installment_Date_Past  
          AND InstallmentAmount>=10
          group by ExposureKey

          ''')
PastInitialInstallmentDate.createOrReplaceTempView("PastInitialInstallmentDate")
PastAccount = spark.sql(f'''
          SELECT exposure_profits.ExposureKey,
                exposure_profits.PortfolioName 
          FROM target_table_profits
          INNER JOIN exposure_profits
          ON target_table_profits.ExposureKeyProfits = exposure_profits.ExposureKey
          WHERE target_table_profits.GBVCalc>0  AND exposure_profits.ExposureStatusCode <> 3
           ''')

PastAccount.createOrReplaceTempView("PastAccount") 

PastInstallments = spark.sql ( f'''  
            SELECT
       c.PortfolioName
      ,a.ExposureKey
      ,a.InstallmentNumber    AS Initial_Installment_Number
      ,a.InstallmentDate      AS Initial_Installment_Date
      ,abs(a.InstallmentAmount)    AS Initial_Installment_Amount
  FROM PaymentSchedule_past               a
  INNER JOIN PastInitialInstallmentDate     b
  ON 
        a.ExposureKey=b.ExposureKey 
    AND b.InitialInstallmentDate=a.InstallmentDate 
    AND InstallmentAmount >= 10 
  INNER JOIN PastAccount c
  ON 
    a.ExposureKey=c.ExposureKey
  WHERE  1=1
  AND ((SourceSystem='profits' and InstallmentType = 'Installment') or (SourceSystem='qca' or SourceSystem='process'))
  AND InstallmentDate > cast(DATEADD(year, -1, {run_date!r}) as date) -- Installment_Date_Past
  AND abs(InstallmentAmount) >= 10 
''')


 
PastInstallments.createOrReplaceTempView("PastInstallments")   

All_installments_Past = spark.sql(f'''
  SELECT 
       b.PortfolioName
      ,a.ExposureKey
      ,b.Initial_Installment_Number
      ,b.Initial_Installment_Date
      ,b.Initial_Installment_Amount
      ,InstallmentNumber
      ,InstallmentDate
      ,InstallmentAmount
  FROM PaymentSchedule_past       a
  INNER JOIN PastInstallments   b
  ON 
    a.ExposureKey=b.ExposureKey
  WHERE 
  ((SourceSystem='profits' and InstallmentType = 'Installment') or (SourceSystem='qca' or SourceSystem='process'))   --'Installment' 
  AND InstallmentDate>b.Initial_Installment_Date AND InstallmentDate <= {run_date!r} -- Installment_End_Date_Past
  AND InstallmentAmount > 1.15* Initial_Installment_Amount
  AND InstallmentAmount -  Initial_Installment_Amount > 30 
  AND Initial_Installment_Number IS NOT NULL

          ''')

All_installments_Past.createOrReplaceTempView("All_installments_Past")   

PastMinIEventDate = spark.sql(''' 
  SELECT
      ExposureKey
      ,MIN(InstallmentDate) AS Event_Date
  FROM All_installments_Past
  GROUP BY ExposureKey     

''') 

PastMinIEventDate.createOrReplaceTempView("PastMinIEventDate")   

PastInstallmentsFinal = spark.sql ('''
   SELECT
       a.PortfolioName         
      ,a.ExposureKey
      ,CASE
          WHEN size(array_distinct(collect_list(a.Initial_Installment_Number))) > 1
            THEN sum(a.Initial_Installment_Amount)
          ELSE max(a.Initial_Installment_Amount)
        END AS Initial_Installment_Amount
      ,max(a.Initial_Installment_Date)   AS Initial_Installment_Date
      ,max(a.Initial_Installment_Number) AS Initial_Installment_Number
      ,CASE
        WHEN size(array_distinct(collect_list(a.InstallmentNumber))) > 1
          THEN sum(a.InstallmentAmount)
        ELSE max(a.InstallmentAmount)
        END AS Event_Amount
        ,max(a.InstallmentNumber) AS Event_InstallmentNumber
      ,max(b.Event_Date)             AS Event_Date
      ,DATEDIFF(month,a.Initial_Installment_Date,b.Event_Date)  AS Event_Months
      ,"Past"                          AS StepUpFlag
   FROM All_installments_Past      a
  INNER JOIN PastMinIEventDate    b
  ON 
    a.ExposureKey=b.ExposureKey 
    AND a.InstallmentDate=b.Event_Date 
    AND Initial_Installment_Amount>=10 
    AND InstallmentAmount>=10
  WHERE InstallmentNumber IS NOT NULL
  group by 
  a.ExposureKey,
  a.PortfolioName,
  Initial_Installment_Date,
  Event_Date
 ''')
 
PastInstallmentsFinal.createOrReplaceTempView("PastInstallmentsFinal")   
 
PastInstallmentsFinal = spark.sql ( """
                                     SELECT ExposureKey as ExposureKey_PastInstallment ,
        CASE WHEN (Event_Amount -Initial_Installment_Amount)/Initial_Installment_Amount>=0.15 AND  (Event_Amount -Initial_Installment_Amount)/Initial_Installment_Amount<0.20  THEN '15%-20%'
        WHEN (Event_Amount -Initial_Installment_Amount)/Initial_Installment_Amount>=0.20 AND  (Event_Amount -Initial_Installment_Amount)/Initial_Installment_Amount<0.30  THEN '20%-30%'
        WHEN (Event_Amount -Initial_Installment_Amount)/Initial_Installment_Amount>=0.30 AND  (Event_Amount -Initial_Installment_Amount)/Initial_Installment_Amount<0.50  THEN '30%-50%'
        WHEN (Event_Amount -Initial_Installment_Amount)/Initial_Installment_Amount>=0.50 AND  (Event_Amount -Initial_Installment_Amount)/Initial_Installment_Amount<1.00  THEN '50%-100%'
        WHEN (Event_Amount -Initial_Installment_Amount)/Initial_Installment_Amount>=1.00 AND  (Event_Amount -Initial_Installment_Amount)/Initial_Installment_Amount<2.00  THEN '100%-200%'
        WHEN (Event_Amount -Initial_Installment_Amount)/Initial_Installment_Amount>=2.00                                            THEN '200+%'
    END AS InstallmentIncreaseGroup
    ,CASE WHEN Initial_Installment_Amount<=50    THEN '<=50'
        WHEN Initial_Installment_Amount<=100    THEN '50-100'
        WHEN Initial_Installment_Amount<=200    THEN '100-200'
        WHEN Initial_Installment_Amount<=400    THEN '200-400'
        WHEN Initial_Installment_Amount<=800    THEN '400-800'
        WHEN Initial_Installment_Amount>800    THEN '800+'
    END AS InitialInstallmentGroup
     ,CASE WHEN Initial_Installment_Amount>800 AND ((Event_Amount -Initial_Installment_Amount)/Initial_Installment_Amount>=0.30)                          THEN 'High Step Up'
    WHEN (Initial_Installment_Amount>400 AND Initial_Installment_Amount<=800) AND (Event_Amount -Initial_Installment_Amount)/Initial_Installment_Amount>=0.50        THEN 'High Step Up'
    WHEN (Initial_Installment_Amount>200 AND Initial_Installment_Amount<=400) AND (Event_Amount -Initial_Installment_Amount)/Initial_Installment_Amount>=1.00        THEN 'High Step Up'
    WHEN (Initial_Installment_Amount>100 AND Initial_Installment_Amount<=200) AND (Event_Amount -Initial_Installment_Amount)/Initial_Installment_Amount>=1.00        THEN 'High Step Up'
    WHEN (Initial_Installment_Amount>50 AND Initial_Installment_Amount<=100)  AND (Event_Amount -Initial_Installment_Amount)/Initial_Installment_Amount>=2.00        THEN 'High Step Up'
    WHEN Initial_Installment_Amount>800 AND (Event_Amount -Initial_Installment_Amount)/Initial_Installment_Amount<0.30                            THEN 'Medium Step up'
    WHEN (Initial_Installment_Amount>400 AND Initial_Installment_Amount<=800) AND ((Event_Amount -Initial_Installment_Amount)/Initial_Installment_Amount>=0.20 
       AND (Event_Amount -Initial_Installment_Amount)/Initial_Installment_Amount<0.50)                                            THEN 'Medium Step up'
    WHEN ((Initial_Installment_Amount>200 AND Initial_Installment_Amount<=400) OR (Initial_Installment_Amount>100 AND Initial_Installment_Amount<=200)) 
       AND ((Event_Amount -Initial_Installment_Amount)/Initial_Installment_Amount>=0.30 
       AND (Event_Amount -Initial_Installment_Amount)/Initial_Installment_Amount<1.00)                                            THEN 'Medium Step up'
    WHEN (Initial_Installment_Amount>50 AND Initial_Installment_Amount<=100) AND ((Event_Amount -Initial_Installment_Amount)/Initial_Installment_Amount>=0.50 
       AND (Event_Amount -Initial_Installment_Amount)/Initial_Installment_Amount<2.00)                                            THEN 'Medium Step up'
    WHEN Initial_Installment_Amount<=50  AND ((Event_Amount -Initial_Installment_Amount)/Initial_Installment_Amount>=1.00)                          THEN 'Medium Step up'
    WHEN (Initial_Installment_Amount>400 AND Initial_Installment_Amount<=800) AND (Event_Amount -Initial_Installment_Amount)/Initial_Installment_Amount<0.20        THEN 'Mild Step Up'
    WHEN ((Initial_Installment_Amount>100 AND Initial_Installment_Amount<=400) AND (Event_Amount -Initial_Installment_Amount)/Initial_Installment_Amount<0.30)        THEN 'Mild Step Up'
    WHEN (Initial_Installment_Amount>50 AND Initial_Installment_Amount<=100) AND ((Event_Amount -Initial_Installment_Amount)/Initial_Installment_Amount>=0.30 
       AND  (Event_Amount -Initial_Installment_Amount)/Initial_Installment_Amount<0.50)                                            THEN 'Mild Step Up'
    WHEN Initial_Installment_Amount<=50 AND ((Event_Amount -Initial_Installment_Amount)/Initial_Installment_Amount>=0.50 
    AND  (Event_Amount -Initial_Installment_Amount)/Initial_Installment_Amount<1.00)                                            THEN 'Mild Step Up'                          
     END AS StepUpSegment
     ,StepUpFlag
  FROM PastInstallmentsFinal
 """)

### Join with target_table_profits

In [0]:
target_table_profits = (
    target_table_profits.join(
        FutureInstallmentsFinal,
        exposure_profits["ExposureKeyProfits"] == FutureInstallmentsFinal["ExposureKey_FutureInstallment"],
        "LEFT",
    ).join(
        PastInstallmentsFinal,
        exposure_profits["ExposureKeyProfits"] == PastInstallmentsFinal["ExposureKey_PastInstallment"],
        "LEFT",
    )
)

## Zeroing GBV

In [0]:

ZeroingGBVExposureMaxDate = spark.sql(f""" 
select MAX(BusinessDate) AS MaxBusinessDate
from `{env}-cdh`.derivative.ZeroingGBVExposure
where BusinessDate<{run_date!r}; 
"""
)

max_date = ZeroingGBVExposureMaxDate.collect()[0]['MaxBusinessDate']

# Convert to string in 'yyyy-mm-dd' format if not None
if max_date is not None:
    max_date = max_date.strftime('%Y-%m-%d')  # Ensure string format
else:
    max_date = ""  # Handle None case explicitly

# Use the value in your query
ZeroingGBVExposure = spark.sql(f""" 
SELECT *
FROM `{env}-cdh`.derivative.ZeroingGBVExposure
WHERE BusinessDate = {max_date!r}
""")

ZeroingGBVExposure.createOrReplaceTempView("ZeroingGBVExposure")

PortfolioGBV = spark.sql(f""" 
select *
from `{env}-cdh`.primary.portfolio
where businessdate = {run_date!r} and IsHAPSFlag='true'; 
"""
)

PortfolioGBV.createOrReplaceTempView("PortfolioGBV")

# Assign zerogbvdate as run_date
zerogbvdate = run_date

#all portfolios
ExposuresWithDefineDelay_df_all = spark.sql(f""" 
select 
distinct 
ExposureKey,
AgreementKey,
PortfolioName,
BusinessDate
from `{env}-cdh`.primary.exposure
where ExposureStatusCode='3' 
and exposure.businessdate = {run_date!r}
and exposure.sourcesystem='profits'; 
"""
)

ExposuresWithDefineDelay_df_all.createOrReplaceTempView("ExposuresWithDefineDelay_df_all")

#only haps portfolios
ExposuresWithDefineDelay_df = spark.sql(f""" 
select 
a.ExposureKey,
a.AgreementKey
from ExposuresWithDefineDelay_df_all as a
INNER JOIN PortfolioGBV  as b on a.PortfolioName=b.PortfolioName
WHERE a.businessdate>=b.GBVInitializationDate; 
"""
)

ExposuresWithDefineDelay_df.createOrReplaceTempView("ExposuresWithDefineDelay_df")

# legal actions without the filtering condition
legalaction_zerogbv_all = spark.sql(f""" 
select 
distinct 
ExposureKey,
PortfolioName,
LegalActionDate,
businessdate
from `{env}-cdh`.primary.legalaction 
where legalaction.sourcesystem='qca' and legalaction.LegalActionTypeCode in('4026','4027') and legalaction.businessdate = {run_date!r}
"""  
)

legalaction_zerogbv_all.createOrReplaceTempView("legalaction_zerogbv_all")

# legal actions with applying the filtering condition
legalaction_zerogbv = spark.sql(f""" 
select 
distinct 
A.ExposureKey
from legalaction_zerogbv_all AS A
INNER JOIN  PortfolioGBV  AS b on A.PortfolioName=b.PortfolioName
where A.businessdate>=b.GBVInitializationDate and A.LegalActionDate>=b.GBVInitializationDate;
"""  
)

legalaction_zerogbv.createOrReplaceTempView("legalaction_zerogbv")

# Legal Actions QC Decommission to Arotron
legalaction_zerogbv_all_arotron = spark.sql(f""" 
select  
ex.ExposureKey,
la.PortfolioName,
max(la.LegalActionDate) LegalActionDate,
la.BusinessDate
from `{env}-cdh`.primary.legalaction la
left join (
    select distinct ExposureKey, CDI, BusinessDate
    from `{env}-cdh`.primary.exposure
    where BusinessDate = {run_date!r}
    and SourceSystem='profits'
) ex
on la.CDI = ex.CDI and la.BusinessDate = ex.BusinessDate and la.SourceSystem = 'arotron'
where 1=1 
and ((LegalActionType = 'Είσπραξη Εκπλειστηριάσματος'
and LegalActionTypeCode = 'SYS_BA_LEG_ACT_CPL_LG_36'
and LegalType = 'Ακίνητης Ιδιοκτησίας - Ενυπόθηκο'
and LegalTypeCode = 'SYS_ABW_SUBACTION_CODE_0035'
and LegalActionStepType is NULL
and LegalActionStepTypeCode is NULL 
and CompletionResult = 'ΟΛΙΚΗ ΕΙΣΠΡΑΞΗ ΕΚΠΛΕΙΣΤΗΡΙΑΣΜΑΤΟΣ' 
and CompletionResultCode = 'SYS_ACTION_CPL_IACT_058')
or
(LegalActionType = 'Κατάσχεση κινητών-ακινήτων' 
and LegalActionTypeCode = 'SYS_BA_LEG_ACT_CPL_LG_06'    
and LegalType = 'Ακίνητα' 
and LegalTypeCode = 'SYS_CPL_SUBACT_CODE_1024'  
and LegalActionStepType is NULL 
and LegalActionStepTypeCode is NULL 
and CompletionResult = 'ΟΡΙΣΤΙΚΗ ΑΔΥΝΑΜΙΑ ΕΠΙΒΟΛΗΣ ΚΑΤΑΣΧΕΣΗΣ' 
and CompletionResultCode = 'SYS_ACTION_CPL_IACT_065'))
and la.BusinessDate = {run_date!r}
group by all
"""  
)

legalaction_zerogbv_all_arotron.createOrReplaceTempView("legalaction_zerogbv_all_arotron") # Legal Actions QC Decommission to Arotron

# Legal Actions QC Decommission to Arotron
legalaction_zerogbv_arotron = spark.sql(f""" 
select 
distinct 
A.ExposureKey
from legalaction_zerogbv_all_arotron AS A
INNER JOIN  PortfolioGBV  AS b on A.PortfolioName=b.PortfolioName
where A.BusinessDate>=b.GBVInitializationDate and A.LegalActionDate>=b.GBVInitializationDate;
"""  
)

# Legal Actions QC Decommission to Arotron
legalaction_zerogbv_arotron.createOrReplaceTempView("legalaction_zerogbv_arotron")


# all collaterals without the filtering date
collateral_zerogbv_all = spark.sql(f""" 
select 
distinct 
agreementkey, CategoryCode, CollateralStatus, PortfolioName, businessdate
from `{env}-cdh`.primary.collateral
where businessdate = {run_date!r}
and sourcesystem='thesis';
"""
)

collateral_zerogbv_all.createOrReplaceTempView("collateral_zerogbv_all")

# collaterals after applying the filtering date 
collateral_zerogbv = spark.sql(f""" 
select 
distinct 
A.agreementkey, A.CategoryCode, A.CollateralStatus
from collateral_zerogbv_all as a
INNER JOIN  PortfolioGBV  AS b on A.PortfolioName=b.PortfolioName
where a.businessdate>=b.GBVInitializationDate;
"""
)

collateral_zerogbv.createOrReplaceTempView("collateral_zerogbv")

# collaterals prior to businessdate 
collateral_zerogbv_priortobusinessdate_all = spark.sql(f""" 
select 
distinct 
agreementkey, CategoryCode, CollateralStatus, PortfolioName, BusinessDate
from `{env}-cdh`.primary.collateral
where businessdate<{run_date!r} and sourcesystem='thesis';
"""
)

collateral_zerogbv_priortobusinessdate_all.createOrReplaceTempView("collateral_zerogbv_priortobusinessdate_all")

# collaterals prior to businessdate and after the filtering date
collateral_zerogbv_priortobusinessdate = spark.sql(f""" 
select 
distinct 
a.agreementkey, a.CategoryCode, a.CollateralStatus
from collateral_zerogbv_priortobusinessdate_all as a
INNER JOIN  PortfolioGBV  AS b on A.PortfolioName=b.PortfolioName
where a.businessdate>=b.GBVInitializationDate;
"""
)

collateral_zerogbv_priortobusinessdate.createOrReplaceTempView("collateral_zerogbv_priortobusinessdate")

NoFurtherLegalActions_DF = spark.sql(f""" 
select 
distinct 
exposure.ExposureKey
from ExposuresWithDefineDelay_df AS exposure
inner join legalaction_zerogbv as legalaction on legalaction.ExposureKey = substring(exposure.ExposureKey,4,50); 
"""
)

NoFurtherLegalActions_DF.createOrReplaceTempView("NoFurtherLegalActions_DF")

# Legal Actions QC Decommission to Arotron
NoFurtherLegalActions_DF_arotron = spark.sql(f""" 
select 
distinct 
exposure.ExposureKey
from ExposuresWithDefineDelay_df AS exposure
inner join legalaction_zerogbv_arotron as legalaction on legalaction.ExposureKey = exposure.ExposureKey; 
"""
)

# Legal Actions QC Decommission to Arotron
NoFurtherLegalActions_DF_arotron.createOrReplaceTempView("NoFurtherLegalActions_DF_arotron")


ExposuresWithInactiveREs_DF = spark.sql(f""" 
select distinct
ExposureKey
from ExposuresWithDefineDelay_df as exposure
inner join collateral_zerogbv as collateral on  exposure.agreementkey = collateral.agreementkey
where CategoryCode = 'Επί ακινήτων' and CollateralStatus='Ανενεργή'; 
"""
)

ExposuresWithInactiveREs_DF.createOrReplaceTempView("ExposuresWithInactiveREs_DF")

ExposuresWithInactiveNonREs_DF = spark.sql(f""" 
select distinct
ExposureKey
from ExposuresWithDefineDelay_df as exposure
inner join collateral_zerogbv as collateral on  exposure.agreementkey = collateral.agreementkey
inner join  `{env}-cdh`.reference.dpo_nonre_parameters as p on p.NonRECollateralCategory = collateral.CategoryCode
where CollateralStatus='Ανενεργή' and LV_rate!= 0; 
"""
)

ExposuresWithInactiveNonREs_DF.createOrReplaceTempView("ExposuresWithInactiveNonREs_DF")

ExposuresWithInactiveCollaterals_DF = spark.sql(f""" 
select *
from ExposuresWithInactiveREs_DF
UNION
select * 
from ExposuresWithInactiveNonREs_DF; 
"""
).distinct()

ExposuresWithInactiveCollaterals_DF.createOrReplaceTempView("ExposuresWithInactiveCollaterals_DF")


ExposuresWithActiveREs_DF = spark.sql(f""" 
select distinct
ExposureKey
from ExposuresWithDefineDelay_df as exposure
inner join collateral_zerogbv as collateral on  exposure.agreementkey = collateral.agreementkey
where CategoryCode = 'Επί ακινήτων' and CollateralStatus!='Ανενεργή'; 
"""
)

ExposuresWithActiveREs_DF.createOrReplaceTempView("ExposuresWithActiveREs_DF")

ExposuresWithActiveNonREs_DF = spark.sql(f""" 
select distinct
ExposureKey
from ExposuresWithDefineDelay_df as exposure
inner join collateral_zerogbv as collateral on  exposure.agreementkey = collateral.agreementkey
inner join  `{env}-cdh`.reference.dpo_nonre_parameters as p on p.NonRECollateralCategory = collateral.CategoryCode
where CollateralStatus!='Ανενεργή' and LV_rate!= 0; 
"""
)

ExposuresWithActiveNonREs_DF.createOrReplaceTempView("ExposuresWithActiveNonREs_DF")

ExposuresWithActiveCollaterals_DF = spark.sql(f""" 
select *
from ExposuresWithActiveREs_DF
UNION
select * 
from ExposuresWithActiveNonREs_DF; 
"""
).distinct()

ExposuresWithActiveCollaterals_DF.createOrReplaceTempView("ExposuresWithActiveCollaterals_DF")

#### Find exposures that had active collaterals after the filter date and prior to businessdate 
ExposuresWithActiveREsPriorToBusinessDate_DF = spark.sql(f""" 
select distinct
ExposureKey
from ExposuresWithDefineDelay_df as exposure
inner join collateral_zerogbv_priortobusinessdate as collateral on  exposure.agreementkey = collateral.agreementkey
where CategoryCode = 'Επί ακινήτων' and CollateralStatus!='Ανενεργή'; 
"""
)

ExposuresWithActiveREsPriorToBusinessDate_DF.createOrReplaceTempView("ExposuresWithActiveREsPriorToBusinessDate_DF")

ExposuresWithActiveNonREsPriorToBusinessDate_DF = spark.sql(f""" 
select distinct
ExposureKey
from ExposuresWithDefineDelay_df as exposure
inner join collateral_zerogbv_priortobusinessdate as collateral on  exposure.agreementkey = collateral.agreementkey
inner join  `{env}-cdh`.reference.dpo_nonre_parameters as p on p.NonRECollateralCategory = collateral.CategoryCode
where CollateralStatus!='Ανενεργή' and LV_rate!= 0; 
"""
)

ExposuresWithActiveNonREsPriorToBusinessDate_DF.createOrReplaceTempView("ExposuresWithActiveNonREsPriorToBusinessDate_DF")

ExposuresWithActiveCollateralsPriorToBusinessDate_DF = spark.sql(f""" 
select *
from ExposuresWithActiveREsPriorToBusinessDate_DF
UNION
select * 
from ExposuresWithActiveNonREsPriorToBusinessDate_DF; 
"""
).distinct()

ExposuresWithActiveCollateralsPriorToBusinessDate_DF.createOrReplaceTempView("ExposuresWithActiveCollateralsPriorToBusinessDate_DF")

#except, compare two dataframes (anenerga from energa)
ExposuresWithOnlyInactiveCollaterals_DF = spark.sql(f""" 
select a.*
FROM ExposuresWithInactiveCollaterals_DF as a
INNER JOIN ExposuresWithActiveCollateralsPriorToBusinessDate_DF as b on a.ExposureKey=b.ExposureKey
EXCEPT
SELECT * 
FROM ExposuresWithActiveCollaterals_DF
"""
)

ExposuresWithOnlyInactiveCollaterals_DF.createOrReplaceTempView("ExposuresWithOnlyInactiveCollaterals_DF")

DistinctExposuresWithDefineDelay_df=ExposuresWithDefineDelay_df

DistinctExposuresWithDefineDelay_df.createOrReplaceTempView("DistinctExposuresWithDefineDelay_df")


# DF with exposures that have:
# 1. CappedDate and CappedAmount
# 2. GBVTriggerEvent (Can be Legal Action or No Active Collateral)
# The above two can cease to apply at any point, and GBV will recalculate (also GBVCapped and GBVTriggerEvent can be recalculated/cancelled)
# Legal Actions QC Decommission to Arotron
zeroGBV_fields = spark.sql(f""" 
SELECT
exposure_profits.ExposureKeyProfits as ExposureKey,
CASE WHEN ZeroingGBVExposure.GBVTriggerEventDate is not null
        AND (coalesce(NoFurtherLegalActions_DF.ExposureKey, NoFurtherLegalActions_DF_arotron.ExposureKey) is NOT null OR ExposuresWithOnlyInactiveCollaterals_DF.ExposureKey is NOT null) then TRUE -- first check auctilary table
    WHEN coalesce(NoFurtherLegalActions_DF.ExposureKey, NoFurtherLegalActions_DF_arotron.ExposureKey) is not null then TRUE -- when no further legal actions 
    WHEN ExposuresWithOnlyInactiveCollaterals_DF.ExposureKey is not null then TRUE-- when only inactive collaterals
    ELSE FALSE
END AS GBVTriggerEvent,
CASE WHEN ZeroingGBVExposure.GBVTriggerEventDate is not null  
        AND (coalesce(NoFurtherLegalActions_DF.ExposureKey, NoFurtherLegalActions_DF_arotron.ExposureKey) is NOT null OR ExposuresWithOnlyInactiveCollaterals_DF.ExposureKey is NOT null) then ZeroingGBVExposure.GBVTriggerEventDate -- first check auctilary table
    WHEN coalesce(NoFurtherLegalActions_DF.ExposureKey, NoFurtherLegalActions_DF_arotron.ExposureKey) is not null then {zerogbvdate!r} -- when no further legal actions 
    WHEN ExposuresWithOnlyInactiveCollaterals_DF.ExposureKey is not null then {zerogbvdate!r} -- when only inactive collaterals 
    ELSE NULL
END AS GBVTriggerEventDate,
CASE 
    WHEN coalesce(NoFurtherLegalActions_DF.ExposureKey, NoFurtherLegalActions_DF_arotron.ExposureKey) is not null then 'No further legal actions' -- when no further legal actions 
    WHEN ExposuresWithOnlyInactiveCollaterals_DF.ExposureKey is not null then 'No active collaterals'
    ELSE NULL
END AS GBVTriggerEventType,
CASE WHEN ZeroingGBVExposure.GBVTriggerEventDate is not null 
         AND (coalesce(NoFurtherLegalActions_DF.ExposureKey, NoFurtherLegalActions_DF_arotron.ExposureKey) is NOT null OR ExposuresWithOnlyInactiveCollaterals_DF.ExposureKey is NOT null) then ZeroingGBVExposure.GBVTailExposure -- first check auctilary table
    WHEN coalesce(NoFurtherLegalActions_DF.ExposureKey, NoFurtherLegalActions_DF_arotron.ExposureKey) is not null then exposure_profits.GBVcalc -- when no further legal actions 
    WHEN ExposuresWithOnlyInactiveCollaterals_DF.ExposureKey is not null then exposure_profits.GBVcalc
    ELSE NULL 
END AS GBVTailExposure,
CASE WHEN ZeroingGBVExposure.GBVCappedDate is not null and DistinctExposuresWithDefineDelay_df.ExposureKey is not null then ZeroingGBVExposure.GBVCappedAmount 
    WHEN DistinctExposuresWithDefineDelay_df.ExposureKey is not null then exposure_profits.GBVcalc
    ELSE NULL
END as GBVCappedAmount,
CASE WHEN ZeroingGBVExposure.GBVCappedDate is not null and DistinctExposuresWithDefineDelay_df.ExposureKey is not null then ZeroingGBVExposure.GBVCappedDate 
    WHEN DistinctExposuresWithDefineDelay_df.ExposureKey is not null then {zerogbvdate!r}
    ELSE NULL
END as GBVCappedDate,
exposure_profits.GBVcalc AS GBVFunctionResult
FROM target_table_profits as exposure_profits 
LEFT JOIN DistinctExposuresWithDefineDelay_df on exposure_profits.ExposureKeyProfits = DistinctExposuresWithDefineDelay_df.ExposureKey
LEFT JOIN ExposuresWithOnlyInactiveCollaterals_DF ON DistinctExposuresWithDefineDelay_df.ExposureKey = ExposuresWithOnlyInactiveCollaterals_DF.ExposureKey
LEFT JOIN NoFurtherLegalActions_DF on DistinctExposuresWithDefineDelay_df.ExposureKey = NoFurtherLegalActions_DF.ExposureKey
LEFT JOIN NoFurtherLegalActions_DF_arotron on DistinctExposuresWithDefineDelay_df.ExposureKey = NoFurtherLegalActions_DF_arotron.ExposureKey
LEFT JOIN ZeroingGBVExposure ON DistinctExposuresWithDefineDelay_df.ExposureKey = ZeroingGBVExposure.ExposureKey 
"""
)

zeroGBV_fields.createOrReplaceTempView("zeroGBV_fields")

#now having the above columns we can finally calculate the GBV (if its not already calculated)
zeroGBV_fields_final = spark.sql(f""" 
SELECT
*,
CASE 
    WHEN GBVTriggerEventType IS NULL THEN LEAST(GBVFunctionResult, GBVCappedAmount)
    WHEN GBVTriggerEventType = 'No active collaterals' THEN LEAST(GBVFunctionResult,GBVCappedAmount, 0.15 * GBVTailExposure)
    WHEN GBVTriggerEventType = 'No further legal actions' THEN 0
END AS FinalGBV
FROM zeroGBV_fields
"""
)


In [0]:
target_table_profits = (
    target_table_profits.join(
        zeroGBV_fields_final,
        exposure_profits["ExposureKeyProfits"] == zeroGBV_fields_final["ExposureKey"],
        "LEFT",
    )
)

## Final select profits

In [0]:
# ================== Selection of Final Columns ==================
result_df_profits = target_table_profits.select(
    target_table_profits.ExposureIdNew.alias("ExposureDerivativeID"),
    exposure_profits.ExposureKeyProfits.alias("ExposureKey"),
    lit('profits').alias("SourceSystem"),
    exposure_profits.PortfolioKey.alias("PortfolioKey"),
    exposure_profits.PortfolioName.alias("PortfolioName"),
    lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
        "IngestionTimestamp"
    ),
    lit(None).alias("BatchId"),
    exposure_profits.ExposureKeyProfits.alias("ExposureKeyProfits"),
    exposure_profits.ExposureSourceSystemKeyProfits.alias("ExposureSourceSystemKeyProfits"),
    lit(None).alias("ExposureKeyQCCepal"),
    lit(None).alias("ExposureSourceSystemKeyQCCepal"),
    exposure_qca.ExposureKeyQCA.alias("ExposureKeyQCAlpha"),
    exposure_qca.ExposureSourceSystemKeyQCA.alias("ExposureSourceSystemKeyQCAlpha"),
    lit(None).alias("ExposureKeyArotron"),
    lit(None).alias("ExposureSourceSystemKeyArotron"),
    exposure_profits.ExposureDebtorKey.alias("ExposureDebtorKey"), 
    exposure_qca.CaseCodeQCA.alias("CaseCode"),#new
    exposure_profits.AgreementKeyProfits.alias("AgreementKey"), 
    exposure_profits.PartyKey.alias("PartyKey"), 
    exposure_profits.PartyName.alias("PartyName"),
    exposure_profits.CDI.alias("CDI"), 
    exposure_profits.PartyTaxRegistrationNumber.alias("PartyTaxRegistrationNumber"), 
    exposure_profits.PartyRelationshipTypeCode.alias("PartyRelationshipTypeCode"), 
    exposure_profits.PartyRelationshipType.alias("PartyRelationshipType"), 
    exposure_profits.PrimaryPartyFlag.alias("PrimaryPartyFlag"), 
    exposure_qca.AllBorrowersDeceasedFlagQCA.alias("AllBorrowersDeceasedFlag"),
    exposure_profits.ProductKey.alias("ProductKey"), 
    exposure_profits.ProductCode.alias("ProductCode"), 
    target_table_profits.SubProductDescription.cast("String").alias("SubProductDescription"), 
    exposure_profits.ExposureTypeCode.alias("ExposureTypeCode"), 
    exposure_profits.ExposureType.alias("ExposureType"), 
    exposure_profits.ExposureStatusCode.alias("ExposureStatusCode"), 
    exposure_profits.ExposureStatusCalc.alias("ExposureStatus"), 
    exposure_profits.ExposureStatusdate.alias("ExposureStatusDate"), 
    exposure_profits.Opendate.alias("Opendate"), 
    exposure_profits.ActiveExpiryDate.alias("ActiveExpiryDate"), 
    exposure_profits.Maturitydate.alias("Maturitydate"), 
    exposure_profits.OriginBank.alias("OriginBank"), 
    exposure_profits.OriginationAmount.alias("OriginationAmount"), 
    target_table_profits.InitialAssignmentAmount.cast("decimal(19,2)").alias("InitialAssignmentAmount"), 
    exposure_profits.Currency.alias("Currency"), 
    exposure_profits.BankingUnitCode.alias("BankingUnitCode"), 
    exposure_profits.BankingUnit.alias("BankingUnit"), 
    exposure_profits.MechanismCode.alias("MechanismCode"), 
    exposure_profits.Mechanism.alias("Mechanism"), 
    exposure_qca.BranchCodeQCA.alias("BranchCode"),
    exposure_qca.BranchQCA.alias("Branch"),
    exposure_profits.PurposeReasonCode.alias("PurposeReasonCode"), 
    exposure_profits.PurposeReason.alias("PurposeReason"), 
    exposure_profits.PurposeKTCode.alias("PurposeKTCode"), 
    exposure_profits.PurposeKT.alias("PurposeKT"), 
    exposure_profits.MaterialityStartDate.alias("MaterialityStartDate"),
    exposure_profits.DaysPastDueWithMateriality.cast("int").alias("DaysPastDueWithMateriality"),
    exposure_profits.DaysPastDue.alias("DaysPastDue"), 
    exposure_profits.DaysPastDueinterest.alias("DaysPastDueinterest"), 
    target_table_profits.FirstArrDate.alias("FirstArreardate"), 
    exposure_profits.Overduedate.alias("Overduedate"), 
    exposure_non_active_profits.ClosedDate.alias("ClosedDate"),
    exposure_profits.IsActiveFlag.alias("IsActiveFlag"),
    exposure_profits.IsActiveFlag.cast("smallint").alias("IsActiveFlagInt"),
    exposure_profits.FirstInstallmentDate.alias("FirstInstallmentDate"),
    exposure_profits.BucketCode.alias("BucketCode"),
    exposure_profits.Bucket.alias("Bucket"),
    AccountTransaction_Write_Off_non_cond.TransactionDateMin.alias("WriteOffDate"),
    target_table_profits.writeoffamountprofits.cast("decimal (19,2)").alias("WriteoffAmount"),
    lps_by_exposure["LPSCode"].cast("string").alias("LPSCode"),
    lps_by_exposure["LPS"].cast("string").alias("LPS"),
    lit(None).cast("string").alias("LPSEng"),
    lps_by_exposure["LPSDate"].cast("string").alias("LPSDate"),
    target_table_profits.DenouncedDateCalc.alias("DenouncedDate"),
    exposure_profits.DaysPastDuePrincipal.alias("DaysPastDuePrincipal"),
    exposure_profits.ExpectedOverduedate.alias("ExpectedOverduedate"),
    exposure_profits.OutstandingTotalBalance.cast("decimal(19,2)").alias("OutstandingTotalBalance"),
    target_table_profits.OffBalanceCalc.cast("decimal(19,2)").alias("OffBalance"),
    target_table_profits.OnBalanceCalc.cast("decimal(19,2)").alias("OnBalance"),
    exposure_profits.DueAmountCalc.cast("decimal(26,4)").alias("DueAmount"), 
    target_table_profits.BoGBalance.cast("decimal(19,2)").alias("BogBalance"),
    coalesce(target_table_profits.FinalGBV,target_table_profits.GBVCalc).cast("double").alias("GBV"),
    target_table_profits.GBVTriggerEvent.cast("Boolean").alias("GBVTriggerEventFlag"),
    target_table_profits.GBVTriggerEventDate.cast("Date"),
    target_table_profits.GBVTriggerEventType.cast("String"),
    target_table_profits.GBVTailExposure.cast("double").alias("TailExposure"),
    target_table_profits.GBVCappedAmount.cast("double"),
    target_table_profits.GBVCappedDate.cast("Date"),
    target_table_profits.GBVCalc.cast("double").alias("GBVFunctionResult"),
    target_table_profits.EverDenouncedDateCalc.alias("EverDenouncedDate"),
    target_table_profits.EverDenouncedFlagCalc.cast("boolean").alias("EverDenouncedFlag"),
    target_table_profits.IsSecuredFlagCalc.alias("IsSecuredFlag"),
    target_table_profits.InterestNextDueDateCalc.alias("InterestNextDueDate"),
    target_table_profits.ClosedReasonCalc.alias("ClosedReasonCode"),  
    target_table_profits.ClosedReasonCalc.alias("ClosedReason"),     
    target_table_profits.PEERestructureTypeCodeCalc.alias("PEERestructureTypeCode"),       
    exposure_profits.RestructuringPEETypeCodeProfits.alias("PEEAdjustmentCode"),      #not calculated
    target_table_profits.DefaultdateCalc.alias("Defaultdate"), 
    target_table_profits.LatestDefaultDateCalc.cast("date").alias("LatestDefaultDate"),
    target_table_profits.NPLDefaultAmountCalc.alias("NPLDefaultAmount"),
    target_table_profits.NonConditionalWriteOffDateCalc.alias("NonConditionalWriteOffDate"),
    target_table_profits.NonConditionalWriteOffAmountCalc.cast("decimal(19,2)").alias("NonConditionalWriteOffAmount"),
    target_table_profits.ConditionalWriteOffDateCalc.alias("ConditionalWriteOffDate"),
    target_table_profits.ConditionalWriteOffAmountCalc.cast("decimal(19,2)").alias("ConditionalWriteOffAmount"),
    Payment_Schedule_Loan_Next_Install.NextInstallmentLoanDate.cast("date").alias("NextInstallmentDate"),
    Payment_schedule_next_capital_interest_installment.NextInstallmentInterestDate.alias("NextInstallmentInterestDate"),
    Payment_schedule_next_capital_installment.NextInstallmentCapitalDate.alias("NextInstallmentCapitalDate"),
    target_table_profits.NextInstallmentMinimumDateCalc.alias("NextInstallmentMinimumDate"),
    Payment_Schedule_Loan_Next_Install.NextInstallmentAmount.cast("decimal(19,2)").alias("NextInstallmentAmount"),
    Payment_schedule_next_capital_installment.NextInstallmentCapitalAmount.cast("decimal(19,2)").alias("NextInstallmentCapitalAmount"),
    target_table_profits.ExposureTenorCalc.alias("ExposureTenor"), 
    target_table_profits.IsSoldFlagCalc.cast("boolean").alias("IsSoldFlag"),
    target_table_profits.IsSubsidyFlagCalc.alias("IsSubsidyFlag"), 
    target_table_profits.IsGuaranteeFlagCalc.alias("IsGuaranteeFlag"), 
    exposure_profits.MasterExposureKey.alias("MasterExposureKey"), 
    exposure_profits.SlaveFlag.alias("SlaveFlag"), 
    target_table_profits.PaymentOrderFlag_Dk.cast("boolean").alias("PaymentOrderFlag"),
    target_table_profits.PaymentOrderDate_Dk.cast("date").alias("PaymentOrderDate"),
    target_table_profits.NumberOfUnpaidBillsCalc.cast("string").alias("NumberOfUnpaidBills"),
    target_table_profits.SettlementBucketCalc.alias("SettlementBucketCode"),
    target_table_profits.SettlementBucketCalc.alias("SettlementBucket"),
    target_table_profits.DaysPastDueFinal.cast("int").alias("DaysPastDueFinal"),
    target_table_profits.HasActiveSettlementFlagCalc.alias("HasActiveSettlementFlag"),
    Law_profits.UnderLawTypeCode.alias("UnderLawTypeCode"),
    Law_profits.UnderLawType.alias("UnderLawType"),
    Law_profits.LawStatusCode.alias("LawStatusCode"),
    Law_profits.LawStatus.alias("LawStatus"),
    Law_profits.LawApplicationStatusDate.alias("LawApplicationStatusDate"),
    Law_profits.LawCourtDate.alias("LawCourtDate"),
    Law_profits.LawProvisionalCourtDecisionDate.alias("LawProvisionalCourtDecisionDate"),
    exposure_qca.InstalmentFrequencyQCA.alias("InstalmentFrequency"),
    exposure_profits.InstallmentFixedAmount.alias("InstallmentFixedAmount"), 
    exposure_profits.NumberOfInstallments.alias("NumberOfInstallments"), 
    exposure_profits.PreviousInstallmentdate.alias("PreviousInstallmentdate"), 
    exposure_profits.LastBillingdate.alias("LastBillingdate"), 
    exposure_profits.LastRestructuringTypeCode.alias("LastRestructuringTypeCode"), 
    exposure_profits.LastRestructuringType.alias("LastRestructuringType"), 
    exposure_profits.LastRestructuringDate.alias("LastRestructuringDate"), 
    exposure_profits.RestructuringPEEDate.alias("RestructuringPEEDate"), 
    exposure_profits.RestructuringPEETypeCodeProfits.alias("RestructuringPEETypeCode"), 
    exposure_profits.RestructuringPEEType.alias("RestructuringPEEType"), 
    exposure_qca.ObjectiveStatusCodeQCA.alias("ObjectiveStatusCode"),
    exposure_qca.ObjectiveStatusQCA.alias("ObjectiveStatus"),
    exposure_qca.ObjectiveStatusdateQCA.alias("ObjectiveStatusdate"),
    exposure_qca.TreatmentBranchCodeQCA.alias("TreatmentBranchCode"),
    exposure_qca.TreatmentBranchQCA.alias("TreatmentBranch"),
    target_table_profits.GracePeriodEndDateCalc.alias("GracePeriodEnddate"), 
    exposure_profits.GracePeriodTypeCode.alias("GracePeriodTypeCode"), 
    exposure_profits.GracePeriodType.alias("GracePeriodType"), 
    exposure_profits.GracePeriodinterestPayableFlag.alias("GracePeriodinterestPayableFlag"), 
    exposure_profits.GracePeriodNextinterestdate.alias("GracePeriodNextinterestdate"), 
    exposure_profits.GracePeriodinterestFrequency.alias("GracePeriodinterestFrequency"), 
    exposure_profits.Principal.alias("Principal"), 
    exposure_profits.interest.alias("interest"), 
    exposure_profits.Unrealizedinterest.alias("Unrealizedinterest"), 
    exposure_profits.Commission.alias("Commission"), 
    exposure_profits.Expense.alias("Expense"), 
    exposure_profits.Accrued.alias("Accrued"), 
    exposure_profits.DuePrincipal.alias("DuePrincipal"), 
    exposure_profits.Dueinterest.alias("Dueinterest"), 
    exposure_profits.DueUnrealizedinterest.alias("DueUnrealizedinterest"), 
    exposure_profits.DueCommission.alias("DueCommission"), 
    exposure_profits.DueExpense.alias("DueExpense"), 
    exposure_profits.DueAccrued.alias("DueAccrued"), 
    exposure_profits.Penaltyinterest.alias("Penaltyinterest"), 
    exposure_profits.PenaltyUnrealizedinterest.alias("PenaltyUnrealizedinterest"), 
    exposure_profits.InterestRateCalc.cast("decimal(19,6)").alias("InterestRate"), 
    exposure_profits.interestCode.alias("interestCode"), 
    exposure_profits.interestDescription.alias("interestDescription"), 
    exposure_profits.BaseRate.alias("BaseRate"), 
    exposure_profits.BaseCode.alias("BaseCode"), 
    exposure_profits.BaseDescription.alias("BaseDescription"), 
    exposure_profits.N128Rate.alias("N128Rate"), 
    exposure_profits.SpreadRate.alias("SpreadRate"), 
    exposure_profits.PenaltyRate.alias("PenaltyRate"), 
    when(
        (exposure_profits["InterestRateTypeCode"] == 'Fixed') & (exposure_profits["InterestDuration"] != 999),
        'Fixed Periodic'
    ).otherwise(exposure_profits["InterestRateTypeCode"]).alias("InterestRateTypeCode"),
    exposure_profits.InterestRateType.alias("InterestRateType"), 
    exposure_profits.SpreadCurrencyCode.alias("SpreadCurrencyCode"), 
    exposure_profits.SpreadCurrency.alias("SpreadCurrency"), 
    exposure_profits.SpreadPeriod.alias("SpreadPeriod"), 
    exposure_profits.InterestDuration.alias("InterestDuration"), 
    exposure_profits.DayCounts.alias("DayCounts"), 
    exposure_profits.InterestFrequency.alias("InterestFrequency"), 
    exposure_profits.InterestFixedDate.alias("InterestFixedDate"), 
    exposure_profits.interestValiditydate.alias("interestValiditydate"), 
    target_table_profits.ValidDate.cast("date").alias("InterestChangeStartdate"),
    exposure_profits.InterestChangeEnddate.alias("InterestChangeEnddate"), 
    exposure_profits.InterestRateUpdatedate.alias("InterestRateUpdatedate"), 
    exposure_profits.InterestCalcStopFlag.alias("InterestCalcStopFlag"), 
    exposure_profits.AutoRenewalFlag.alias("AutoRenewalFlag"), 
    target_table_profits.AssetSegmentCodeCalc.alias("AssetSegmentCode"), 
    target_table_profits.AssetSegmentCalc.alias("AssetSegment"), 
    exposure_profits.DiasNumber.alias("DiasNumber"), 
    exposure_profits.ProvisionFrozenInterestAmount.alias("ProvisionFrozenInterestAmount"), 
    exposure_qca.AccountNticeBUKeyQCA.alias("AccountNticeBUKey"),
    exposure_qca.SayFlagQCA.alias("SayFlag"),
    exposure_qca.InsuranceCompanyQCA.alias("InsuranceCompany"),
    exposure_qca.PhysicalNumberQCA.alias("PhysicalNumber"),
    exposure_qca.PreCourtDateQCA.alias("PreCourtDate"),
    exposure_qca.CourtDateQCA.alias("CourtDate"),
    exposure_qca.ContractSigningDateQCA.alias("ContractSigningDate"),
    exposure_profits.ExposureLimitAmount.alias("ExposureLimitAmount"),
    exposure_profits.BaloonAmount.alias("BaloonAmount"),
    exposure_qca.AdministrationStatusCodeQCA.alias("AdministrationStatusCode"),
    exposure_qca.AdministrationStatusQCA.alias("AdministrationStatus"),
    exposure_qca.TenorQCA.alias("Tenor"),
    exposure_qca.SPVCodeQCA.alias("SPVCode"),
    exposure_qca.SPVQCA.alias("SPV"),
    exposure_profits.TotalOverdueInsterest.alias("TotalOverdueInsterest"), 
    exposure_profits.TotalOverdueInstallments.alias("TotalOverdueInstallments"), 
    exposure_profits.OutstandingTotalBalanceMLT.cast("decimal(19,2)").alias("OutstandingTotalBalanceMLT"), 
    target_table_profits.NextInterestChangeDateCalc.alias("NextInterestChangeDate"),
    target_table_profits.PrincipalNextDueDateCalc.alias("PrincipalNextDueDate"),            
    FutureInstallmentsFinal.InstallmentIncreaseGroup.alias("FutureInstallmentIncreaseGroup"),    
    FutureInstallmentsFinal.InitialInstallmentGroup.alias("FutureInitialInstallmentGroup"),    
    FutureInstallmentsFinal.StepUpSegment.alias("FutureStepUpSegment"),
    FutureInstallmentsFinal.StepUpFlag.alias("FutureStepUpIndicator"),
    PastInstallmentsFinal.InstallmentIncreaseGroup.alias("PastInstallmentIncreaseGroup"),    
    PastInstallmentsFinal.InitialInstallmentGroup.alias("PastInitialInstallmentGroup"),    
    PastInstallmentsFinal.StepUpSegment.alias("PastStepUpSegment"),
    PastInstallmentsFinal.StepUpFlag.alias("PastStepUpIndicator"),
    target_table_profits.Principal_Dk.alias("TotalPrincipal"),
    target_table_profits.Interest_Dk.alias("TotalInterest"), 
    target_table_profits.Accruals_Dk.alias("TotalAccruals"),
    target_table_profits.Expense_Dk.alias("TotalExpense"), 
    target_table_profits.Coupon_Dk.alias("Coupon"),
    target_table_profits.ActiveModificationProgramStatus_Dk.alias("ActiveModificationProgramStatus"), 
    target_table_profits.DenouncementIndicator_Dk.alias("DenouncementIndicator"), 
    target_table_profits.ActiveExposureRestructureStartDate_Dk.cast("date").alias("ActiveExposureRestructureStartDate"),
    target_table_profits.MonthsOfCleanPayHistory_Dk.alias("MonthsOfCleanPayHistory"),
    target_table_profits.TotalPaymentAgreementPaid_Dk.alias("TotalPaymentAgreementPaid"),
    target_table_profits.TotalPaymentAgreementDue_Dk.alias("TotalPaymentAgreementDue"),
    target_table_profits.ExposureRestructureOrigination_Dk.alias("ExposureRestructureOrigination"),
    target_table_profits.ExposureRestructureStatus_Dk.alias("ExposureRestructureStatus"),
    target_table_profits.ExposureRestructureType_Dk.alias("ExposureRestructureType"),
    target_table_profits.CodeofConductStage_Qca_Dk.alias("CodeofConductStage"),
    exposure_qca.LetterStatusCodeQCA.alias("LetterStatusCode"),
    exposure_qca.LetterStatusQCA.alias("LetterStatus"),
    exposure_qca.LetterStatusDateQCA.alias("LetterStatusDate"),
    exposure_qca.CoCStatusCodeQCA.alias("CoCStatusCode"),
    exposure_qca.CoCStatusQCA.alias("CoCStatus"),
    exposure_qca.CoCAccountStatusDateQCA.alias("CoCAccountStatusDate"),
    exposure_qca.FirstLetterDateQCA.alias("FirstLetterDate"),
    exposure_qca.SecondLetterDateQCA.alias("SecondLetterDate"),
    exposure_qca.ThirdLetterDateQCA.alias("ThirdLetterDate"),
    exposure_qca.FirstLetterReminderDateQCA.alias("FirstLetterReminderDate"),
    exposure_qca.FirstLetterFlagQCA.alias("FirstLetterFlag"),
    exposure_qca.FirstLetterReminderFlagQCA.alias("FirstLetterReminderFlag"),
    exposure_qca.SecondLetterFlagQCA.alias("SecondLetterFlag"),
    exposure_qca.ThirdLetterFlagQCA.alias("ThirdLetterFlag"),
    exposure_qca.ExclusionDateQCA.alias("ExclusionDate"),
    exposure_qca.ExclusionReasonCodeQCA.alias("ExclusionReasonCode"),
    exposure_qca.ExclusionReasonQCA.alias("ExclusionReason"),
    target_table_profits.L3869Flag_Dk.cast("int").alias("L3869Flag"),
    target_table_profits.L3869Status_Dk.cast("string").alias("L3869Status"),
    target_table_profits.L3869SubmissionDate_Dk.cast("date").alias("L3869SubmissionDate"),
    target_table_profits.L3869ProvisionalCourtDecisionDate_Dk.cast("date").alias("L3869ProvisionalCourtDecisionDate"),
    target_table_profits.L3869CourtDate_Dk.cast("date").alias("L3869CourtDate"),
    target_table_profits.PrincipalAtDefault_Dk.alias("PrincipalAtDefault"),
    target_table_profits.InterestAtDefault_Dk.alias("InterestAtDefault"),
    target_table_profits.AmountDueAtDefault_Dk.cast("decimal(26,4)").alias("AmountDueAtDefault"),
    target_table_profits.LastModificationActionType_Dk.alias("LastModificationActionType"),
    target_table_profits.LastModificationDate_Dk.alias("LastModificationDate"), 
    target_table_profits.HasBalloonFlag,
    target_table_profits.HasZeroInterestTodayFlag,
    target_table_profits.InterestRateTypeTaxonomy,
    target_table_profits.InterestRateResetInterval,
    target_table_profits.HasSplitBalanceFlag,
    target_table_profits.FixInterestEndDate,
    target_table_profits.RepaymentMethod,
    target_table_profits.RepaymentFrequency,
    target_table_profits.HasStepUpInstallmentFlag,
    target_table_profits.UnpaidInterestHandling,
    target_table_profits.HasFutureSpreadFlag,
    target_table_profits.IsBulletFlag,
    target_table_profits.HasFutureInterestChangesFlag,
    target_table_profits.OCWFlag,
    target_table_profits.OCWApplicationStatus,
    target_table_profits.L3869InterestFreePeriodEndDate,
    target_table_profits.InArrearsFlag,
    target_table_profits.HasPaymentPlanFlag,
    target_table_profits.DateLastCurrent,
    target_table_profits.DateLastInArrears,     
    Payment_Schedule_Loan_Last_Install.LastInstallmentLoanDate.cast("date").alias("LastInstallmentDate"),
    target_table_profits.SettlementDaysPastDue, 
    exposure_qca.LegalActionPriorityQCA.alias("LegalActionPriority"),
    exposure_profits.CDI.isin(hb_high_balance_cdis_list).cast("boolean").alias("IsHighBalanceFlag"),
    # coalesce(high_balance_by_exposure["IsHighBalanceFlag"], lit(False)).cast("boolean").alias("IsHighBalanceFlag"),
    lit(None).cast("string").alias("ApplicationDecision"), 
    target_table_profits.ExtrajudicialLetterEverFlag_Dk.cast("boolean").alias("ExtrajudicialLetterEverFlag"),
    target_table_profits.ExtrajudicialLetterLastDate_Dk.cast("date").alias("ExtrajudicialLetterLastDate"),
    target_table_profits.CourtRegion_Dk.cast("string").alias("CourtRegion"),
    target_table_profits.SubServicer.cast("string").alias("SubServicer"),
    lit(None).cast("decimal(19,2)").alias("IFRSGBV"),
    exposure_qca.DueDebtAmountQCA.cast("decimal(19,4)").alias("DueDebtAmount"),
    target_table_profits.TotalUnrealizedInterest.cast("decimal(19,2)").alias("TotalUnrealizedInterest"),
    target_table_profits.NetDueAmount.cast("decimal(19,2)").alias("NetDueAmount"),
    target_table_profits.TotalInterestAccruals.cast("decimal(19,2)").alias("TotalInterestAccruals"),
    target_table_profits.TotalInterestL3869.cast("decimal(19,2)").alias("TotalInterestL3869"),
    exposure_profits.JustificationCode.cast("int").alias("JustificationCode"), 
    exposure_profits.JustificationComments.cast("string").alias("JustificationComments"), 
    target_table_profits.FutureMargin.alias("FutureMargin"),
    target_table_profits.FutureMarginApplyDate.alias("FutureMarginApplyDate"),
    target_table_profits.FirstPeriodNumberOfInstallements,
    target_table_profits.InstalmentSNFrom,
    target_table_profits.InstalmentSNTo,
    target_table_profits.newInstalmentAmount,
    target_table_profits.L3869WaterfallMainBorrower,
    exposure_profits.BusinessDate.alias("BusinessDate"),
    lit(None).cast("Boolean").alias("ForcedOnboarding"),
    lit(None).cast("Boolean").alias("HolisticFlag"),
    lit(None).cast("String").alias("ForcedOnboardReason"),
    target_table_profits.FutureStepUpEventDate.cast("date").alias("FutureStepUpEventDate"),
    target_table_profits.FutureStepUpEventAmount.cast("decimal(38,7)").alias("FutureStepUpEventAmount")
).distinct()

## Transformations (QCA Core Retail)

In [0]:

# ========================== Core Retail  ===================================

exposure_qca = exposure_qca.filter(exposure_qca.PortfolioName == 'Core Retail').distinct()
exposure_qca.createOrReplaceTempView("exposure_qca")

Settlement_Exposure_qca = spark.sql('''

         select distinct Bucket as Settlement_Bucket ,ExposureKey as ExposureKey_se
         from 
         (
         	select Bucket,ExposureKey,
         	row_number() over (partition by ExposureKey order by SubmissionDate desc) as rown
         	from SettlementExposure
         	where 1=1
         	and SourceSystem='qca'
         	and SettlementType!='Νόμος 3869 – Προσωρινή Διαταγή'
         	and SettlementStatus IN ('Running','Partially Fulfilled')
         )a 
         where rown=1
                                    '''
)     

Law_qca = spark.sql ( '''
     
    select  ExposureKey as ExposureKey_l ,CalcLawStatus as LawStatus,CalcLawStatusCode as LawStatusCode,UnderLawType,UnderLawTypeCode, 
    cast(ApplicationStatusDate as date) as LawApplicationStatusDate,cast(LawProvisionalCourtDecisionDate as date) as LawProvisionalCourtDecisionDate 
    ,cast(LawCourtDate as date) as LawCourtDate
    from (
    
    select temp1.*,temp2.CDI,temp2.PartyRelationshipType
    ,rank() over (partition by temp1.ExposureKey,temp1.PortfolioName,temp1.ApplicationStatusDate order by 
    CASE    WHEN temp2.PartyRelationshipType='Primary Owner' then 1
        WHEN temp2.PartyRelationshipType='Guarantor' then 2
        WHEN temp2.PartyRelationshipType='Co-Owner' then 3
        ELSE 4 END
        ,CDI) as rnk
    from (
    
        select distinct temp1.ExposureKey
        ,temp1.PortfolioName
        ,temp1.CalcLawStatus
        ,temp1.CalcLawStatusCode
        ,temp1.UnderLawType
        ,temp1.UnderLawTypeCode
        ,min(temp1.ApplicationStatusDate) as ApplicationStatusDate
        ,case when temp1.CalcLawStatus='Pending Court Decision'   then max(temp1.CourtDate) end as LawProvisionalCourtDecisionDate
        ,case when temp1.CalcLawStatus='Positive Court Decision'  then max(temp1.CourtDate) end as LawCourtDate
        from (
        
            select distinct ex.ExposureKey
            ,ex.CDI
            ,ex.PortfolioName
            ,ex.PartyRelationshipType
            ,row_number() OVER (PARTITION BY ex.ExposureKey,ex.CDI ORDER BY 
            CASE    WHEN ex.PartyRelationshipType='Primary Owner' then 1
                    WHEN ex.PartyRelationshipType='Guarantor' then 2
                    WHEN ex.PartyRelationshipType='Co-Owner' then 3                
             ELSE 4 END ) rn
            ,law.CalcLawStatus
            ,law.CalcLawStatusCode
            ,law.UnderLawType
            ,law.UnderLawTypeCode
            ,law.ApplicationStatusDate
            ,law.CourtDate
            ,rank() over (partition by ex.ExposureKey order by coalesce(CalcLawStatus,'a') desc) as rnk
            from exposure_sec ex
            left join dunderlaw law
            on law.CDI=ex.CDI
            where 1=1
            and law.CalcLawStatus is not null
            and ex.PortfolioKey='-9999' and ex.SourceSystem ='qca'
        )temp1
        where 1=1
        and temp1.rnk=1
        and temp1.rn=1
        group by temp1.ExposureKey,temp1.PortfolioName,temp1.CalcLawStatus,temp1.UnderLawType,temp1.CalcLawStatusCode,temp1.UnderLawTypeCode
    )temp1
        inner join (
            select distinct ex.ExposureKey
            ,ex.CDI
            ,ex.PortfolioName
            ,ex.PartyRelationshipType
            ,law.ApplicationStatusDate
            ,case when law.CalcLawStatus='Pending Court Decision'   then law.CourtDate end as LawProvisionalCourtDecisionDate
            ,case when law.CalcLawStatus='Positive Court Decision'  then law.CourtDate end as LawCourtDate
            ,row_number() OVER (PARTITION BY ex.ExposureKey,ex.CDI ORDER BY 
            CASE    WHEN ex.PartyRelationshipType='Primary Owner' then 1
                    WHEN ex.PartyRelationshipType='Guarantor' then 2
                    WHEN ex.PartyRelationshipType='Co-Owner' then 3
                    ELSE 4 END ) rn
            from exposure_sec ex
            left join dunderlaw law
            on law.CDI=ex.CDI
            where 1=1
            and law.CalcLawStatus is not null
            and ex.PortfolioKey='-9999' and ex.SourceSystem ='qca'
        )temp2
        on temp2.ExposureKey=temp1.ExposureKey
        and temp2.ApplicationStatusDate=temp1.ApplicationStatusDate
        and temp2.PortfolioName=temp1.PortfolioName
        and temp2.rn=1
    )a
    where a.rnk=1  '''
   
)


exposure_ever_min_status_write_off_qca = spark.sql( '''
        SELECT ExposureKey as ExposureKey_mwoff
                ,min(BusinessDate) AS MinDate
            FROM exposure_ever ex
            where ex.ExposureStatusCode='074-WRITE OFF'
            and  
            PortfolioKey='-9999' and SourceSystem ='qca' 
            GROUP BY ExposureKey 
          '''
)


Payment_Schedule_Loan_Next_Install_qca = spark.sql ( 
                "    select ExposureKey"
                "    ,min(CASE WHEN InstallmentAmount >0 THEN InstallmentDate END) AS NextInstallmentLoanDate" 
                "    ,SUM(InstallmentAmount) AS NextInstallmentAmount" 
                "    from ("
                "        select InstallmentAmount, InstallmentDate, Exposurekey, Businessdate," 
                "        ROW_NUMBER() OVER (PARTITION BY ExposureKey ORDER BY InstallmentDate asc) as InstallmentRank" 
                "    from PaymentSchedule" 
                "    where InstallmentDate>BusinessDate"
                "    and InstallmentStatus='Active'"
                "    and InterestAmount>0 and SourceSystem='qca'"
                "    ) a" 
                "    where InstallmentRank=1" 
                "    group by ExposureKey, BusinessDate"
                )


Payment_Schedule_Loan_Last_Install_qca = spark.sql ( 
                "    select ExposureKey as ExposureKeyLoanLastInstall"
                "    ,min(CASE WHEN InstallmentAmount >0 THEN InstallmentDate END) AS LastInstallmentLoanDate" 
                "    ,SUM(InstallmentAmount) AS LastInstallmentAmount" 
                "    from ("
                "        select InstallmentAmount, InstallmentDate, Exposurekey, Businessdate," 
                "        ROW_NUMBER() OVER (PARTITION BY ExposureKey ORDER BY InstallmentDate asc) as InstallmentRank" 
                "    from PaymentSchedule" 
                "    where 1=1" 
                "    and InstallmentDate<BusinessDate"
                "    and InstallmentStatus='Active'"
                "    and InterestAmount>0 and SourceSystem='qca'"
                "    ) a" 
                "    where InstallmentRank=1" 
                "    group by ExposureKey, BusinessDate"
                )


Payment_Schedule_Loan_Next_Install_qca.createOrReplaceTempView("Payment_Schedule_Loan_Next_Install_qca")
Payment_Schedule_Loan_Last_Install_qca.createOrReplaceTempView("Payment_Schedule_Loan_Last_Install_qca")
L3869_qca=spark.sql('''
                        Select  ex.ExposureKeyQCA as ExposureKeyL3869,
                        max(case when du.InLawFlag=1 and du.UnderLawType='N3869' then 1 else  0 end) L3869Flag_qca,
                        max(case when du.InLawFlag=1 and du.UnderLawType='N3869' then du.CalcLawStatus else  Null end)          L3869Status_qca,
                        max(case when du.InLawFlag=1 and du.UnderLawType='N3869' then du.SubmissionDate else  Null end)             L3869SubmissionDate_qca,
                        max(case when du.InLawFlag=1 and du.UnderLawType='N3869' and du.CalcLawStatus = 'Pending Court Decision'  then du.CourtDate else  Null end) L3869ProvisionalCourtDecisionDate_qca,
                        max(case when du.InLawFlag=1 and du.UnderLawType='N3869' and du.CalcLawStatus = 'Positive Court Decision'  then du.CourtDate else  Null end) L3869CourtDate_qca
                        from             exposure_qca ex
                        inner join dunderlaw du
                        on du.CDI = ex.CDI
                        group by Ex.ExposureKeyQCA
                        ''')



In [0]:
# Second DataFrame: Where FirstDateReceiptValuation contains data
ext_report_unified_with_data = spark.sql(f"""
    SELECT
        ApplicationNumber,
        ApplicationAutoCredit,
        FirstDateSendDecisionToProcessor,
        BusinessDate,
        CASE
            WHEN UnifiedApplicationStatus = 'Ακυρωμένη' THEN '2.ΑΚΥΡΩΣΗ'
            WHEN AssetSegment IN ('CL', 'ML') AND Decision = 'Ναι' THEN '4.ΈΓΚΡΙΣΗ'
            WHEN AssetSegment IN ('CL', 'ML') AND Decision = 'Όχι' THEN '1.ΑΠΟΡΡΙΨΗ'
            WHEN AssetSegment = 'SB' AND Decision = 'Έγκριση' THEN '4.ΈΓΚΡΙΣΗ'
            WHEN AssetSegment = 'SB' AND Decision = 'Απόρριψη' THEN '1.ΑΠΟΡΡΙΨΗ'
            ELSE '3.ΕΚΚΡΕΜΗ'
        END AS Apofasi,
        ExposureKey,
        FirstDateReceiptValuation
    FROM 
        `{env}-cdh`.primary.extreportunified
    WHERE 
        AssetSegment IN ('CL', 'ML', 'SB')
        AND (FirstDateReceiptValuation IS NOT NULL AND FirstDateReceiptValuation != '')
        AND BusinessDate = {run_date!r}
""")

ext_report_unified_with_data.createOrReplaceTempView("ext_report_unified_with_data")

apofasi_df_first_date_exists = spark.sql(f"""
    SELECT DISTINCT
        a.ExposureKey as ExposureKeyApofasis,
        a.BusinessDate,
        v.MaxFirstDateReceiptValuation AS FirstDateReceiptValuation,
        CASE
            WHEN a.ApplicationNumber = b.ApplicationNumber 
                 AND b.ApplicationNumber IS NOT NULL THEN '2.ΑΚΥΡΩΣΗ'
            WHEN Apofasi = '4.ΈΓΚΡΙΣΗ'
                 AND (a.ApplicationAutoCredit = 'Credit Review' OR a.ApplicationAutoCredit = 'Credit Agreement')
                 AND (a.FirstDateSendDecisionToProcessor IS NULL OR a.FirstDateSendDecisionToProcessor = '')
                 AND LEFT(a.ApplicationNumber, 6) > 190902 THEN '3.ΕΚΚΡΕΜΗ'
            ELSE Apofasi
        END AS Apofasi
    FROM 
        ext_report_unified_with_data a
    INNER JOIN (
        SELECT
          ExposureKey,
          BusinessDate,
          MAX(FirstDateReceiptValuation) 
            AS MaxFirstDateReceiptValuation,
          MAX(
            CASE 
              WHEN FirstDateReceiptValuation 
                     > CAST(CAST(FirstDateReceiptValuation AS date) AS timestamp)
              THEN 1 ELSE 0
            END
          ) AS HasTime,
          MAX(CASE WHEN Apofasi = '4.ΈΓΚΡΙΣΗ' THEN 1 ELSE 0 END)
            AS HasApproval
        FROM ext_report_unified_with_data
        GROUP BY ExposureKey, BusinessDate
    ) v
      ON a.ExposureKey = v.ExposureKey
     AND a.BusinessDate = v.BusinessDate
     AND a.FirstDateReceiptValuation = v.MaxFirstDateReceiptValuation
     AND (
          v.HasTime    = 1  
        OR (v.HasTime    = 0 AND v.HasApproval = 1 AND a.Apofasi = '4.ΈΓΚΡΙΣΗ')
        OR (v.HasTime    = 0 AND v.HasApproval = 0)
     )

    LEFT JOIN `{env}-cdh`.`reference`.`core_paymentjumpmasscancelation` b 
      ON b.ApplicationNumber = a.ApplicationNumber 
     AND a.ExposureKey      = b.ExposureKey
""")

# First DataFrame: Where FirstDateReceiptValuation is NULL or an empty string
ext_report_unified_df_null_or_empty = spark.sql(f"""
    SELECT
        ApplicationNumber,
        ApplicationAutoCredit,
        FirstDateSendDecisionToProcessor,
        UnifiedApplicationStatus,
        BusinessDate,
        CASE
            WHEN UnifiedApplicationStatus = 'Ακυρωμένη' THEN '2.ΑΚΥΡΩΣΗ'
            WHEN AssetSegment IN ('CL', 'ML') AND Decision = 'Ναι' THEN '4.ΈΓΚΡΙΣΗ'
            WHEN AssetSegment IN ('CL', 'ML') AND Decision = 'Όχι' THEN '1.ΑΠΟΡΡΙΨΗ'
            WHEN AssetSegment = 'SB' AND Decision = 'Έγκριση' THEN '4.ΈΓΚΡΙΣΗ'
            WHEN AssetSegment = 'SB' AND Decision = 'Απόρριψη' THEN '1.ΑΠΟΡΡΙΨΗ'
            ELSE '3.ΕΚΚΡΕΜΗ'
        END AS Apofasi,
        ExposureKey,
        FirstDateReceiptValuation
    FROM 
        `{env}-cdh`.primary.extreportunified
    WHERE 
        AssetSegment IN ('CL', 'ML', 'SB')
        AND (FirstDateReceiptValuation IS NULL OR FirstDateReceiptValuation = '')
        AND BusinessDate = {run_date!r}
""")

ext_report_unified_df_null_or_empty.createOrReplaceTempView("ext_report_unified_df_null_or_empty")

apofasi_df_first_date_not_exists = spark.sql(f"""
    SELECT
        a.ExposureKey as ExposureKeyApofasis,
        a.BusinessDate,
        FirstDateReceiptValuation,
        CASE
            WHEN a.ApplicationNumber = b.ApplicationNumber 
                 AND b.ApplicationNumber IS NOT NULL THEN '2.ΑΚΥΡΩΣΗ'
            WHEN Apofasi = '4.ΈΓΚΡΙΣΗ'
                 AND (a.ApplicationAutoCredit = 'Credit Review' OR a.ApplicationAutoCredit = 'Credit Agreement')
                 AND (a.FirstDateSendDecisionToProcessor IS NULL OR a.FirstDateSendDecisionToProcessor = '')
                 AND LEFT(a.ApplicationNumber, 6) > 190902 THEN '3.ΕΚΚΡΕΜΗ'
            ELSE Apofasi
        END AS Apofasi
    FROM 
        ext_report_unified_df_null_or_empty a
    LEFT JOIN 
        `{env}-cdh`.`reference`.`core_paymentjumpmasscancelation` b 
        ON b.ApplicationNumber = a.ApplicationNumber 
        AND a.ExposureKey = b.ExposureKey
    WHERE a.UnifiedApplicationStatus = 'Ενεργή'
""")

# Left Anti Join: Keep NULL rows that don't have a matching non-NULL row
apofasi_df_first_date_not_exists = apofasi_df_first_date_not_exists.join(apofasi_df_first_date_exists, on="ExposureKeyApofasis", how="leftanti")

# Union the final NULL rows with the non-NULL rows
apofasi_df = apofasi_df_first_date_exists.union(apofasi_df_first_date_not_exists)

## Join conditions (QCA Core Retail)

In [0]:
exposure_qca_exposure_non_active_qca_join_condition = (
    exposure_qca["ExposureKeyQCA"] == exposure_non_active_qca["ExposureKeyNonActive"]
)

exposure_qca_Settlement_Exposure_qca_join_condition = (
    exposure_qca["ExposureKeyQCA"] == Settlement_Exposure_qca["ExposureKey_se"]
)

exposure_qca_PaymentSchedule_min_qca_join_condition = (
    exposure_qca["ExposureKeyQCA"] == Payment_Schedule_Loan_Next_Install_qca["ExposureKey"]
)

exposure_qca_PaymentSchedule_max_qca_join_condition = (
    exposure_qca["ExposureKeyQCA"] == Payment_Schedule_Loan_Last_Install_qca["ExposureKeyLoanLastInstall"]
)

exposure_qca_LegalAction_po_join_condition = (
    (exposure_qca["ExposureKeyQCA"] == LegalAction_po["ExposureKey_lapo"])
)

exposure_qca_exposure_ever_min_status_write_off_qca_condition = (
    exposure_qca["ExposureKeyQCA"] == exposure_ever_min_status_write_off_qca["ExposureKey_mwoff"]
)

exposure_qca_Law_qca_join_condition = (
    exposure_qca["ExposureKeyQCA"] == Law_qca["ExposureKey_l"]
)

exposure_qca_L3869_qca_join_condition = (
    exposure_qca["ExposureKeyQCA"]  == L3869_qca["ExposureKeyL3869"]
) 

exposure_qca_apofasi_qca_join_condition = (
    exposure_qca["ExposureKeyQCA"]  == apofasi_df["ExposureKeyApofasis"]
) 

exposure_qca_SubServicer_join_condition = (
    (exposure_qca['CDI'] == party_sub_servicer['CDI']) &
    (party_sub_servicer['SubServicer'] == 'EUCLID DV01 - ASSIGN FOR SERVICING')
)

exposure_qca_extunionfile_join_condition =(
    exposure_qca["ExposureKeyQCA"] == extunionfile["ExposureNumber"]
)

exposure_qca_exposure_product_qca_join_condition = (
    exposure_qca["ExposureKeyQCA"] == exposure_product_qca["exposurekeyqcqca"]
)

# Joins
target_table_qca = (
    exposure_qca.join(
        exposure_non_active_qca,
        exposure_qca_exposure_non_active_qca_join_condition,
        "left",
    ).join(
        Settlement_Exposure_qca,
        exposure_qca_Settlement_Exposure_qca_join_condition,
        "left",
    ).join(
        Payment_Schedule_Loan_Next_Install_qca,
        exposure_qca_PaymentSchedule_min_qca_join_condition,
        "left",
    ).join(
        LegalAction_po,
        exposure_qca_LegalAction_po_join_condition,
        "left",
    ).join(
        exposure_ever_min_status_write_off_qca,
        exposure_qca_exposure_ever_min_status_write_off_qca_condition,
        "left",
    ).join(
        Law_qca,
        exposure_qca_Law_qca_join_condition,
        "left",
    ).join(
        L3869_qca,
        exposure_qca_L3869_qca_join_condition,
        "left"
    ).join(
        initializationexposureinfo,
        initializationexposureinfo.ExposureKey == exposure_qca.ExposureKeyQCA,
        "left",
    ).join(
        Payment_Schedule_Loan_Last_Install_qca,
        exposure_qca_PaymentSchedule_max_qca_join_condition,
        "left",
    ).join(
        apofasi_df,
        exposure_qca_apofasi_qca_join_condition,
        "left",
    ).join(
        party_sub_servicer,
        exposure_qca_SubServicer_join_condition,
        "left",
    ).join(
        extunionfile,
        exposure_qca_extunionfile_join_condition,
        "left",
    ).join(
        exposure_product_qca,
        exposure_qca_exposure_product_qca_join_condition,
        "left",
    )
)

## Calculations (QCA Core Retail)

In [0]:

target_table_qca_ExposureId_primary_key_fields = [
        exposure_qca.ExposureKeyQCA
]
 
target_table_qca = (
            target_table_qca.withColumn(
                "ExposureIdNew",
                sha2(
                    concat_ws(";", *target_table_qca_ExposureId_primary_key_fields),
                    256,
                ),
            ).withColumn(
                "ExpectedOverduedateCalc",
                when(exposure_qca.BusinessDate < exposure_qca.ExpectedOverduedate, exposure_qca.ExpectedOverduedate)
                .otherwise(lit(None))
            )
            .withColumn(
                "IsSubsidyFlagCalc",
                 when(exposure_qca.IsSubsidyFlagQCA == 'Y' , True)
                .when(exposure_qca.IsSubsidyFlagQCA == 'N' , False)
                .otherwise(lit(None))
            ).withColumn(
                "HasActiveSettlementFlagCalc",
                 when(Settlement_Exposure_qca.Settlement_Bucket.isNotNull() , True)
                .otherwise(False)
            ).withColumn(
                "NPLDefaultAmountCalc",
                 when(exposure_qca.ExposureStatus.isin ('Writen Off', 'Overdue', 'Define Delay') , exposure_qca.OutstandingTotalBalance)
                .otherwise(lit(None))
            ).withColumn(
                "InterestNextDueDateCalc",
                 when((exposure_qca.ExposureStatus.isin ('Writen Off', 'Define Delay')) | (exposure_qca.Maturitydate <= exposure_qca.BusinessDate) , lit(None))
                .otherwise(coalesce(Payment_Schedule_Loan_Next_Install_qca.NextInstallmentLoanDate.cast("date"),exposure_qca.Maturitydate))
            ).withColumn(
                "BucketCalc",
                 when(exposure_qca.IsActiveFlag != True,lit(0))
                .when(Settlement_Exposure_qca.Settlement_Bucket.isNotNull() , Settlement_Exposure_qca.Settlement_Bucket)
                .when(exposure_qca.ExposureStatus == 'Defined Delay', lit(98))
                .when(exposure_qca.ExposureStatus == 'Writen Off', lit(99))
                .when(exposure_qca.ExposureStatus.isin ('Overdue', 'Normal'), exposure_qca.Bucket)
            ).withColumn(
                "EverDenouncedDateCalc",
                 when(exposure_qca.EverDenouncedDateQCA.cast("date") != '1753-01-01' , exposure_qca.EverDenouncedDateQCA.cast("date") 
                ).otherwise(lit(None))

            ).withColumn( 
                "PaymentOrderFlagCalc", 
                when(LegalAction_po.ExposureKey_lapo.isNotNull() , True
                ).otherwise(False)
            ).withColumn( 
                "PaymentOrderDateCalc", 
                LegalAction_po.LegalActionDate
            ).withColumn(
                "NonCondWriteOffDateCalc",
                 when(exposure_qca.ExposureStatusCode =='074-WRITE OFF', exposure_ever_min_status_write_off_qca.MinDate)
                .when(exposure_qca.WriteOffdate.isNotNull() , exposure_qca.WriteOffdate)
            ).withColumn(
                "NonCondWriteOffAmountCalc",
                 when(exposure_qca.ExposureStatusCode =='074-WRITE OFF', exposure_qca.OutstandingTotalBalance.cast("decimal(19,2)") + coalesce(exposure_qca.WriteOffAmount,lit(0)))
                .when(exposure_qca.WriteOffAmount.isNotNull() , exposure_qca.WriteOffAmount)
            ).withColumn(
                "CurrencyLatinOnly",
                trim(regexp_replace(exposure_qca.Currency, 'Ε', 'E'))  # Replace Greek Epsilon with Latin E and trim spaces
            ).withColumn(
                "InitialAssignmentAmount",
                 coalesce(target_table_qca.InitialAssignmentBalance,exposure_qca.OriginationAmount)
            ).withColumn(
                "GracePeriodEndDateCalc",
                when(
                (exposure_qca.GracePeriodEnddate < "1950-01-01") | 
                (exposure_qca.GracePeriodEnddate > "2100-01-01"), 
                lit(None)
                ).otherwise(exposure_qca.GracePeriodEnddate)
            ).withColumn(
                "TotalUnrealizedInterest",
                coalesce(exposure_qca.Unrealizedinterest,lit(0)) + coalesce(exposure_qca.DueUnrealizedinterest,lit(0))
            ).withColumn(
                "NetDueAmount",
                coalesce(exposure_qca.DuePrincipal,lit(0)) + coalesce(exposure_qca.Dueinterest,lit(0))
                + coalesce(exposure_qca.DueUnrealizedinterest,lit(0)) + coalesce(exposure_qca.DueExpense,lit(0))
            ).withColumn(
                "TotalInterestAccruals",
                coalesce(exposure_qca.interest, lit(0)) + coalesce(exposure_qca.Dueinterest, lit(0)) + 
                coalesce(exposure_qca.DueUnrealizedinterest, lit(0)) + coalesce(exposure_qca.Penaltyinterest, lit(0)) + 
                coalesce(exposure_qca.PenaltyUnrealizedinterest, lit(0)) + coalesce(exposure_qca.Unrealizedinterest, lit(0)) + 
                coalesce(exposure_qca.Accrued, lit(0)) + coalesce(exposure_qca.DueAccrued, lit(0))
            )
)


target_table_qca = (
    target_table_qca.withColumn(
        "SettlementDaysPastDue",
        when(
            target_table_qca.HasActiveSettlementFlagCalc == True, 
            when(
                coalesce(datediff(exposure_qca.Overduedate, exposure_qca.BusinessDate), lit(0)) < 0,
                -coalesce(datediff(exposure_qca.Overduedate, exposure_qca.BusinessDate), lit(0))
            ).otherwise(
                coalesce(datediff(exposure_qca.Overduedate, exposure_qca.BusinessDate), lit(0))
            )
        ).otherwise(lit(None))
    )
)

## Final Select (QCA Core Retail)

In [0]:

result_df_qca = target_table_qca.select(
    target_table_qca.ExposureIdNew.alias("ExposureDerivativeID"),
    exposure_qca.ExposureKeyQCA.alias("ExposureKey"),
    lit('qca').alias("SourceSystem"),
    exposure_qca.PortfolioKey.alias("PortfolioKey"),
    exposure_qca.PortfolioName.alias("PortfolioName"),
    lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias("IngestionTimestamp"),
    lit(None).alias("BatchId"),
    lit(None).alias("ExposureKeyProfits"),
    lit(None).alias("ExposureSourceSystemKeyProfits"),
    lit(None).alias("ExposureKeyQCCepal"),
    lit(None).alias("ExposureSourceSystemKeyQCCepal"),
    exposure_qca.ExposureKeyQCA.alias("ExposureKeyQCAlpha"),
    exposure_qca.ExposureSourceSystemKeyQCA.alias("ExposureSourceSystemKeyQCAlpha"),
    lit(None).alias("ExposureKeyArotron"),
    lit(None).alias("ExposureSourceSystemKeyArotron"),
    exposure_qca.ExposureDebtorKey.alias("ExposureDebtorKey"),
    exposure_qca.CaseCodeQCA.alias("CaseCode"),
    exposure_qca.AgreementKey.alias("AgreementKey"),
    exposure_qca.PartyKey.alias("PartyKey"),
    exposure_qca.PartyName.alias("PartyName"),
    exposure_qca.CDI.alias("CDI"),
    exposure_qca.PartyTaxRegistrationNumber.alias("PartyTaxRegistrationNumber"),
    exposure_qca.PartyRelationshipTypeCode.alias("PartyRelationshipTypeCode"),
    exposure_qca.PartyRelationshipType.alias("PartyRelationshipType"),
    exposure_qca.PrimaryPartyFlag.alias("PrimaryPartyFlag"),
    exposure_qca.AllBorrowersDeceasedFlagQCA.alias("AllBorrowersDeceasedFlag"),
    exposure_qca.ProductKey.alias("ProductKey"),
    exposure_qca.ProductCode.alias("ProductCode"),
    target_table_qca.SubProductDescription.cast("String").alias("SubProductDescription"),
    exposure_qca.ExposureTypeCode.alias("ExposureTypeCode"),
    exposure_qca.ExposureType.alias("ExposureType"),
    exposure_qca.ExposureStatusCode.alias("ExposureStatusCode"),
    exposure_qca.ExposureStatus.alias("ExposureStatus"),
    exposure_qca.ExposureStatusdate.alias("ExposureStatusDate"),
    exposure_qca.Opendate.alias("Opendate"),
    exposure_qca.ActiveExpiryDate.alias("ActiveExpiryDate"),
    exposure_qca.Maturitydate.alias("Maturitydate"),
    exposure_qca.OriginBank.alias("OriginBank"),
    exposure_qca.OriginationAmount.alias("OriginationAmount"),
    target_table_qca.InitialAssignmentAmount.cast("decimal(19,2)").alias("InitialAssignmentAmount"),
    target_table_qca.CurrencyLatinOnly.alias("Currency"),
    exposure_qca.BankingUnitCode.alias("BankingUnitCode"),
    exposure_qca.BankingUnit.alias("BankingUnit"),
    exposure_qca.MechanismCode.alias("MechanismCode"),
    exposure_qca.Mechanism.alias("Mechanism"),
    exposure_qca.BranchCodeQCA.alias("BranchCode"),
    exposure_qca.BranchQCA.alias("Branch"),
    exposure_qca.PurposeReasonCode.alias("PurposeReasonCode"),
    exposure_qca.PurposeReason.alias("PurposeReason"),
    exposure_qca.PurposeKTCode.alias("PurposeKTCode"),
    exposure_qca.PurposeKT.alias("PurposeKT"),
    lit(None).alias("MaterialityStartDate"),
    exposure_qca.DaysPastDueWithMateriality.cast("int").alias("DaysPastDueWithMateriality"),
    exposure_qca.DaysPastDue.alias("DaysPastDue"),
    exposure_qca.DaysPastDueinterest.alias("DaysPastDueinterest"),
    exposure_qca.FirstArreardate.alias("FirstArreardate"),
    exposure_qca.Overduedate.alias("Overduedate"),
    exposure_non_active_qca.ClosedDate.alias("ClosedDate"),
    exposure_qca.IsActiveFlag.alias("IsActiveFlag"),
    exposure_qca.IsActiveFlag.cast("smallint").alias("IsActiveFlagInt"),
    lit(None).alias("FirstInstallmentDate"),
    exposure_qca.BucketCode.alias("BucketCode"),
    exposure_qca.Bucket.alias("Bucket"),
    exposure_qca.WriteOffdate.alias("WriteOffDate"),
    exposure_qca.WriteOffAmount.cast("decimal (19,2)").alias("WriteoffAmount"),
    exposure_qca.LPSCodeQCA.alias("LPSCode"),
    exposure_qca.LPSQCA.alias("LPS"),
    exposure_qca.LPSEngQCA.alias("LPSEng"),
    exposure_qca.LPSDateQCA.alias("LPSDate"),
    exposure_qca.DenouncedDateQCA.alias("DenouncedDate"),
    exposure_qca.DaysPastDuePrincipal.alias("DaysPastDuePrincipal"),
    target_table_qca.ExpectedOverduedateCalc.alias("ExpectedOverduedate"),
    exposure_qca.OutstandingTotalBalance.cast("decimal(19,2)").alias("OutstandingTotalBalance"),
    exposure_qca.OffBalance.cast("decimal(19,2)").alias("OffBalance"),
    exposure_qca.OnBalance.cast("decimal(19,2)").alias("OnBalance"),
    exposure_qca.DueAmount.cast("decimal(26,4)").alias("DueAmount"),
    lit(None).cast("decimal(19,2)").alias("BogBalance"),
    lit(None).cast("double").alias("GBV"),
    lit(None).cast("Boolean").alias("GBVTriggerEventFlag"),
    lit(None).cast("Date").alias("GBVTriggerEventDate"),
    lit(None).cast("String").alias("GBVTriggerEventType"),
    lit(None).cast("double").alias("TailExposure"),
    lit(None).cast("double").alias("GBVCappedAmount"),
    lit(None).cast("Date").alias("GBVCappedDate"),
    lit(None).cast("double").alias("GBVFunctionResult"),
    target_table_qca.EverDenouncedDateCalc.cast("date").alias("EverDenouncedDate"),
    exposure_qca.EverDenouncedFlagQCA.cast("boolean").alias("EverDenouncedFlag"),
    lit(None).alias("IsSecuredFlag"),
    target_table_qca.InterestNextDueDateCalc.alias("InterestNextDueDate"),
    lit(None).alias("ClosedReasonCode"),
    lit(None).alias("ClosedReason"),
    lit(None).alias("PEERestructureTypeCode"),
    lit(None).alias("PEEAdjustmentCode"),
    exposure_qca.Defaultdate.alias("Defaultdate"),
    exposure_qca.Defaultdate.cast("date").alias("LatestDefaultDate"),
    target_table_qca.NPLDefaultAmountCalc.cast("decimal(19,2)").alias("NPLDefaultAmount"),
    target_table_qca.NonCondWriteOffDateCalc.alias("NonConditionalWriteOffDate"),
    target_table_qca.NonCondWriteOffAmountCalc.cast("decimal(19,2)").alias("NonConditionalWriteOffAmount"),
    lit(None).alias("ConditionalWriteOffDate"),
    lit(None).alias("ConditionalWriteOffAmount"),
    exposure_qca.NextInstallmentdate.cast("date").alias("NextInstallmentdate"),
    lit(None).alias("NextInstallmentInterestDate"),
    lit(None).alias("NextInstallmentCapitalDate"),
    lit(None).alias("NextInstallmentMinimumDate"),
    lit(None).alias("NextInstallmentAmount"),
    lit(None).alias("NextInstallmentCapitalAmount"),
    exposure_qca.TenorQCA.cast("int").alias("ExposureTenor"),
    lit(None).alias("IsSoldFlag"),
    target_table_qca.IsSubsidyFlagCalc.alias("IsSubsidyFlag"),
    exposure_qca.IsGuaranteeFlag.alias("IsGuaranteeFlag"),
    exposure_qca.MasterExposureKey.alias("MasterExposureKey"),
    exposure_qca.SlaveFlag.alias("SlaveFlag"),
    target_table_qca.PaymentOrderFlagCalc.cast("boolean").alias("PaymentOrderFlag"),
    target_table_qca.PaymentOrderDateCalc.cast("date").alias("PaymentOrderDate"),
    exposure_qca.NumberOfUnpaidBills.cast("string").alias("NumberOfUnpaidBills"),
    Settlement_Exposure_qca.Settlement_Bucket.alias("SettlementBucketCode"),
    Settlement_Exposure_qca.Settlement_Bucket.alias("SettlementBucket"),
    lit(None).alias("DaysPastDueFinal"),
    target_table_qca.HasActiveSettlementFlagCalc.alias("HasActiveSettlementFlag"),
    Law_qca.UnderLawTypeCode.alias("UnderLawTypeCode"),
    Law_qca.UnderLawType.alias("UnderLawType"),
    Law_qca.LawStatusCode.alias("LawStatusCode"),
    Law_qca.LawStatus.alias("LawStatus"),
    Law_qca.LawApplicationStatusDate.alias("LawApplicationStatusDate"),
    Law_qca.LawCourtDate.alias("LawCourtDate"),
    Law_qca.LawProvisionalCourtDecisionDate.alias("LawProvisionalCourtDecisionDate"),
    exposure_qca.InstalmentFrequencyQCA.alias("InstalmentFrequency"),
    exposure_qca.InstallmentFixedAmount.alias("InstallmentFixedAmount"),
    exposure_qca.NumberOfInstallments.alias("NumberOfInstallments"),
    exposure_qca.PreviousInstallmentdate.alias("PreviousInstallmentdate"),
    exposure_qca.LastBillingdate.alias("LastBillingdate"),
    exposure_qca.LastRestructuringTypeCode.alias("LastRestructuringTypeCode"),
    exposure_qca.LastRestructuringType.alias("LastRestructuringType"),
    exposure_qca.LastRestructuringDate.alias("LastRestructuringDate"),
    exposure_qca.RestructuringPEEDate.alias("RestructuringPEEDate"),
    exposure_qca.RestructuringPEETypeCodeQCA.alias("RestructuringPEETypeCode"),
    exposure_qca.RestructuringPEEType.alias("RestructuringPEEType"),
    exposure_qca.ObjectiveStatusCodeQCA.alias("ObjectiveStatusCode"),
    exposure_qca.ObjectiveStatusQCA.alias("ObjectiveStatus"),
    exposure_qca.ObjectiveStatusdateQCA.alias("ObjectiveStatusdate"),
    exposure_qca.TreatmentBranchCodeQCA.alias("TreatmentBranchCode"),
    exposure_qca.TreatmentBranchQCA.alias("TreatmentBranch"),
    target_table_qca.GracePeriodEndDateCalc.alias("GracePeriodEnddate"),
    exposure_qca.GracePeriodTypeCode.alias("GracePeriodTypeCode"),
    exposure_qca.GracePeriodType.alias("GracePeriodType"),
    exposure_qca.GracePeriodinterestPayableFlag.alias("GracePeriodinterestPayableFlag"),
    exposure_qca.GracePeriodNextinterestdate.alias("GracePeriodNextinterestdate"),
    exposure_qca.GracePeriodinterestFrequency.alias("GracePeriodinterestFrequency"),
    exposure_qca.Principal.alias("Principal"),
    exposure_qca.interest.alias("interest"),
    exposure_qca.Unrealizedinterest.alias("Unrealizedinterest"),
    exposure_qca.Commission.alias("Commission"),
    exposure_qca.Expense.alias("Expense"),
    exposure_qca.Accrued.alias("Accrued"),
    exposure_qca.DuePrincipal.alias("DuePrincipal"),
    exposure_qca.Dueinterest.alias("Dueinterest"),
    exposure_qca.DueUnrealizedinterest.alias("DueUnrealizedinterest"),
    exposure_qca.DueCommission.alias("DueCommission"),
    exposure_qca.DueExpense.alias("DueExpense"),
    exposure_qca.DueAccrued.alias("DueAccrued"),
    exposure_qca.Penaltyinterest.alias("Penaltyinterest"),
    exposure_qca.PenaltyUnrealizedinterest.alias("PenaltyUnrealizedinterest"),
    exposure_qca.InterestRate.cast("decimal(19,6)").alias("InterestRate"),
    exposure_qca.interestCode.alias("interestCode"),
    exposure_qca.interestDescription.alias("interestDescription"),
    exposure_qca.BaseRate.alias("BaseRate"),
    exposure_qca.BaseCode.alias("BaseCode"),
    exposure_qca.BaseDescription.alias("BaseDescription"),
    exposure_qca.N128Rate.alias("N128Rate"),
    exposure_qca.SpreadRate.alias("SpreadRate"),
    exposure_qca.PenaltyRate.alias("PenaltyRate"),
    exposure_qca.InterestRateTypeCode.alias("InterestRateTypeCode"),
    exposure_qca.InterestRateType.alias("InterestRateType"),
    exposure_qca.SpreadCurrencyCode.alias("SpreadCurrencyCode"),
    exposure_qca.SpreadCurrency.alias("SpreadCurrency"),
    exposure_qca.SpreadPeriod.alias("SpreadPeriod"),
    exposure_qca.InterestDuration.alias("InterestDuration"),
    exposure_qca.DayCounts.alias("DayCounts"),
    exposure_qca.InterestFrequency.alias("InterestFrequency"),
    exposure_qca.InterestFixedDate.alias("InterestFixedDate"),
    exposure_qca.interestValiditydate.alias("interestValiditydate"),
    exposure_qca.InterestChangeStartdate.alias("InterestChangeStartdate"),
    exposure_qca.InterestChangeEnddate.alias("InterestChangeEnddate"),
    exposure_qca.InterestRateUpdatedate.alias("InterestRateUpdatedate"),
    exposure_qca.InterestCalcStopFlag.alias("InterestCalcStopFlag"),
    exposure_qca.AutoRenewalFlag.alias("AutoRenewalFlag"),
    exposure_qca.AssetSegmentCode.alias("AssetSegmentCode"),
    exposure_qca.AssetSegment.alias("AssetSegment"),
    exposure_qca.DiasNumber.alias("DiasNumber"),
    exposure_qca.ProvisionFrozenInterestAmount.alias("ProvisionFrozenInterestAmount"),
    exposure_qca.AccountNticeBUKeyQCA.alias("AccountNticeBUKey"),
    exposure_qca.SayFlagQCA.alias("SayFlag"),
    exposure_qca.InsuranceCompanyQCA.alias("InsuranceCompany"),
    exposure_qca.PhysicalNumberQCA.alias("PhysicalNumber"),
    exposure_qca.PreCourtDateQCA.alias("PreCourtDate"),
    exposure_qca.CourtDateQCA.alias("CourtDate"),
    exposure_qca.ContractSigningDateQCA.alias("ContractSigningDate"),
    exposure_qca.ExposureLimitAmount.alias("ExposureLimitAmount"),
    exposure_qca.BaloonAmount.alias("BaloonAmount"),
    exposure_qca.AdministrationStatusCodeQCA.alias("AdministrationStatusCode"),
    exposure_qca.AdministrationStatusQCA.alias("AdministrationStatus"),
    exposure_qca.TenorQCA.alias("Tenor"),
    exposure_qca.SPVCodeQCA.alias("SPVCode"),
    exposure_qca.SPVQCA.alias("SPV"),
    exposure_qca.TotalOverdueInsterest.alias("TotalOverdueInsterest"),
    exposure_qca.TotalOverdueInstallments.alias("TotalOverdueInstallments"),
    exposure_qca.OutstandingTotalBalanceMLT.cast("decimal(19,2)").alias("OutstandingTotalBalanceMLT"),
    lit(None).alias("NextInterestChangeDate"),
    lit(None).alias("PrincipalNextDueDate"),
    lit(None).alias("FutureInstallmentIncreaseGroup"),
    lit(None).alias("FutureInitialInstallmentGroup"),
    lit(None).alias("FutureStepUpSegment"),
    lit(None).alias("FutureStepUpIndicator"),
    lit(None).alias("PastInstallmentIncreaseGroup"),
    lit(None).alias("PastInitialInstallmentGroup"),
    lit(None).alias("PastStepUpSegment"),
    lit(None).alias("PastStepUpIndicator"),
    lit(None).alias("TotalPrincipal"),
    lit(None).alias("TotalInterest"),
    lit(None).alias("TotalAccruals"),
    lit(None).alias("TotalExpense"),
    lit(None).alias("Coupon"),
    lit(None).alias("ActiveModificationProgramStatus"),
    lit(None).alias("DenouncementIndicator"),
    lit(None).alias("ActiveExposureRestructureStartDate"),
    lit(None).alias("MonthsOfCleanPayHistory"),
    lit(None).alias("TotalPaymentAgreementPaid"),
    lit(None).alias("TotalPaymentAgreementDue"),
    lit(None).alias("ExposureRestructureOrigination"),
    lit(None).alias("ExposureRestructureStatus"),
    lit(None).alias("ExposureRestructureType"),
    lit(None).alias("CodeofConductStage"),
    exposure_qca.LetterStatusCodeQCA.alias("LetterStatusCode"),
    exposure_qca.LetterStatusQCA.alias("LetterStatus"),
    exposure_qca.LetterStatusDateQCA.alias("LetterStatusDate"),
    exposure_qca.CoCStatusCodeQCA.alias("CoCStatusCode"),
    exposure_qca.CoCStatusQCA.alias("CoCStatus"),
    exposure_qca.CoCAccountStatusDateQCA.alias("CoCAccountStatusDate"),
    exposure_qca.FirstLetterDateQCA.alias("FirstLetterDate"),
    exposure_qca.SecondLetterDateQCA.alias("SecondLetterDate"),
    exposure_qca.ThirdLetterDateQCA.alias("ThirdLetterDate"),
    exposure_qca.FirstLetterReminderDateQCA.alias("FirstLetterReminderDate"),
    exposure_qca.FirstLetterFlagQCA.alias("FirstLetterFlag"),
    exposure_qca.FirstLetterReminderFlagQCA.alias("FirstLetterReminderFlag"),
    exposure_qca.SecondLetterFlagQCA.alias("SecondLetterFlag"),
    exposure_qca.ThirdLetterFlagQCA.alias("ThirdLetterFlag"),
    exposure_qca.ExclusionDateQCA.alias("ExclusionDate"),
    exposure_qca.ExclusionReasonCodeQCA.alias("ExclusionReasonCode"),
    exposure_qca.ExclusionReasonQCA.alias("ExclusionReason"),
    L3869_qca.L3869Flag_qca.cast("int").alias("L3869Flag"),
    L3869_qca.L3869Status_qca.cast("string").alias("L3869Status"),
    L3869_qca.L3869SubmissionDate_qca.cast("date").alias("L3869SubmissionDate"),
    L3869_qca.L3869ProvisionalCourtDecisionDate_qca.cast("date").alias("L3869ProvisionalCourtDecisionDate"),
    L3869_qca.L3869CourtDate_qca.cast("date").alias("L3869CourtDate"),
    lit(None).alias("PrincipalAtDefault"),
    lit(None).alias("InterestAtDefault"),
    lit(None).cast("decimal(26,4)").alias("AmountDueAtDefault"),
    lit(None).alias("LastModificationActionType"),
    lit(None).alias("LastModificationDate"),
    lit(None).alias("HasBalloonFlag"),
    lit(None).alias("HasZeroInterestTodayFlag"),
    lit(None).cast("string").alias("InterestRateTypeTaxonomy"),
    lit(None).cast("string").alias("InterestRateResetInterval"),
    lit(None).cast("boolean").alias("HasSplitBalanceFlag"),
    lit(None).cast("date").alias("FixInterestEndDate"),
    lit(None).cast("string").alias("RepaymentMethod"),
    lit(None).cast("int").alias("RepaymentFrequency"),
    lit(None).cast("boolean").alias("HasStepUpInstallmentFlag"),
    lit(None).cast("string").alias("UnpaidInterestHandling"),
    lit(None).cast("boolean").alias("HasFutureSpreadFlag"),
    lit(None).cast("boolean").alias("IsBulletFlag"),
    lit(None).cast("boolean").alias("HasFutureInterestChangesFlag"),
    lit(None).cast("boolean").alias("OCWFlag"),
    lit(None).cast("string").alias("OCWApplicationStatus"),
    lit(None).cast("timestamp").alias("L3869InterestFreePeriodEndDate"),
    lit(None).cast("boolean").alias("InArrearsFlag"),
    lit(None).cast("boolean").alias("HasPaymentPlanFlag"),
    lit(None).cast("date").alias("DateLastCurrent"),
    lit(None).cast("date").alias("DateLastInArrears"),
    target_table_qca.LastInstallmentLoanDate.cast("date").alias("LastInstallmentDate"),
    target_table_qca.SettlementDaysPastDue.cast("int").alias("SettlementDaysPastDue"),
    exposure_qca.LegalActionPriorityQCA.alias("LegalActionPriority"),
    exposure_qca.CDI.isin(hb_high_balance_cdis_list).cast("boolean").alias("IsHighBalanceFlag"),
    target_table_qca.Apofasi.cast("string").alias("ApplicationDecision"),
    lit(None).cast("boolean").alias("ExtrajudicialLetterEverFlag"),
    lit(None).cast("date").alias("ExtrajudicialLetterLastDate"),
    lit(None).cast("string").alias("CourtRegion"),
    target_table_qca.SubServicer.cast("string").alias("SubServicer"),
    target_table_qca.IFRSGrossCarryingAmount.cast("decimal(19,2)").alias("IFRSGBV"),
    exposure_qca.DueDebtAmountQCA.cast("decimal(19,4)").alias("DueDebtAmount"),
    target_table_qca.TotalUnrealizedInterest.cast("decimal(19,2)").alias("TotalUnrealizedInterest"),
    target_table_qca.NetDueAmount.cast("decimal(19,2)").alias("NetDueAmount"),
    target_table_qca.TotalInterestAccruals.cast("decimal(19,2)").alias("TotalInterestAccruals"),
    lit(None).cast("decimal(19,2)").alias("TotalInterestL3869"),
    lit(None).cast("int").alias("JustificationCode"),
    lit(None).cast("string").alias("JustificationComments"),
    lit(None).cast("decimal(9,4)").alias("FutureMargin"),
    lit(None).cast("timestamp").alias("FutureMarginApplyDate"),
    lit(None).cast("string").alias("FirstPeriodNumberOfInstallements"),
    lit(None).cast("decimal(4,0)").alias("InstalmentSNFrom"),
    lit(None).cast("decimal(4,0)").alias("InstalmentSNTo"),
    lit(None).cast("decimal(15,2)").alias("newInstalmentAmount"),
    lit(None).cast("string").alias("L3869WaterfallMainBorrower"),
    exposure_qca.BusinessDate.alias("BusinessDate"),
    exposure_qca.ForcedOnboarding.cast("Boolean"),
    exposure_qca.HolisticFlag.cast("Boolean"),
    exposure_qca.ForcedOnboardReason.cast("String"),
    lit(None).cast("date").alias("FutureStepUpEventDate"),
    lit(None).cast("decimal(38,7)").alias("FutureStepUpEventAmount")
)


## Transformation & join conditions & calculations Arotron (Core Wholesale)

In [0]:
# ========================== Core Wholesale  ===================================

exposure_arotron = exposure_arotron.filter(exposure_arotron.PortfolioName == 'Core Wholesale').distinct()


exposure_ever_min_status_write_off_arotron = spark.sql( '''
                SELECT ExposureKey as ExposureKey_mwoff
                        ,min(BusinessDate) AS MinDate
                    FROM exposure_ever ex
                    where ex.ExposureStatusCode='4'
                    and  
                    PortfolioKey='-9998' and SourceSystem ='arotron' 
                    GROUP BY ExposureKey 
                  '''
        )
      
exposure_arotron_exposure_non_active_arotron_join_condition = (
    exposure_arotron["ExposureKeyArotron"] == exposure_non_active_arotron["ExposureKeyNonActive"]
)

exposure_arotron_exposure_ever_min_status_write_off_arotron_condition = (
    exposure_arotron["ExposureKeyArotron"] == exposure_ever_min_status_write_off_arotron["ExposureKey_mwoff"]
)

# MM added SubServicer in arotron
exposure_arotron_party_SubServicer = (
    (exposure_arotron['CDI'] == party_sub_servicer['CDI']) &
    (party_sub_servicer['SubServicer'] == 'EUCLID DV01 - ASSIGN FOR SERVICING')
)
 
# Joins
target_table_arotron = (
            exposure_arotron.join(
                exposure_non_active_arotron,
                exposure_arotron_exposure_non_active_arotron_join_condition,
                "left",
            ).join(
                exposure_ever_min_status_write_off_arotron,
                exposure_arotron_exposure_ever_min_status_write_off_arotron_condition,
                "left",
            ).join(
                initializationexposureinfo,
                initializationexposureinfo.ExposureKey == exposure_arotron.ExposureKeyArotron,
                "left",
            ).join(
                party_sub_servicer,
                exposure_arotron_party_SubServicer,
                "left",
            ) 
        )


target_table_arotron_ExposureId_primary_key_fields = [
        exposure_arotron.ExposureKeyArotron
]
 
target_table_arotron = (
            target_table_arotron.withColumn(
                "ExposureIdNew",
                sha2(
                    concat_ws(";", *target_table_arotron_ExposureId_primary_key_fields),
                    256,
                ),
            ).withColumn(
                "ExposureStatusCalc",
                when(
                    target_table_arotron.ExposureStatusCode == "1","Normal"
                ).when(
                    target_table_arotron.ExposureStatusCode == "2","Overdue"
                ).when(
                    target_table_arotron.ExposureStatusCode == "3","Define Delay"
                ).when(
                    target_table_arotron.ExposureStatusCode == "4","Writen Off"
                )
            ).withColumn(
                "OnBalanceCalc",
                coalesce(exposure_arotron.OutstandingTotalBalance.cast("decimal(19,2)"),lit(0)) - coalesce(exposure_arotron.OffBalance,lit(0))
            ).withColumn(
                "NPLDefaultAmountCalc",
                 when(exposure_arotron.ExposureStatusCode.isin ('2', '3', '4') , exposure_arotron.OutstandingTotalBalance
                ).otherwise(lit(None))
            ).withColumn(
                "MasterAccountCalc",
                 when(exposure_arotron.MasterExposureKey =='00000000' , lit(None)
                ).otherwise(exposure_arotron.MasterExposureKey )
            ).withColumn(
                "NonCondWriteOffDateCalc",
                 when(exposure_arotron.ExposureStatusCode =='4' , exposure_ever_min_status_write_off_arotron.MinDate)
                .when(exposure_arotron.WriteOffdate.isNotNull() ,exposure_arotron.WriteOffdate 
                ).otherwise(lit(None) )
            ).withColumn(
                "NonCondWriteOffAmountCalc",
                 when(exposure_arotron.ExposureStatusCode =='4' , exposure_arotron.OutstandingTotalBalance.cast("decimal(19,2)")+coalesce(exposure_arotron.WriteOffAmount,lit(0)))
                .when(exposure_arotron.WriteOffAmount.isNotNull() ,exposure_arotron.WriteOffAmount 
                ).otherwise(lit(None) )
            ).withColumn(
                "InitialAssignmentAmount",
                 coalesce(target_table_arotron.InitialAssignmentBalance,exposure_arotron.OriginationAmount)
            ).withColumn(
                "GracePeriodEndDateCalc",
                when(
                (exposure_arotron.GracePeriodEnddate < "1950-01-01") | 
                (exposure_arotron.GracePeriodEnddate > "2100-01-01"), 
                lit(None)
                ).otherwise(exposure_arotron.GracePeriodEnddate)
            ).withColumn(
                "TotalUnrealizedInterest",
                coalesce(exposure_arotron.Unrealizedinterest,lit(0)) + coalesce(exposure_arotron.DueUnrealizedinterest,lit(0))
            ).withColumn(
                "NetDueAmount",
                coalesce(exposure_arotron.DuePrincipal,lit(0)) + coalesce(exposure_arotron.Dueinterest,lit(0))
                + coalesce(exposure_arotron.DueUnrealizedinterest,lit(0)) + coalesce(exposure_arotron.DueExpense,lit(0))
            ).withColumn(
                "TotalInterestAccruals",
                coalesce(exposure_arotron.interest, lit(0)) + coalesce(exposure_arotron.Dueinterest, lit(0)) + 
                coalesce(exposure_arotron.DueUnrealizedinterest, lit(0)) + coalesce(exposure_arotron.Penaltyinterest, lit(0)) + 
                coalesce(exposure_arotron.PenaltyUnrealizedinterest, lit(0)) + coalesce(exposure_arotron.Unrealizedinterest, lit(0)) + 
                coalesce(exposure_arotron.Accrued, lit(0)) + coalesce(exposure_arotron.DueAccrued, lit(0))
            )
)
 
 


## Final select Arotron (Core Wholesale)

In [0]:
result_df_arotron = target_table_arotron.select(
    target_table_arotron.ExposureIdNew.alias("ExposureDerivativeID"),
    exposure_arotron.ExposureKeyArotron.alias("ExposureKey"),
    lit('arotron').alias("SourceSystem"),
    exposure_arotron.PortfolioKey.alias("PortfolioKey"),
    exposure_arotron.PortfolioName.alias("PortfolioName"),
    lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
        "IngestionTimestamp"
    ),
    lit(None).alias("BatchId"),
    lit(None).alias("ExposureKeyProfits"),
    lit(None).alias("ExposureSourceSystemKeyProfits"),
    lit(None).alias("ExposureKeyQCCepal"),
    lit(None).alias("ExposureSourceSystemKeyQCCepal"),
    lit(None).alias("ExposureKeyQCAlpha"),
    lit(None).alias("ExposureSourceSystemKeyQCAlpha"),
    exposure_arotron.ExposureKeyArotron.alias("ExposureKeyArotron"),
    exposure_arotron.ExposureSourceSystemKeyArotron.alias("ExposureSourceSystemKeyArotron"),
    exposure_arotron.ExposureDebtorKey.alias("ExposureDebtorKey"), 
    exposure_arotron.CaseCode.alias("CaseCode"), 
    exposure_arotron.AgreementKey.alias("AgreementKey"), 
    exposure_arotron.PartyKey.alias("PartyKey"), 
    exposure_arotron.PartyName.alias("PartyName"),
    exposure_arotron.CDI.alias("CDI"), 
    exposure_arotron.PartyTaxRegistrationNumber.alias("PartyTaxRegistrationNumber"), 
    exposure_arotron.PartyRelationshipTypeCode.alias("PartyRelationshipTypeCode"), 
    exposure_arotron.PartyRelationshipType.alias("PartyRelationshipType"), 
    exposure_arotron.PrimaryPartyFlag.alias("PrimaryPartyFlag"), 
    exposure_arotron.AllBorrowersDeceasedFlag.alias("AllBorrowersDeceasedFlag"), 
    exposure_arotron.ProductKey.alias("ProductKey"), 
    exposure_arotron.ProductCode.alias("ProductCode"), 
    lit(None).cast("String").alias("SubProductDescription"),
    exposure_arotron.ExposureTypeCode.alias("ExposureTypeCode"), 
    exposure_arotron.ExposureType.alias("ExposureType"), 
    exposure_arotron.ExposureStatusCode.alias("ExposureStatusCode"), 
    target_table_arotron.ExposureStatusCalc.alias("ExposureStatus"), 
    exposure_arotron.ExposureStatusdate.alias("ExposureStatusDate"), 
    exposure_arotron.Opendate.alias("Opendate"), 
    exposure_arotron.ActiveExpiryDate.alias("ActiveExpiryDate"), 
    exposure_arotron.Maturitydate.alias("Maturitydate"), 
    exposure_arotron.OriginBank.alias("OriginBank"), 
    exposure_arotron.OriginationAmount.alias("OriginationAmount"), 
    target_table_arotron.InitialAssignmentAmount.cast("decimal(19,2)").alias("InitialAssignmentAmount"), 
    exposure_arotron.Currency.alias("Currency"), 
    exposure_arotron.BankingUnitCode.alias("BankingUnitCode"), 
    exposure_arotron.BankingUnit.alias("BankingUnit"), 
    exposure_arotron.MechanismCode.alias("MechanismCode"), 
    exposure_arotron.Mechanism.alias("Mechanism"), 
    exposure_arotron.BranchCode.alias("BranchCode"), 
    exposure_arotron.Branch.alias("Branch"), 
    exposure_arotron.PurposeReasonCode.alias("PurposeReasonCode"), 
    exposure_arotron.PurposeReason.alias("PurposeReason"), 
    exposure_arotron.PurposeKTCode.alias("PurposeKTCode"), 
    exposure_arotron.PurposeKT.alias("PurposeKT"), 
    lit(None).alias("MaterialityStartDate"), 
    exposure_arotron.DaysPastDueWithMateriality.cast("int").alias("DaysPastDueWithMateriality"),
    exposure_arotron.DaysPastDue.alias("DaysPastDue"), 
    exposure_arotron.DaysPastDueinterest.alias("DaysPastDueinterest"), 
    exposure_arotron.FirstArreardate.alias("FirstArreardate"), 
    exposure_arotron.Overduedate.alias("Overduedate"), 
    exposure_non_active_arotron.ClosedDate.alias("ClosedDate"),
    exposure_arotron.IsActiveFlag.alias("IsActiveFlag"),
    exposure_arotron.IsActiveFlag.cast("smallint").alias("IsActiveFlagInt"),
    lit(None).alias("FirstInstallmentDate"),
    exposure_arotron.Bucket.alias("BucketCode"),
    exposure_arotron.Bucket.alias("Bucket"),
    exposure_arotron.WriteOffdate.alias("WriteOffDate"),
    exposure_arotron.WriteOffAmount.cast("decimal (19,2)").alias("WriteoffAmount"),
    col("LPSCodeArotron").cast("string").alias("LPSCode"),
    col("LPSArotron").cast("string").alias("LPS"),
    lit(None).cast("string").alias("LPSEng"),
    col("LPSDateArotron").cast("string").alias("LPSDate"),
    exposure_arotron.DenouncedDateArotron.alias("DenouncedDate"),
    exposure_arotron.DaysPastDuePrincipal.alias("DaysPastDuePrincipal"),
    exposure_arotron.ExpectedOverduedate.alias("ExpectedOverduedate"),
    exposure_arotron.OutstandingTotalBalance.cast("decimal(19,2)").alias("OutstandingTotalBalance"),
    exposure_arotron.OffBalance.cast("decimal(19,2)").alias("OffBalance"),
    target_table_arotron.OnBalanceCalc.cast("decimal(19,2)").alias("OnBalance"),
    exposure_arotron.DueAmount.cast("decimal(26,4)").alias("DueAmount"), 
    lit(None).cast("decimal(19,2)").alias("BogBalance"),
    lit(None).cast("double").alias("GBV"),
    lit(None).cast("Boolean").alias("GBVTriggerEventFlag"),
    lit(None).cast("Date").alias("GBVTriggerEventDate"),
    lit(None).cast("String").alias("GBVTriggerEventType"),
    lit(None).cast("double").alias("TailExposure"),
    lit(None).cast("double").alias("GBVCappedAmount"),
    lit(None).cast("Date").alias("GBVCappedDate"),
    lit(None).cast("double").alias("GBVFunctionResult"),
    exposure_arotron.EverDenouncedDate.cast("date").alias("EverDenouncedDate"),  
    exposure_arotron.EverDenouncedFlagArotron.cast("boolean").alias("EverDenouncedFlag"),
    lit(None).alias("IsSecuredFlag"),
    lit(None).alias("InterestNextDueDate") ,  
    lit(None).alias("ClosedReasonCode"), #not calculated
    lit(None).alias("ClosedReason"),     #not calculated
    lit(None).alias("PEERestructureTypeCode"),  
    lit(None).alias("PEEAdjustmentCode"),    
    exposure_arotron.Defaultdate.alias("Defaultdate"),
    exposure_arotron.Defaultdate.cast("date").alias("LatestDefaultDate"),
    target_table_arotron.NPLDefaultAmountCalc.cast("decimal(19,2)").alias("NPLDefaultAmount"), 
    target_table_arotron.NonCondWriteOffDateCalc.alias("NonConditionalWriteOffDate") , 
    target_table_arotron.NonCondWriteOffAmountCalc.cast("decimal(19,2)").alias("NonConditionalWriteOffAmount"),  
    lit(None).alias("ConditionalWriteOffDate"),  
    lit(None).alias("ConditionalWriteOffAmount"), 
    exposure_arotron.NextInstallmentdate.cast("date").alias("NextInstallmentDate"),  
    lit(None).alias("NextInstallmentInterestDate"),   
    lit(None).alias("NextInstallmentCapitalDate"),   
    lit(None).alias("NextInstallmentMinimumDate"),    
    lit(None).alias("NextInstallmentAmount"),   
    lit(None).alias("NextInstallmentCapitalAmount"),         
    exposure_arotron.Tenor.cast("int").alias("ExposureTenor"),
    lit(None).alias("IsSoldFlag"),
    exposure_arotron.IsSubsidyFlagArotron.cast("boolean").alias("IsSubsidyFlag"),
    exposure_arotron.IsGuaranteeFlag.alias("IsGuaranteeFlag"),
    target_table_arotron.MasterAccountCalc.alias("MasterExposureKey"),
    exposure_arotron.SlaveFlag.alias("SlaveFlag"),
    lit(None).alias("PaymentOrderFlag"),
    lit(None).alias("PaymentOrderDate"),
    exposure_arotron.NumberOfUnpaidBills.cast("string").alias("NumberOfUnpaidBills"), 
    lit(None).alias("SettlementBucketCode"), 
    lit(None).alias("SettlementBucket"), 
    lit(None).alias("DaysPastDueFinal"), 
    lit(None).alias("HasActiveSettlementFlag"), 
    lit(None).alias("UnderLawTypeCode"), #not calculated
    lit(None).alias("UnderLawType"), #not calculated
    lit(None).alias("LawStatusCode"), #not calculated
    lit(None).alias("LawStatus"), #not calculated
    lit(None).alias("LawApplicationStatusDate"), #not calculated
    lit(None).alias("LawCourtDate"),  #not calculated
    lit(None).alias("LawProvisionalCourtDecisionDate"),  #not calculated
    exposure_arotron.InstalmentFrequency.alias("InstalmentFrequency"), 
    exposure_arotron.InstallmentFixedAmount.alias("InstallmentFixedAmount"), 
    exposure_arotron.NumberOfInstallments.alias("NumberOfInstallments"), 
    exposure_arotron.PreviousInstallmentdate.alias("PreviousInstallmentdate"), 
    exposure_arotron.LastBillingdate.alias("LastBillingdate"), 
    exposure_arotron.LastRestructuringTypeCode.alias("LastRestructuringTypeCode"), 
    exposure_arotron.LastRestructuringType.alias("LastRestructuringType"), 
    exposure_arotron.LastRestructuringDate.alias("LastRestructuringDate"), 
    exposure_arotron.RestructuringPEEDate.alias("RestructuringPEEDate"), 
    exposure_arotron.RestructuringPEETypeCodeArotron.alias("RestructuringPEETypeCode"), 
    exposure_arotron.RestructuringPEEType.alias("RestructuringPEEType"), 
    exposure_arotron.ObjectiveStatusCode.alias("ObjectiveStatusCode"), 
    exposure_arotron.ObjectiveStatus.alias("ObjectiveStatus"), 
    exposure_arotron.ObjectiveStatusdate.alias("ObjectiveStatusdate"), 
    exposure_arotron.TreatmentBranchCode.alias("TreatmentBranchCode"),
    exposure_arotron.TreatmentBranchCode.alias("TreatmentBranch"),
    target_table_arotron.GracePeriodEndDateCalc.alias("GracePeriodEnddate"), 
    exposure_arotron.GracePeriodTypeCode.alias("GracePeriodTypeCode"), 
    exposure_arotron.GracePeriodType.alias("GracePeriodType"), 
    exposure_arotron.GracePeriodinterestPayableFlag.alias("GracePeriodinterestPayableFlag"), 
    exposure_arotron.GracePeriodNextinterestdate.alias("GracePeriodNextinterestdate"), 
    exposure_arotron.GracePeriodinterestFrequency.alias("GracePeriodinterestFrequency"), 
    exposure_arotron.Principal.alias("Principal"), 
    exposure_arotron.interest.alias("interest"), 
    exposure_arotron.Unrealizedinterest.alias("Unrealizedinterest"), 
    exposure_arotron.Commission.alias("Commission"), 
    exposure_arotron.Expense.alias("Expense"), 
    exposure_arotron.Accrued.alias("Accrued"), 
    exposure_arotron.DuePrincipal.alias("DuePrincipal"), 
    exposure_arotron.Dueinterest.alias("Dueinterest"), 
    exposure_arotron.DueUnrealizedinterest.alias("DueUnrealizedinterest"), 
    exposure_arotron.DueCommission.alias("DueCommission"), 
    exposure_arotron.DueExpense.alias("DueExpense"), 
    exposure_arotron.DueAccrued.alias("DueAccrued"), 
    exposure_arotron.Penaltyinterest.alias("Penaltyinterest"), 
    exposure_arotron.PenaltyUnrealizedinterest.alias("PenaltyUnrealizedinterest"), 
    exposure_arotron.InterestRate.cast("decimal(19,6)").alias("InterestRate"), 
    exposure_arotron.interestCode.alias("interestCode"), 
    exposure_arotron.interestDescription.alias("interestDescription"), 
    exposure_arotron.BaseRate.alias("BaseRate"), 
    exposure_arotron.BaseCode.alias("BaseCode"), 
    exposure_arotron.BaseDescription.alias("BaseDescription"), 
    exposure_arotron.N128Rate.alias("N128Rate"), 
    exposure_arotron.SpreadRate.alias("SpreadRate"), 
    exposure_arotron.PenaltyRate.alias("PenaltyRate"), 
    exposure_arotron.InterestRateTypeCode.alias("InterestRateTypeCode"), 
    exposure_arotron.InterestRateType.alias("InterestRateType"), 
    exposure_arotron.SpreadCurrencyCode.alias("SpreadCurrencyCode"), 
    exposure_arotron.SpreadCurrency.alias("SpreadCurrency"), 
    exposure_arotron.SpreadPeriod.alias("SpreadPeriod"), 
    exposure_arotron.InterestDuration.alias("InterestDuration"), 
    exposure_arotron.DayCounts.alias("DayCounts"), 
    exposure_arotron.InterestFrequency.alias("InterestFrequency"), 
    exposure_arotron.InterestFixedDate.alias("InterestFixedDate"), 
    exposure_arotron.interestValiditydate.alias("interestValiditydate"), 
    exposure_arotron.InterestChangeStartdate.alias("InterestChangeStartdate"), 
    exposure_arotron.InterestChangeEnddate.alias("InterestChangeEnddate"), 
    exposure_arotron.InterestRateUpdatedate.alias("InterestRateUpdatedate"), 
    exposure_arotron.InterestCalcStopFlag.alias("InterestCalcStopFlag"), 
    exposure_arotron.AutoRenewalFlag.alias("AutoRenewalFlag"), 
    exposure_arotron.AssetSegmentCode.alias("AssetSegmentCode"), 
    exposure_arotron.AssetSegment.alias("AssetSegment"), 
    exposure_arotron.DiasNumber.alias("DiasNumber"), 
    exposure_arotron.ProvisionFrozenInterestAmount.alias("ProvisionFrozenInterestAmount"), 
    exposure_arotron.AccountNticeBUKey.alias("AccountNticeBUKey"), 
    exposure_arotron.SayFlag.alias("SayFlag"), 
    exposure_arotron.InsuranceCompany.alias("InsuranceCompany"), 
    exposure_arotron.PhysicalNumber.alias("PhysicalNumber"), 
    exposure_arotron.PreCourtDate.alias("PreCourtDate"), 
    exposure_arotron.CourtDate.alias("CourtDate"), 
    exposure_arotron.ContractSigningDate.alias("ContractSigningDate"), 
    exposure_arotron.ExposureLimitAmount.alias("ExposureLimitAmount"), 
    exposure_arotron.BaloonAmount.alias("BaloonAmount"), 
    exposure_arotron.AdministrationStatusCode.alias("AdministrationStatusCode"), 
    exposure_arotron.AdministrationStatus.alias("AdministrationStatus"), 
    exposure_arotron.Tenor.alias("Tenor"), 
    exposure_arotron.SPVCode.alias("SPVCode"), 
    exposure_arotron.SPV.alias("SPV"), 
    exposure_arotron.TotalOverdueInsterest.alias("TotalOverdueInsterest"), 
    exposure_arotron.TotalOverdueInstallments.alias("TotalOverdueInstallments"), 
    exposure_arotron.OutstandingTotalBalanceMLT.cast("decimal(19,2)").alias("OutstandingTotalBalanceMLT"), 
    lit(None).alias("NextInterestChangeDate"),
    lit(None).alias("PrincipalNextDueDate"),    
    lit(None).alias("FutureInstallmentIncreaseGroup"),    
    lit(None).alias("FutureInitialInstallmentGroup"),    
    lit(None).alias("FutureStepUpSegment"),
    lit(None).alias("FutureStepUpIndicator"),
    lit(None).alias("PastInstallmentIncreaseGroup"),    
    lit(None).alias("PastInitialInstallmentGroup"),    
    lit(None).alias("PastStepUpSegment"),
    lit(None).alias("PastStepUpIndicator"),
    lit(None).alias("TotalPrincipal"),
    lit(None).alias("TotalInterest"), 
    lit(None).alias("TotalAccruals"),
    lit(None).alias("TotalExpense"), 
    lit(None).alias("Coupon"),
    lit(None).alias("ActiveModificationProgramStatus"), 
    lit(None).alias("DenouncementIndicator"), 
    lit(None).alias("ActiveExposureRestructureStartDate"),
    lit(None).alias("MonthsOfCleanPayHistory"),
    lit(None).alias("TotalPaymentAgreementPaid"),
    lit(None).alias("TotalPaymentAgreementDue"),
    lit(None).alias("ExposureRestructureOrigination"),
    lit(None).alias("ExposureRestructureStatus"),
    lit(None).alias("ExposureRestructureType"),
    lit(None).alias("CodeofConductStage"),
    lit(None).alias("LetterStatusCode"),
    lit(None).alias("LetterStatus"),
    lit(None).alias("LetterStatusDate"),
    lit(None).alias("CoCStatusCode"),
    lit(None).alias("CoCStatus"),
    lit(None).alias("CoCAccountStatusDate"),
    lit(None).alias("FirstLetterDate"),
    lit(None).alias("SecondLetterDate"),
    lit(None).alias("ThirdLetterDate"),
    lit(None).alias("FirstLetterReminderDate"),
    lit(None).alias("FirstLetterFlag"),
    lit(None).alias("FirstLetterReminderFlag"),
    lit(None).alias("SecondLetterFlag"),
    lit(None).alias("ThirdLetterFlag"),
    lit(None).alias("ExclusionDate"),
    lit(None).alias("ExclusionReasonCode"),
    lit(None).alias("ExclusionReason"),
    lit(None).cast("int").alias("L3869Flag"),
    lit(None).cast("string").alias("L3869Status"),
    lit(None).cast("date").alias("L3869SubmissionDate"),
    lit(None).cast("date").alias("L3869ProvisionalCourtDecisionDate"),
    lit(None).cast("date").alias("L3869CourtDate"),
    lit(None).alias("PrincipalAtDefault"),
    lit(None).alias("InterestAtDefault"),
    lit(None).cast("decimal(26,4)").alias("AmountDueAtDefault"),
    lit(None).alias("LastModificationActionType"),
    lit(None).alias("LastModificationDate"),
    lit(None).alias("HasBalloonFlag"),
    lit(None).alias("HasZeroInterestTodayFlag"),
    lit(None).cast("string").alias("InterestRateTypeTaxonomy"),  
    lit(None).cast("string").alias("InterestRateResetInterval"),   
    lit(None).cast("boolean").alias("HasSplitBalanceFlag"),
    lit(None).cast("date").alias("FixInterestEndDate"),
    lit(None).cast("string").alias("RepaymentMethod"),
    lit(None).cast("int").alias("RepaymentFrequency"),
    lit(None).cast("boolean").alias("HasStepUpInstallmentFlag"),
    lit(None).cast("string").alias("UnpaidInterestHandling"),
    lit(None).cast("boolean").alias("HasFutureSpreadFlag"),
    lit(None).cast("boolean").alias("IsBulletFlag"),
    lit(None).cast("boolean").alias("HasFutureInterestChangesFlag"),
    lit(None).cast("boolean").alias("OCWFlag"),
    lit(None).cast("string").alias("OCWApplicationStatus"),
    lit(None).cast("timestamp").alias("L3869InterestFreePeriodEndDate"),
    lit(None).cast("boolean").alias("InArrearsFlag"),
    lit(None).cast("boolean").alias("HasPaymentPlanFlag"),
    lit(None).cast("date").alias("DateLastCurrent"),
    lit(None).cast("date").alias("DateLastInArrears"), 
    lit(None).cast("date").alias("LastInstallmentDate"),
    lit(None).cast("int").alias("SettlementDaysPastDue"),
    lit(None).alias("LegalActionPriority"), 
    exposure_arotron.CDI.isin(hb_high_balance_cdis_list).cast("boolean").alias("IsHighBalanceFlag"),
    lit(None).cast("string").alias("ApplicationDecision"),
    lit(None).cast("boolean").alias("ExtrajudicialLetterEverFlag"),
    lit(None).cast("date").alias("ExtrajudicialLetterLastDate"), 
    lit(None).cast("string").alias("CourtRegion"),  
    target_table_arotron.SubServicer.cast("string").alias("SubServicer"), 
    lit(None).cast("decimal(19,2)").alias("IFRSGBV"),
    lit(None).cast("decimal(19,4)").alias("DueDebtAmount"),
    target_table_arotron.TotalUnrealizedInterest.cast("decimal(19,2)").alias("TotalUnrealizedInterest"),
    target_table_arotron.NetDueAmount.cast("decimal(19,2)").alias("NetDueAmount"),
    target_table_arotron.TotalInterestAccruals.cast("decimal(19,2)").alias("TotalInterestAccruals"),
    lit(None).cast("decimal(19,2)").alias("TotalInterestL3869"),
    lit(None).cast("int").alias("JustificationCode"), 
    lit(None).cast("string").alias("JustificationComments"),
    lit(None).cast("decimal(9,4)").alias("FutureMargin"),
    lit(None).cast("timestamp").alias("FutureMarginApplyDate"),
    lit(None).cast("string").alias("FirstPeriodNumberOfInstallements"),
    lit(None).cast("decimal(4,0)").alias("InstalmentSNFrom"),
    lit(None).cast("decimal(4,0)").alias("InstalmentSNTo"),
    lit(None).cast("decimal(15,2)").alias("newInstalmentAmount"),
    lit(None).cast("string").alias("L3869WaterfallMainBorrower"),
    exposure_arotron.BusinessDate.alias("BusinessDate"),
    lit(None).cast("Boolean").alias("ForcedOnboarding"),
    lit(None).cast("Boolean").alias("HolisticFlag"),
    lit(None).cast("String").alias("ForcedOnboardReason"),
    lit(None).cast("date").alias("FutureStepUpEventDate"),
    lit(None).cast("decimal(38,7)").alias("FutureStepUpEventAmount")

)

## Final union & table write

In [0]:
result_df = result_df_profits.union(result_df_qca).union(result_df_arotron)

if unity == "True":
    result_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(derivative_table_path) # in order to write in delta_lake
else: 
    result_df.write \
        .format("delta") \
        .mode("append") \
        .save(derivative_table_path) # in order to write in delta_lake